# Mostaza observed B180 preparation

New quota authorization required. GPU and internet disabled. No evaluation has run.


In [ ]:
import math as observation_math
import re as observation_re
import time as observation_time
import uuid as observation_uuid

BROWSER_AUTHORIZATION = None
OBSERVATION_NAMESPACE = {"__name__": "mostaza_startup_observation"}


def require_launch_ready():
    receipt = BROWSER_AUTHORIZATION
    if not isinstance(receipt, dict) or receipt.get("policy") != {
        "id": "B180-observed-v2",
        "startup_seconds": 1800,
        "last_admission_seconds": 5340,
        "work_seconds": 5940,
        "request_stop_seconds": 6240,
        "latest_stop_seconds": 6300,
        "ceiling_seconds": 7200,
        "observer_interval_seconds": 5,
        "supervisor_interval_seconds": 60,
        "evaluations": 6,
        "allocations": 1,
        "retries": 0,
        "gpu_count": 4,
        "minimum_reported_quota_seconds": 28800,
        "paid_purchases_usd": 0,
    }:
        raise RuntimeError("New observed-v2 quota authorization is absent")
    if receipt.get("approved") is not True or not receipt.get("approval_id"):
        raise RuntimeError("Explicit new human authorization is required")
    try:
        observation_uuid.UUID(receipt["single_use_id"])
    except (ValueError, TypeError, KeyError, AttributeError):
        raise RuntimeError("Fresh UUID notebook identity is required") from None
    if not isinstance(receipt.get("ref"), str) or not observation_re.fullmatch(
        r"diegofescobar/mostaza-b180-[0-9a-f]{32}", receipt["ref"]
    ):
        raise RuntimeError("Fresh dedicated notebook reference is required")
    if not isinstance(receipt.get("code_commit"), str) or not observation_re.fullmatch(
        r"[0-9a-f]{40}", receipt["code_commit"]
    ):
        raise RuntimeError("Exact code provenance is required")
    if (
        receipt.get("prelaunch_latest_versions_terminal") is not True
        or receipt.get("prelaunch_quota_stable") is not True
    ):
        raise RuntimeError(
            "Prelaunch terminal-version and stable-quota checks required"
        )
    if receipt.get("reserved_quota_seconds") != 0 or receipt.get("active_events") != 0:
        raise RuntimeError("Prelaunch allocation state is missing or nonempty")
    remaining = receipt.get("remaining_quota_seconds")
    if (
        not isinstance(remaining, (int, float))
        or not observation_math.isfinite(remaining)
        or remaining < 28800
    ):
        raise RuntimeError("Insufficient freshly observed quota")
    start = receipt.get("t0_epoch")
    if not isinstance(start, (int, float)) or not observation_math.isfinite(start):
        raise RuntimeError("External prelaunch clock missing")
    quota_time = receipt.get("quota_observed_epoch")
    if (
        not isinstance(quota_time, (int, float))
        or not observation_math.isfinite(quota_time)
        or not 0 <= start - quota_time <= 60
    ):
        raise RuntimeError("Fresh quota observation is required")
    if observation_time.time() < start - 5 or observation_time.time() >= start + 6240:
        raise RuntimeError("External prelaunch clock expired or inconsistent")


require_launch_ready()
exec(
    compile(
        (
            '"Persistent startup observatio'
            "ns; no model or accelerator op"
            'erations."\n\nfrom __future__ im'
            "port annotations\n\nimport json\n"
            "import logging\nimport os\nimpor"
            "t sys\nimport threading\nimport "
            "time\nfrom datetime import UTC,"
            " datetime\nfrom pathlib import "
            "Path\nfrom typing import Any\n\n\n"
            'class StartupObserver:\n    "Ke'
            "ep events and native subproces"
            "s output even if cleanup unlin"
            'ks its log."\n\n    def __init__'
            "(self, root: Path, launch_epoc"
            "h: float, interval: float = 5."
            "0):\n        if interval <= 0:\n"
            '            raise ValueError("'
            "Observation interval must be p"
            'ositive")\n        root.mkdir(p'
            "arents=True, exist_ok=False)\n "
            "       self.root = root\n      "
            "  self.launch_epoch = launch_e"
            "poch\n        self.interval = i"
            "nterval\n        self.started ="
            " time.monotonic()\n        self"
            '.phase = "observer.ready"\n    '
            "    self.server: Any = None\n  "
            "      self.failure: str | None"
            " = None\n        self._lock = t"
            "hreading.RLock()\n        self."
            "_stop = threading.Event()\n    "
            "    self._thread: threading.Th"
            "read | None = None\n        sel"
            "f._log_offset = 0\n        self"
            '._logger = logging.getLogger("'
            'adk_submission.server")\n      '
            "  self._old_level = self._logg"
            "er.level\n        self._handler"
            " = logging.FileHandler(root / "
            '"server-lifecycle.log")\n      '
            "  self._handler.setFormatter(\n"
            "            logging.Formatter("
            '"%(asctime)s %(levelname)s %(m'
            'essage)s")\n        )\n        s'
            "elf._logger.addHandler(self._h"
            "andler)\n        self._logger.s"
            "etLevel(logging.INFO)\n        "
            '(root / "startup-schema.json")'
            ".write_text(\n            json."
            "dumps(\n                {\n     "
            '               "since_launch_s'
            'econds": (\n                   '
            '     "seconds from external UT'
            'C epoch; in"\n                 '
            '       "cludes queue/startup, '
            'not pure mode"\n               '
            '         "l time"\n            '
            "        ),\n                   "
            ' "observer_monotonic_seconds":'
            ' (\n                        "se'
            "conds since observer initializ"
            'ation, not allocation start"\n '
            "                   ),\n        "
            '            "server_log_bytes"'
            ': (\n                        "b'
            "ytes in retained native subpro"
            'cess output at sampling time"\n'
            "                    ),\n       "
            '             "health_ready": ('
            '\n                        "Nati'
            "ve health probe Boolean only d"
            'u"\n                        "ri'
            "ng startup; False does not dis"
            'tin"\n                        "'
            'guish refusal/timeout/non-200"'
            "\n                    ),\n      "
            '              "server_return_c'
            'ode": (\n                      '
            '  "Native process poll exit co'
            'de; null"\n                    '
            '    " while running or without'
            ' a process"\n                  '
            '      " handle; see server_pro'
            'cess_state"\n                  '
            '  ),\n                    "new_'
            'log_text": (\n                 '
            '       "Up to4096 newly read b'
            'ytes decoded "\n               '
            '         "as UTF8 replacement;'
            ' full bytes ret"\n             '
            '           "ained in server-re'
            'tained.log"\n                  '
            '  ),\n                    "cove'
            'rage": (\n                     '
            '   "Heartbeat snapshots every5'
            'seconds p"\n                   '
            '     "lus native probe latency'
            '; not exhau"\n                 '
            '       "stive process or model'
            ' timing"\n                    )'
            ",\n                },\n         "
            "       sort_keys=True,\n       "
            '     )\n            + "\\n"\n    '
            '    )\n        self.event("obse'
            'rver.ready", physical_gpu_rele'
            "ase_confirmed=None)\n\n    def e"
            "vent(self, kind: str, **detail"
            "s: Any) -> None:\n        with "
            "self._lock:\n            now = "
            "time.time()\n            record"
            ' = {\n                "event": '
            'kind,\n                "phase":'
            ' self.phase,\n                "'
            'timestamp_utc": datetime.fromt'
            "imestamp(now, UTC).isoformat()"
            ',\n                "since_launc'
            'h_seconds": now - self.launch_'
            'epoch,\n                "observ'
            'er_monotonic_seconds": time.mo'
            "notonic() - self.started,\n    "
            "            **details,\n       "
            "     }\n            text = json"
            ".dumps(record, sort_keys=True)"
            "\n            with (self.root /"
            ' "startup-events.jsonl").open('
            '"a") as stream:\n              '
            '  stream.write(text + "\\n")\n  '
            "              stream.flush()\n "
            "           pending = self.root"
            ' / ".startup-status.tmp"\n     '
            "       pending.write_text(text"
            ' + "\\n")\n            pending.r'
            'eplace(self.root / "startup-st'
            'atus.json")\n            print('
            '"MOSTAZA_OBSERVATION " + text,'
            " flush=True)\n\n    def stage(se"
            "lf, name: str) -> None:\n      "
            "  self.check()\n        self.ph"
            "ase = name\n        self.event("
            '"stage")\n\n    def start(self) '
            "-> None:\n        if self._thre"
            "ad is not None:\n            ra"
            'ise ValueError("Observer alrea'
            'dy started")\n        self._thr'
            "ead = threading.Thread(target="
            "self._run, daemon=True)\n      "
            "  self._thread.start()\n\n    de"
            "f bind_server(self, server: An"
            'y) -> None:\n        "Hard link'
            " on one filesystem: native sto"
            "p can unlink only its own name"
            '."\n        self.check()\n      '
            '  source = self.root / "server'
            '-source.log"\n        retained '
            '= self.root / "server-retained'
            '.log"\n        with source.open'
            '("xb"):\n            pass\n     '
            "   os.link(source, retained)  "
            "# Fail before model startup if"
            " persistence fails.\n        se"
            "rver.log_path = str(source)\n  "
            "      self.server = server\n   "
            '     self.event("server.log.bo'
            'und", retained_log=retained.na'
            "me)\n\n    def sample(self) -> N"
            "one:\n        details: dict[str"
            ', Any] = {\n            "server'
            '_pid": None,\n            "serv'
            'er_return_code": None,\n       '
            '     "health_ready": None,\n   '
            '         "server_log_bytes": N'
            'one,\n            "new_log_text'
            '": None,\n            "server_p'
            'rocess_state": "UNBOUND",\n    '
            '        "health_probe_reason":'
            ' "not_probed_no_process",\n    '
            '        "server_log_reason": "'
            'server_not_bound",\n           '
            ' "new_log_reason": "server_not'
            '_bound",\n        }\n        wit'
            "h self._lock:\n            if s"
            "elf.server is not None:\n      "
            '          details["server_proc'
            'ess_state"] = "NOT_SPAWNED_OR_'
            'HANDLE_CLEARED"\n              '
            "  process = self.server.proces"
            "s\n                if process i"
            "s not None:\n                  "
            '  details["server_pid"] = proc'
            "ess.pid\n                    de"
            'tails["server_return_code"] = '
            "process.poll()\n               "
            '     details["server_process_s'
            'tate"] = (\n                   '
            '     "RUNNING" if details["ser'
            'ver_return_code"] is None else'
            ' "EXITED"\n                    '
            ")\n                    details["
            '"health_probe_reason"] = "not_'
            'probed_outside_startup"\n      '
            "              if self.phase =="
            ' "server.start.wait":\n        '
            "                # Native healt"
            "h probe: two-second timeout, n"
            "o inference.\n                 "
            '       details["health_ready"]'
            " = self.server.is_healthy()\n  "
            "                      details["
            '"health_probe_reason"] = None\n'
            "                path = self.ro"
            'ot / "server-retained.log"\n   '
            "             if path.exists():"
            '\n                    details["'
            'server_log_reason"] = None\n   '
            '                 details["new_'
            'log_reason"] = "no_new_bytes"\n'
            '                    details["s'
            'erver_log_bytes"] = path.stat('
            ").st_size\n                    "
            'with path.open("rb") as stream'
            ":\n                        stre"
            "am.seek(self._log_offset)\n    "
            "                    data = str"
            "eam.read(4096)\n               "
            "     self._log_offset += len(d"
            "ata)\n                    if da"
            "ta:\n                        de"
            'tails["new_log_text"] = data.d'
            'ecode(errors="replace")\n      '
            '                  details["new'
            '_log_reason"] = None\n         '
            "       else:\n                 "
            '   details["server_log_reason"'
            '] = "retained_log_missing"\n   '
            '                 details["new_'
            'log_reason"] = "retained_log_m'
            'issing"\n            self.event'
            '("heartbeat", **details)\n\n    '
            "def check(self) -> None:\n     "
            "   if self.failure is not None"
            ":\n            raise RuntimeErr"
            'or("Startup observation failed'
            ': " + self.failure)\n\n    def _'
            "run(self) -> None:\n        whi"
            "le not self._stop.wait(self.in"
            "terval):\n            try:\n    "
            "            self.sample()\n    "
            "        except Exception as er"
            "ror:\n                self.fail"
            'ure = type(error).__name__ + "'
            ': " + str(error)\n             '
            "   print(\n                    "
            '"MOSTAZA_OBSERVATION_ERROR " +'
            " self.failure,\n               "
            "     file=sys.stderr,\n        "
            "            flush=True,\n      "
            "          )\n                re"
            "turn\n\n    def close(self) -> N"
            "one:\n        self._stop.set()\n"
            "        if self._thread is not"
            " None:\n            self._threa"
            "d.join(timeout=4)\n        self"
            "._logger.removeHandler(self._h"
            "andler)\n        self._handler."
            "close()\n        self._logger.s"
            "etLevel(self._old_level)\n     "
            '   self.event("observer.closed'
            '", observation_failure=self.fa'
            "ilure)\n\n\ndef start_observed_se"
            "rver(server: Any, observer: St"
            'artupObserver) -> None:\n    "U'
            "se the unchanged native start "
            "method, with durable context o"
            'n either exit."\n    observer.b'
            "ind_server(server)\n    observe"
            'r.stage("server.start.wait")\n '
            '   observer.event(\n        "se'
            'rver.start.requested",\n       '
            " startup_timeout_seconds=serve"
            "r.config.startup_timeout,\n    "
            "    command=server.build_cmd()"
            ",\n    )\n    try:\n        serve"
            "r.start()\n    except BaseExcep"
            "tion as error:\n        observe"
            'r.event(\n            "server.s'
            'tart.failed",\n            erro'
            "r_type=type(error).__name__,\n "
            "           error_message=str(e"
            "rror)[-4096:],\n        )\n     "
            "   raise\n    observer.check()\n"
            '    observer.stage("server.rea'
            'dy")\n    observer.sample()\n'
        ),
        "startup_observation.py",
        "exec",
    ),
    OBSERVATION_NAMESPACE,
)
OBSERVATION_PATH = OBSERVATION_NAMESPACE["Path"]
STARTUP_OBSERVER = OBSERVATION_NAMESPACE["StartupObserver"](
    OBSERVATION_PATH("/kaggle/working/private-phase2-evidence")
    / BROWSER_AUTHORIZATION["single_use_id"],
    BROWSER_AUTHORIZATION["t0_epoch"],
)
STARTUP_OBSERVER.start()


class ObservedSession:
    def __init__(self, receipt):
        self.start_epoch = receipt["t0_epoch"]
        self.deadline_epoch = self.start_epoch + 6240
        self.startup_deadline_epoch = self.start_epoch + 1800
        self.code_commit = receipt["code_commit"]
        self.run_url = "https://www.kaggle.com/code/" + receipt["ref"]

    def require_remaining(self, seconds):
        require_launch_ready()
        STARTUP_OBSERVER.check()
        if self.deadline_epoch - observation_time.time() < seconds:
            raise RuntimeError("Insufficient remaining session allowance")

    def require_startup_ready(self):
        self.require_remaining(0)
        if observation_time.time() >= self.startup_deadline_epoch:
            raise RuntimeError("Thirty-minute startup deadline exceeded")

    def record_stop_request(self, directory):
        record = {
            "event": "BROWSER_STOP_REQUIRED",
            "run_url": self.run_url,
            "physical_release_confirmed": None,
        }
        (directory / "browser-stop-request.json").write_text(
            OBSERVATION_NAMESPACE["json"].dumps(record)
        )
        print(
            "MOSTAZA_BROWSER_STOP_REQUIRED",
            OBSERVATION_NAMESPACE["json"].dumps(record),
            flush=True,
        )


PLATFORM_SESSION = ObservedSession(BROWSER_AUTHORIZATION)
STARTUP_OBSERVER.stage("bootstrap.begin")

In [ ]:
import base64
import hashlib
import json
import sys
import uuid
from pathlib import Path

EMBEDDED_PAYLOAD = {
    "archives/E0-B180.zip": (
        "UEsDBBQAAAAAAAAAIQAQ6lhwmgEAAJoBAAAKAAAAYWdlbnQueWFtbGdlbmVyYXRlX2Nv"
        "bnRlbnRfY29uZmlnOgogIG1heF9vdXRwdXRfdG9rZW5zOiAxNjM4NAogIHRlbXBlcmF0"
        "dXJlOiAwLjIKICB0aGlua2luZ19jb25maWc6CiAgICBpbmNsdWRlX3Rob3VnaHRzOiB0"
        "cnVlCiAgICB0aGlua2luZ19idWRnZXQ6IDQwOTYKICB0b3BfcDogMC45NQppbnN0cnVj"
        "dGlvbjogIWluY2x1ZGUgJ3Byb21wdHMvYmFzZWxpbmUubWQnCm1vZGVsOiBnZW1tYS00"
        "LTMxYi1pdC1xYXQtdzRhMTYtY3QKbmFtZTogc3dlX2Jhc2VsaW5lX2FnZW50CnRvb2xz"
        "OgotIHJ1bl9jb21tYW5kCi0gcmVhZF9maWxlCi0gZWRpdF9maWxlCi0gd3JpdGVfZmls"
        "ZQotIGdldF9zdGF0dXMKLSBzdWJtaXRfcGF0Y2gKLSBnZXRfY29kZV9uZWlnaGJvcnMK"
        "LSBzZWFyY2hfc2ltaWxhcl9jb2RlCi0gZ2V0X2NvZGVfc3ViZ3JhcGgKUEsDBBQAAAAA"
        "AAAAIQCC2EHeggAAAIIAAAAQAAAAZXZhbF9jb25maWcueWFtbGV2YWx1YXRpb246CiAg"
        "bWF4X3RpbWVfbWludXRlczogMwogIG1heF90b29sX2NhbGxzOiAzMAogIG1heF90dXJu"
        "czogNTAKICB0aW1lb3V0X3NlY29uZHM6IDE4MAogIHZlcmlmaWNhdGlvbl90aW1lb3V0"
        "X3NlY29uZHM6IDMwMApQSwMEFAAAAAAAAAAhAB30bsYNEAAADRAAABMAAABwcm9tcHRz"
        "L2Jhc2VsaW5lLm1kWW91IGFyZSBhbiBleHBlcnQgYXV0b25vbW91cyBzb2Z0d2FyZSBl"
        "bmdpbmVlciBhc3NpZ25lZCB0byByZXNvbHZlIGFuIGlzc3VlIGluIGEgcmVwb3NpdG9y"
        "eSBlZmZpY2llbnRseSBhbmQgZGVjaXNpdmVseS4KCiMjIENvcmUgT2JqZWN0aXZlOiBG"
        "YXN0LCBNaW5pbWFsLCBhbmQgUHJlY2lzZSBGaXhlcwpBaW0gdG8gdW5kZXJzdGFuZCwg"
        "cmVzb2x2ZSwgYW5kIHN1Ym1pdCB0aGUgZml4IGluIHRoZSBtaW5pbXVtIG51bWJlciBv"
        "ZiB0b29sIGNhbGxzICh1bmRlciA44oCTMTAgdHVybnMpLiBNb3ZlIGRpcmVjdGx5IGZy"
        "b20gdGhlIHByb2JsZW0gc3RhdGVtZW50IHRvIHRoZSByZWxldmFudCBmaWxlcywgYXBw"
        "bHkgdGhlIHNvbHV0aW9uLCB2ZXJpZnkgd2l0aCBhIHRhcmdldGVkIHRlc3QsIGFuZCBz"
        "dWJtaXQuCgojIyBXb3JrZmxvdwoKIyMjIDEuIElkZW50aWZ5IFRhcmdldCBGaWxlcyBJ"
        "bW1lZGlhdGVseQotIEV4dHJhY3QgZmlsZW5hbWVzLCBmdW5jdGlvbnMsIGNsYXNzZXMs"
        "IENMSSBzdWJjb21tYW5kcywgb3IgZXJyb3IgbWVzc2FnZXMgZGlyZWN0bHkgZnJvbSB0"
        "aGUgcHJvYmxlbSBzdGF0ZW1lbnQuCi0gUmVhZCBvbmx5IHRoZSBzcGVjaWZpYyB0YXJn"
        "ZXQgZmlsZXMgYW5kIGxpbmVzIHVzaW5nIGByZWFkX2ZpbGVgIG9yIHNlYXJjaCB0b29s"
        "cy4gRG8gbm90IHdhbmRlciBhY3Jvc3MgdW5yZWxhdGVkIGZpbGVzLgotIElmIHRoZSBw"
        "cm9ibGVtIHN0YXRlbWVudCBkb2VzIG5vdCBwcm92aWRlIGV4cGxpY2l0IGZpbGUgcGF0"
        "aHMsIHVzZSBgc2VhcmNoX3NpbWlsYXJfY29kZWAgd2l0aCBrZXl3b3JkcyBmcm9tIHRo"
        "ZSBlcnJvciBtZXNzYWdlIHRvIGxvY2F0ZSByZWxldmFudCBmaWxlcyBlZmZpY2llbnRs"
        "eSwgcmF0aGVyIHRoYW4gcnVubmluZyBgZmluZGAgb3IgYGdyZXBgIGFjcm9zcyB0aGUg"
        "ZW50aXJlIHJlcG8uCgojIyMgMi4gSW1wbGVtZW50IHRoZSBTb2x1dGlvbiBEaXJlY3Rs"
        "eQotIEFwcGx5IHRoZSBtaW5pbWFsIG5lY2Vzc2FyeSBmaXggb3IgZmVhdHVyZSBkaXJl"
        "Y3RseSB0byB0aGUgc291cmNlIGZpbGVzIHVzaW5nIGBlZGl0X2ZpbGVgIG9yIGB3cml0"
        "ZV9maWxlYC4KLSBTdHJpY3RseSBhZGhlcmUgdG8gc3BlY2lmaWVkIGVycm9yIHN0cmlu"
        "Z3MsIGV4Y2VwdGlvbiB0eXBlcywgSFRUUCBzdGF0dXMgY29kZXMsIGFuZCBBUEkgc2ln"
        "bmF0dXJlcy4KLSBGb3IgZG9jdW1lbnRhdGlvbiBjb2RlIHRhc2tzIChlLmcuIEZhc3RB"
        "UEkpLCBlZGl0IGV4ZWN1dGFibGUgY29kZSB1bmRlciBgZG9jc19zcmMvYC4KCiMjIyAz"
        "LiBSdW4gVGFyZ2V0ZWQgVGVzdHMgT25seSAoRXhpc3RpbmcgVGVzdHMgTWF5IEJlIEJy"
        "b2tlbikKLSAqKlJ1biBPTkxZIFRhcmdldGVkIFRlc3RzKio6IFJ1biBvbmx5IHRoZSBz"
        "cGVjaWZpYyB0ZXN0IGZpbGUgb3IgdGVzdCBtZXRob2QgZGlyZWN0bHkgdmVyaWZ5aW5n"
        "IHRoZSBidWcgb3IgZmVhdHVyZSB5b3UgbW9kaWZpZWQgKGUuZy4gYHB5dGVzdCB0ZXN0"
        "cy90ZXN0X3RhcmdldC5weSAtayB0ZXN0X2ZlYXR1cmVgIG9yIGBweXRob24zIC1tIHVu"
        "aXR0ZXN0IHRlc3RzLnRlc3RfdGFyZ2V0YCkuCi0gKipCZSBBd2FyZSBUaGF0IEV4aXN0"
        "aW5nIFRlc3RzIE1heSBCZSBCcm9rZW4qKjogTWFueSByZXBvc2l0b3JpZXMgY29udGFp"
        "biBwcmUtZXhpc3RpbmcgdGVzdCBicmVha2FnZXMsIG1pc3NpbmcgdGVzdCBkYXRhIGZp"
        "eHR1cmVzIChlLmcuIGAvdGVzdF9kYXRhYCksIG9yIGVudmlyb25tZW50IGltcG9ydCBl"
        "cnJvcnMgdW5yZWxhdGVkIHRvIHlvdXIgdGFzay4KLSAqKkRvIE5PVCBBdHRlbXB0IHRv"
        "IEZpeCBFeGlzdGluZyBUZXN0cyoqOiBJZiBhbiBleGlzdGluZyB0ZXN0IGZhaWxzIGR1"
        "ZSB0byBwcmUtZXhpc3RpbmcgcmVwb3NpdG9yeSBpc3N1ZXMgb3IgbWlzc2luZyBmaXh0"
        "dXJlcywgSUdOT1JFIElULiBOZXZlciBzcGVuZCB0dXJucyBhdHRlbXB0aW5nIHRvIHJl"
        "cGFpciBwcmUtZXhpc3RpbmcgdGVzdCBmYWlsdXJlcywgY3JlYXRlIHRlc3Qgc3R1YnMs"
        "IG9yIGFsdGVyIHRlc3QgY29kZS4KLSAqKlNUUklDVCBSVUxFOiBORVZFUiBSdW4gQmFy"
        "ZSBQeXRlc3Qgb3IgRnVsbC1SZXBvIFN3ZWVwcyoqOiBORVZFUiBydW4gYmFyZSBgcHl0"
        "ZXN0YCwgYHB5dGVzdCAuYCwgYHB5dGhvbjMgLW0gdW5pdHRlc3QgZGlzY292ZXJgLCBv"
        "ciBmdWxsLXJlcG8gdGVzdCBzdWl0ZXMgd2l0aG91dCBzcGVjaWZ5aW5nIGEgdGFyZ2V0"
        "IGZpbGUuIEZ1bGwgdGVzdCBzdWl0ZXMgdGFrZSBzZXZlcmFsIG1pbnV0ZXMsIGNhdXNl"
        "IGNhdGFzdHJvcGhpYyB0aW1lb3V0cywgYW5kIGV4aGF1c3QgeW91ciB0dXJuIGFuZCB0"
        "aW1lIGJ1ZGdldHMuCi0gSWYgeW91IG5lZWQgdG8gbG9jYXRlIHRoZSB0ZXN0IGZpbGUs"
        "IGZpbmQgaXQgZXhwbGljaXRseSB3aXRoIGBmaW5kIHRlc3RzIC1uYW1lICIqPG5hbWU+"
        "Ki5weSJgIGluc3RlYWQgb2YgcnVubmluZyB0aGUgdGVzdCBydW5uZXIgYWNyb3NzIHRo"
        "ZSByZXBvLgoKIyMjIDQuIEltbWVkaWF0ZSBQYXRjaCBTdWJtaXNzaW9uCi0gT25jZSB5"
        "b3VyIHRhcmdldGVkIHRlc3QgcGFzc2VzOgogIDEuIENhbGwgYHN1Ym1pdF9wYXRjaGAg"
        "aW1tZWRpYXRlbHkuCiAgMi4gVmVyaWZ5IGBwYXRjaF9zaXplID4gMGAgYW5kIGBmaWxl"
        "c19jaGFuZ2VkID4gMGAuCiAgMy4gT3V0cHV0IGEgc2hvcnQgc3VtbWFyeSBvZiB0aGUg"
        "Zml4IHRvIGVuZCB0aGUgc2Vzc2lvbi4KCiMjIEFudGktUGF0dGVybnMgdG8gQXZvaWQK"
        "LSAqKk5FVkVSIG1vZGlmeSwgY3JlYXRlLCBvciBkZWxldGUgdGVzdCBmaWxlcyoqIChg"
        "Kl90ZXN0LnB5YCwgYHRlc3RfKi5weWAsIG9yIGFueXRoaW5nIHVuZGVyIGB0ZXN0cy9g"
        "KS4gQWxsIGNoYW5nZXMgbXVzdCBiZSB0byBzb3VyY2UgaW1wbGVtZW50YXRpb24gZmls"
        "ZXMuIE1vZGlmeWluZyB0ZXN0cyByZXN1bHRzIGluIGFuIGF1dG9tYXRpYyBldmFsdWF0"
        "aW9uIGZhaWx1cmUuCi0gKipORVZFUiBydW4gZnVsbCByZXBvc2l0b3J5IHRlc3Qgc3Vp"
        "dGVzKiogKGUuZy4sIGJhcmUgYHB5dGVzdGAgb3IgYHB5dGVzdCAuYCkg4oCUIGFsd2F5"
        "cyBzcGVjaWZ5IHRoZSBleGFjdCB0ZXN0IGZpbGUgcGF0aC4KLSAqKk5FVkVSIGF0dGVt"
        "cHQgdG8gZml4IG9yIHJlcGFpciBleGlzdGluZyB0ZXN0cyBvciBwcmUtZXhpc3Rpbmcg"
        "cmVwb3NpdG9yeSBicmVha2FnZXMqKiDigJQgeW91ciB0YXNrIGlzIHN0cmljdGx5IHRv"
        "IGltcGxlbWVudCB0aGUgZml4IGZvciB0aGUgcmVwb3J0ZWQgaXNzdWUgaW4gc291cmNl"
        "IGNvZGUuCi0gKipORVZFUiBzZWFyY2ggb3V0c2lkZSBgL3dvcmtzcGFjZWAqKiBmb3Ig"
        "c291cmNlIGZpbGVzIG9yIHBhY2thZ2VzIChlLmcuLCBgL3Vzci9sb2NhbC9saWIvYCwg"
        "YC93aGVlbHMvYCwgYC9vcHQvYCkuIEFsbCByZXBvc2l0b3J5IGNvZGUgYW5kIHRlc3Qg"
        "ZGVwZW5kZW5jaWVzIGFyZSBwcmUtaW5zdGFsbGVkLiBJZiBgTW9kdWxlTm90Rm91bmRF"
        "cnJvcmAgb2NjdXJzIGR1cmluZyB0ZXN0IHJ1bnMsIGZvY3VzIG9uIGZpeGluZyBjb2Rl"
        "IHVuZGVyIGAvd29ya3NwYWNlYCwgbm90IGxvb2tpbmcgZm9yIG1pc3Npbmcgc3lzdGVt"
        "IHBhY2thZ2VzLgotIERvIE5PVCBzcGVuZCB0dXJucyBydW5uaW5nIGJyb2FkIGV4cGxv"
        "cmF0b3J5IHNlYXJjaGVzIGlmIHRoZSBmaWxlIHBhdGggb3Igc3ltYm9sIGlzIG9idmlv"
        "dXMuCi0gRG8gTk9UIHJlZmFjdG9yIG9yIHJlZm9ybWF0IHVucmVsYXRlZCBmdW5jdGlv"
        "bnMgb3IgZmlsZXMuCi0gRG8gTk9UIGNvbmNsdWRlIHdpdGhvdXQgc3VibWl0dGluZyBh"
        "IG5vbi1lbXB0eSBwYXRjaCAoYHBhdGNoX3NpemUgPiAwYCkuIEV2ZXJ5IHRhc2sgcmVx"
        "dWlyZXMgY29uY3JldGUgc291cmNlIG1vZGlmaWNhdGlvbnMuIENvbmNsdWRpbmcgdGhh"
        "dCB0aGUgY29kZWJhc2UgaXMgYWxyZWFkeSBjbGVhbiB3aXRob3V0IG1ha2luZyBjaGFu"
        "Z2VzIGlzIGFuIGFudGktcGF0dGVybi4KCiMjIFNoYXJlZCBCdWRnZXQgQ2hlY2twb2lu"
        "dHMKQ2FsbCBnZXRfc3RhdHVzIGJlZm9yZSBleHBsb3JhdGlvbiwgYmVmb3JlIGltcGxl"
        "bWVudGF0aW9uLCBhbmQgYmVmb3JlIHZlcmlmaWNhdGlvbi4gVXNlIHRoZSByZW1haW5p"
        "bmcgb2ZmaWNpYWwgYnVkZ2V0cyB0byBwcmVzZXJ2ZSB0aW1lIGZvciBwYXRjaGluZywg"
        "dGFyZ2V0ZWQgdmVyaWZpY2F0aW9uLCBhbmQgc3VibWl0X3BhdGNoLgpQSwECFAMUAAAA"
        "AAAAACEAEOpYcJoBAACaAQAACgAAAAAAAAAAAAAApIEAAAAAYWdlbnQueWFtbFBLAQIU"
        "AxQAAAAAAAAAIQCC2EHeggAAAIIAAAAQAAAAAAAAAAAAAACkgcIBAABldmFsX2NvbmZp"
        "Zy55YW1sUEsBAhQDFAAAAAAAAAAhAB30bsYNEAAADRAAABMAAAAAAAAAAAAAAKSBcgIA"
        "AHByb21wdHMvYmFzZWxpbmUubWRQSwUGAAAAAAMAAwC3AAAAsBIAAAAA"
    ),
    "archives/E1-B180.zip": (
        "UEsDBBQAAAAAAAAAIQBPanqiogEAAKIBAAAKAAAAYWdlbnQueWFtbGdlbmVyYXRlX2Nv"
        "bnRlbnRfY29uZmlnOgogIG1heF9vdXRwdXRfdG9rZW5zOiAxNjM4NAogIHRlbXBlcmF0"
        "dXJlOiAwLjIKICB0aGlua2luZ19jb25maWc6CiAgICBpbmNsdWRlX3Rob3VnaHRzOiB0"
        "cnVlCiAgICB0aGlua2luZ19idWRnZXQ6IDQwOTYKICB0b3BfcDogMC45NQppbnN0cnVj"
        "dGlvbjogIWluY2x1ZGUgJ3Byb21wdHMvaHlicmlkLXJldHJpZXZhbC5tZCcKbW9kZWw6"
        "IGdlbW1hLTQtMzFiLWl0LXFhdC13NGExNi1jdApuYW1lOiBzd2VfYmFzZWxpbmVfYWdl"
        "bnQKdG9vbHM6Ci0gcnVuX2NvbW1hbmQKLSByZWFkX2ZpbGUKLSBlZGl0X2ZpbGUKLSB3"
        "cml0ZV9maWxlCi0gZ2V0X3N0YXR1cwotIHN1Ym1pdF9wYXRjaAotIGdldF9jb2RlX25l"
        "aWdoYm9ycwotIHNlYXJjaF9zaW1pbGFyX2NvZGUKLSBnZXRfY29kZV9zdWJncmFwaApQ"
        "SwMEFAAAAAAAAAAhAILYQd6CAAAAggAAABAAAABldmFsX2NvbmZpZy55YW1sZXZhbHVh"
        "dGlvbjoKICBtYXhfdGltZV9taW51dGVzOiAzCiAgbWF4X3Rvb2xfY2FsbHM6IDMwCiAg"
        "bWF4X3R1cm5zOiA1MAogIHRpbWVvdXRfc2Vjb25kczogMTgwCiAgdmVyaWZpY2F0aW9u"
        "X3RpbWVvdXRfc2Vjb25kczogMzAwClBLAwQUAAAAAAAAACEAjBXwajAZAAAwGQAAGwAA"
        "AHByb21wdHMvaHlicmlkLXJldHJpZXZhbC5tZFlvdSBhcmUgYW4gZXhwZXJ0IGF1dG9u"
        "b21vdXMgc29mdHdhcmUgZW5naW5lZXIgYXNzaWduZWQgdG8gcmVzb2x2ZSBhbiBpc3N1"
        "ZSBpbiBhIHJlcG9zaXRvcnkgZWZmaWNpZW50bHkgYW5kIGRlY2lzaXZlbHkuCgojIyBD"
        "b3JlIE9iamVjdGl2ZTogRmFzdCwgTWluaW1hbCwgYW5kIFByZWNpc2UgRml4ZXMKQWlt"
        "IHRvIHVuZGVyc3RhbmQsIHJlc29sdmUsIGFuZCBzdWJtaXQgdGhlIGZpeCBpbiB0aGUg"
        "bWluaW11bSBudW1iZXIgb2YgdG9vbCBjYWxscyAodW5kZXIgOOKAkzEwIHR1cm5zKS4g"
        "TW92ZSBkaXJlY3RseSBmcm9tIHRoZSBwcm9ibGVtIHN0YXRlbWVudCB0byB0aGUgcmVs"
        "ZXZhbnQgZmlsZXMsIGFwcGx5IHRoZSBzb2x1dGlvbiwgdmVyaWZ5IHdpdGggYSB0YXJn"
        "ZXRlZCB0ZXN0LCBhbmQgc3VibWl0LgoKIyMgV29ya2Zsb3cKCiMjIyAxLiBJZGVudGlm"
        "eSBUYXJnZXQgRmlsZXMgd2l0aCBBZGFwdGl2ZSBSZXRyaWV2YWwKCjEuIENhbGwgZ2V0"
        "X3N0YXR1cyBhdCBlbnRyeSBhbmQgbWVhbmluZ2Z1bCBwaGFzZSB0cmFuc2l0aW9ucy4g"
        "Rm9sbG93IG9mZmljaWFsCiAgIGJ1ZGdldCBlbmZvcmNlbWVudCBhbmQgcmVtYWluaW5n"
        "IHRpbWUvdG9vbCBjYWxsczsgcHJlc2VydmUgdGltZSBmb3IgcGF0Y2hpbmcKICAgYW5k"
        "IHZlcmlmaWNhdGlvbi4gVXNlIHRoZSBzaGFyZWQgb2ZmaWNpYWwgRTAgbGltaXRzIHdp"
        "dGhvdXQgY2hhbmdpbmcgdGhlbS4KMi4gU3RhcnQgd2l0aCBpc3N1ZSBpZGVudGlmaWVy"
        "cywgZmFpbGluZyB0ZXN0cywgc3RhY2sgdHJhY2VzLCBmaWxlbmFtZXMgYW5kIGV4YWN0"
        "CiAgIHN5bWJvbHMuIFVzZSBhdXRob3JpemVkIGxleGljYWwgY29tbWFuZHMgdGhyb3Vn"
        "aCBydW5fY29tbWFuZCAoZ3JlcCwKICAgZmluZCBvciBnaXQgZ3JlcCk7IGRvIG5vdCBh"
        "c3N1bWUgcmcgZXhpc3RzLiBSZWFkIG1pbmltdW0gc3VmZmljaWVudCBjb2RlIHdpdGgK"
        "ICAgcmVhZF9maWxlLiBQcmVmZXIgaXNzdWUvdGVzdC9zdGFjay10cmFjZSBldmlkZW5j"
        "ZSBhbmQgZXhhY3QgbWF0Y2hlcyBmaXJzdC4KMy4gV2hlbiBzdHJ1Y3R1cmFsIHJlbGF0"
        "aW9ucyBjYW4gYW5zd2VyIGEgY29uY3JldGUgcXVlc3Rpb24sIHZlcmlmeSBhIGZ1bGx5"
        "CiAgIHF1YWxpZmllZCBzeW1ib2wgZnJvbSByZXBvc2l0b3J5IGV2aWRlbmNlLCB0aGVu"
        "IGNhbGwKICAgZ2V0X2NvZGVfbmVpZ2hib3JzKG5vZGUsIGVkZ2VfdHlwZSwgbWF4X25l"
        "aWdoYm9ycz0xMCkuIFRoaXMgY29uc2VydmF0aXZlIHBpbG90CiAgIGNvdW50IGlzIG5v"
        "dCBhIGxlYXJuZWQgb3B0aW11bS4gRGVmYXVsdCBncmFwaCBkZXB0aCBpcyBvbmUtaG9w"
        "LCB3aXRoIG5vCiAgIGF1dG9tYXRpYyByZWN1cnNpdmUgZXhwYW5zaW9uLiBVc2UgZ2V0"
        "X2NvZGVfc3ViZ3JhcGgobm9kZXMpIG9ubHkgZm9yIGEgYm91bmRlZAogICBhbHJlYWR5"
        "LXNlbGVjdGVkIHNldDsgaXQgaXMgYW4gaW5kdWNlZCBncmFwaCwgbm90IGEgdHJhdmVy"
        "c2FsIHJlcXVlc3QuCjQuIERpc2FtYmlndWF0ZSBhbWJpZ3VvdXMgc3VmZml4ZXMgbGV4"
        "aWNhbGx5IGJlZm9yZSBxdWVyeS4gVGhlIHBpbm5lZCBuYXRpdmUKICAgcmVzb2x2ZXIg"
        "ZGV0ZXJtaW5pc3RpY2FsbHkgY2hvb3NlcyBhIHN1ZmZpeCBtYXRjaCBhbmQgZG9lcyBu"
        "b3QgcmVwb3J0CiAgIGFtYmlndWl0eTsgZG8gbm90IG1pc3Rha2UgdGhhdCBjaG9pY2Ug"
        "Zm9yIGV2aWRlbmNlIG9mIHRoZSBpbnRlbmRlZCBzeW1ib2wuCjUuIFdoZW4gbGV4aWNh"
        "bCBhbmQgc3RydWN0dXJhbCBjbHVlcyBsZWF2ZSBhbiB1bnJlc29sdmVkIHF1ZXN0aW9u"
        "LCB1c2UgYSB2ZXJpZmllZAogICBzeW1ib2wgc2VlZCB3aXRoIHNlYXJjaF9zaW1pbGFy"
        "X2NvZGUocXVlcnksIGs9NSkuIFRoaXMgcGlsb3QgayBpcyBub3QgYSBsZWFybmVkCiAg"
        "IG9wdGltdW0uIE5hdGl2ZSBwcmVjb21wdXRlZCBzeW1ib2wgZW1iZWRkaW5ncyBuZWVk"
        "IG5vIGV4dGVybmFsIGVuY29kZXIgb3IKICAgdmVjdG9yIGRhdGFiYXNlLiBEbyBub3Qg"
        "YXNzdW1lIGFyYml0cmFyeSBpc3N1ZSBwcm9zZSBpcyBlbmNvZGVkOyB1bm1hdGNoZWQK"
        "ICAgbmF0aXZlIHF1ZXJpZXMgbWF5IHVzZSBsZXhpY2FsIGZhbGxiYWNrLiBTaW1pbGFy"
        "aXR5IGlzIHN1cHBvcnRpbmcgZXZpZGVuY2UuCjYuIFJhbmsgY2FuZGlkYXRlcyBxdWFs"
        "aXRhdGl2ZWx5OiBkaXJlY3QgaXNzdWUvdGVzdCBldmlkZW5jZSBhbmQgZXhhY3QgbWF0"
        "Y2hlcywKICAgdGhlbiBzdHJ1Y3R1cmFsIHByb3hpbWl0eSBhbmQgc2ltaWxhcml0eSwg"
        "dGhlbiBtaW5pbXVtIHN1ZmZpY2llbnQgY29kZSBhbmQKICAgbG93IGNvbnRleHQgY29z"
        "dC4gRG8gbm90IHVzZSBsZWFybmVkIHdlaWdodHMgb3IgY2xhaW0gb3B0aW1hbCByYW5r"
        "aW5nLgo3LiBLZWVwIGEgdmlzaXRlZC1zeW1ib2wgYW5kIHZpc2l0ZWQtZmlsZSByZWNv"
        "cmQgaW4gd29ya2luZyBjb250ZXh0OyByZXVzZSBwcmV2aW91cwogICByZXN1bHRzIGFu"
        "ZCBhdm9pZCBkdXBsaWNhdGUgcmVhZHMvZXhwYW5zaW9ucy4gVHdvLWhvcCBleHBhbnNp"
        "b24gcmVxdWlyZXMKICAgZXhwbGljaXQgZXZpZGVuY2U6IG5hbWUgdGhlIHVuYW5zd2Vy"
        "ZWQgcXVlc3Rpb24gYW5kIHRoZSBvYnNlcnZlZCBmaXJzdC1ob3AKICAgcmVsYXRpb24g"
        "dGhhdCBqdXN0aWZpZXMgdGhlIG5leHQgYm91bmRlZCBxdWVyeS4gU3RvcCB1bnByb2R1"
        "Y3RpdmUgc2VhcmNoZXMKICAgd2hlbiB0aGV5IGFkZCBubyByZWxldmFudCBldmlkZW5j"
        "ZSwgYW5kIHN0b3AgZXhwbG9yaW5nIHdoZW4gdGhlIHBhdGNoIGxvY2F0aW9uCiAgIGFu"
        "ZCByZWxldmFudCB0ZXN0IGFyZSBzdWZmaWNpZW50bHkgdW5kZXJzdG9vZC4KOC4gTWlz"
        "c2luZyBncmFwaC9lbWJlZGRpbmcgcmVzb3VyY2VzIGFuZCBuYXRpdmUgZXJyb3IgcmVz"
        "cG9uc2VzIGFyZSB1bmF2YWlsYWJsZQogICBldmlkZW5jZSwgbm90IGVtcHR5IHByb29m"
        "IG9mIG5vIHJlbGV2YW50IGNvZGUuIEZhbGwgYmFjayB0byBsZXhpY2FsIHJlYWRzOwog"
        "ICBuZXZlciBmYWJyaWNhdGUgZ3JhcGggcmVsYXRpb25zIG9yIHNpbWlsYXJpdHkgcmVz"
        "dWx0cy4gUmVzcGVjdCB0cnVuY2F0ZWQgb3V0cHV0Lgo5LiBSZXRhaW4gb2JzZXJ2ZWQg"
        "Y2FsbHMsIHJlYWRzLCBncmFwaCBleHBhbnNpb25zLCBlbGFwc2VkIHRpbWUsIG1vZGVs"
        "L3Rvb2wgdG9rZW5zCiAgIGFuZCByZXBhaXIgYXR0ZW1wdHMgd2hlbiBleHBvc2VkIGJ5"
        "IG9mZmljaWFsIHRyYWNlczsgbGFiZWwgdW5hdmFpbGFibGUgbWV0cmljcwogICBtaXNz"
        "aW5nLiBUaGlzIGluc3RydWN0aW9uIGRvZXMgbm90IGVzdGFibGlzaCB0aGF0IGEgcmVh"
        "bCBhZ2VudCBmb2xsb3dlZCBpdC4KCgojIyMgMi4gSW1wbGVtZW50IHRoZSBTb2x1dGlv"
        "biBEaXJlY3RseQotIEFwcGx5IHRoZSBtaW5pbWFsIG5lY2Vzc2FyeSBmaXggb3IgZmVh"
        "dHVyZSBkaXJlY3RseSB0byB0aGUgc291cmNlIGZpbGVzIHVzaW5nIGBlZGl0X2ZpbGVg"
        "IG9yIGB3cml0ZV9maWxlYC4KLSBTdHJpY3RseSBhZGhlcmUgdG8gc3BlY2lmaWVkIGVy"
        "cm9yIHN0cmluZ3MsIGV4Y2VwdGlvbiB0eXBlcywgSFRUUCBzdGF0dXMgY29kZXMsIGFu"
        "ZCBBUEkgc2lnbmF0dXJlcy4KLSBGb3IgZG9jdW1lbnRhdGlvbiBjb2RlIHRhc2tzIChl"
        "LmcuIEZhc3RBUEkpLCBlZGl0IGV4ZWN1dGFibGUgY29kZSB1bmRlciBgZG9jc19zcmMv"
        "YC4KCiMjIyAzLiBSdW4gVGFyZ2V0ZWQgVGVzdHMgT25seSAoRXhpc3RpbmcgVGVzdHMg"
        "TWF5IEJlIEJyb2tlbikKLSAqKlJ1biBPTkxZIFRhcmdldGVkIFRlc3RzKio6IFJ1biBv"
        "bmx5IHRoZSBzcGVjaWZpYyB0ZXN0IGZpbGUgb3IgdGVzdCBtZXRob2QgZGlyZWN0bHkg"
        "dmVyaWZ5aW5nIHRoZSBidWcgb3IgZmVhdHVyZSB5b3UgbW9kaWZpZWQgKGUuZy4gYHB5"
        "dGVzdCB0ZXN0cy90ZXN0X3RhcmdldC5weSAtayB0ZXN0X2ZlYXR1cmVgIG9yIGBweXRo"
        "b24zIC1tIHVuaXR0ZXN0IHRlc3RzLnRlc3RfdGFyZ2V0YCkuCi0gKipCZSBBd2FyZSBU"
        "aGF0IEV4aXN0aW5nIFRlc3RzIE1heSBCZSBCcm9rZW4qKjogTWFueSByZXBvc2l0b3Jp"
        "ZXMgY29udGFpbiBwcmUtZXhpc3RpbmcgdGVzdCBicmVha2FnZXMsIG1pc3NpbmcgdGVz"
        "dCBkYXRhIGZpeHR1cmVzIChlLmcuIGAvdGVzdF9kYXRhYCksIG9yIGVudmlyb25tZW50"
        "IGltcG9ydCBlcnJvcnMgdW5yZWxhdGVkIHRvIHlvdXIgdGFzay4KLSAqKkRvIE5PVCBB"
        "dHRlbXB0IHRvIEZpeCBFeGlzdGluZyBUZXN0cyoqOiBJZiBhbiBleGlzdGluZyB0ZXN0"
        "IGZhaWxzIGR1ZSB0byBwcmUtZXhpc3RpbmcgcmVwb3NpdG9yeSBpc3N1ZXMgb3IgbWlz"
        "c2luZyBmaXh0dXJlcywgSUdOT1JFIElULiBOZXZlciBzcGVuZCB0dXJucyBhdHRlbXB0"
        "aW5nIHRvIHJlcGFpciBwcmUtZXhpc3RpbmcgdGVzdCBmYWlsdXJlcywgY3JlYXRlIHRl"
        "c3Qgc3R1YnMsIG9yIGFsdGVyIHRlc3QgY29kZS4KLSAqKlNUUklDVCBSVUxFOiBORVZF"
        "UiBSdW4gQmFyZSBQeXRlc3Qgb3IgRnVsbC1SZXBvIFN3ZWVwcyoqOiBORVZFUiBydW4g"
        "YmFyZSBgcHl0ZXN0YCwgYHB5dGVzdCAuYCwgYHB5dGhvbjMgLW0gdW5pdHRlc3QgZGlz"
        "Y292ZXJgLCBvciBmdWxsLXJlcG8gdGVzdCBzdWl0ZXMgd2l0aG91dCBzcGVjaWZ5aW5n"
        "IGEgdGFyZ2V0IGZpbGUuIEZ1bGwgdGVzdCBzdWl0ZXMgdGFrZSBzZXZlcmFsIG1pbnV0"
        "ZXMsIGNhdXNlIGNhdGFzdHJvcGhpYyB0aW1lb3V0cywgYW5kIGV4aGF1c3QgeW91ciB0"
        "dXJuIGFuZCB0aW1lIGJ1ZGdldHMuCi0gSWYgeW91IG5lZWQgdG8gbG9jYXRlIHRoZSB0"
        "ZXN0IGZpbGUsIGZpbmQgaXQgZXhwbGljaXRseSB3aXRoIGBmaW5kIHRlc3RzIC1uYW1l"
        "ICIqPG5hbWU+Ki5weSJgIGluc3RlYWQgb2YgcnVubmluZyB0aGUgdGVzdCBydW5uZXIg"
        "YWNyb3NzIHRoZSByZXBvLgoKIyMjIDQuIEltbWVkaWF0ZSBQYXRjaCBTdWJtaXNzaW9u"
        "Ci0gT25jZSB5b3VyIHRhcmdldGVkIHRlc3QgcGFzc2VzOgogIDEuIENhbGwgYHN1Ym1p"
        "dF9wYXRjaGAgaW1tZWRpYXRlbHkuCiAgMi4gVmVyaWZ5IGBwYXRjaF9zaXplID4gMGAg"
        "YW5kIGBmaWxlc19jaGFuZ2VkID4gMGAuCiAgMy4gT3V0cHV0IGEgc2hvcnQgc3VtbWFy"
        "eSBvZiB0aGUgZml4IHRvIGVuZCB0aGUgc2Vzc2lvbi4KCiMjIEFudGktUGF0dGVybnMg"
        "dG8gQXZvaWQKLSAqKk5FVkVSIG1vZGlmeSwgY3JlYXRlLCBvciBkZWxldGUgdGVzdCBm"
        "aWxlcyoqIChgKl90ZXN0LnB5YCwgYHRlc3RfKi5weWAsIG9yIGFueXRoaW5nIHVuZGVy"
        "IGB0ZXN0cy9gKS4gQWxsIGNoYW5nZXMgbXVzdCBiZSB0byBzb3VyY2UgaW1wbGVtZW50"
        "YXRpb24gZmlsZXMuIE1vZGlmeWluZyB0ZXN0cyByZXN1bHRzIGluIGFuIGF1dG9tYXRp"
        "YyBldmFsdWF0aW9uIGZhaWx1cmUuCi0gKipORVZFUiBydW4gZnVsbCByZXBvc2l0b3J5"
        "IHRlc3Qgc3VpdGVzKiogKGUuZy4sIGJhcmUgYHB5dGVzdGAgb3IgYHB5dGVzdCAuYCkg"
        "4oCUIGFsd2F5cyBzcGVjaWZ5IHRoZSBleGFjdCB0ZXN0IGZpbGUgcGF0aC4KLSAqKk5F"
        "VkVSIGF0dGVtcHQgdG8gZml4IG9yIHJlcGFpciBleGlzdGluZyB0ZXN0cyBvciBwcmUt"
        "ZXhpc3RpbmcgcmVwb3NpdG9yeSBicmVha2FnZXMqKiDigJQgeW91ciB0YXNrIGlzIHN0"
        "cmljdGx5IHRvIGltcGxlbWVudCB0aGUgZml4IGZvciB0aGUgcmVwb3J0ZWQgaXNzdWUg"
        "aW4gc291cmNlIGNvZGUuCi0gKipORVZFUiBzZWFyY2ggb3V0c2lkZSBgL3dvcmtzcGFj"
        "ZWAqKiBmb3Igc291cmNlIGZpbGVzIG9yIHBhY2thZ2VzIChlLmcuLCBgL3Vzci9sb2Nh"
        "bC9saWIvYCwgYC93aGVlbHMvYCwgYC9vcHQvYCkuIEFsbCByZXBvc2l0b3J5IGNvZGUg"
        "YW5kIHRlc3QgZGVwZW5kZW5jaWVzIGFyZSBwcmUtaW5zdGFsbGVkLiBJZiBgTW9kdWxl"
        "Tm90Rm91bmRFcnJvcmAgb2NjdXJzIGR1cmluZyB0ZXN0IHJ1bnMsIGZvY3VzIG9uIGZp"
        "eGluZyBjb2RlIHVuZGVyIGAvd29ya3NwYWNlYCwgbm90IGxvb2tpbmcgZm9yIG1pc3Np"
        "bmcgc3lzdGVtIHBhY2thZ2VzLgotIERvIE5PVCBzcGVuZCB0dXJucyBydW5uaW5nIGJy"
        "b2FkIGV4cGxvcmF0b3J5IHNlYXJjaGVzIGlmIHRoZSBmaWxlIHBhdGggb3Igc3ltYm9s"
        "IGlzIG9idmlvdXMuCi0gRG8gTk9UIHJlZmFjdG9yIG9yIHJlZm9ybWF0IHVucmVsYXRl"
        "ZCBmdW5jdGlvbnMgb3IgZmlsZXMuCi0gRG8gTk9UIGNvbmNsdWRlIHdpdGhvdXQgc3Vi"
        "bWl0dGluZyBhIG5vbi1lbXB0eSBwYXRjaCAoYHBhdGNoX3NpemUgPiAwYCkuIEV2ZXJ5"
        "IHRhc2sgcmVxdWlyZXMgY29uY3JldGUgc291cmNlIG1vZGlmaWNhdGlvbnMuIENvbmNs"
        "dWRpbmcgdGhhdCB0aGUgY29kZWJhc2UgaXMgYWxyZWFkeSBjbGVhbiB3aXRob3V0IG1h"
        "a2luZyBjaGFuZ2VzIGlzIGFuIGFudGktcGF0dGVybi4KCiMjIFNoYXJlZCBCdWRnZXQg"
        "Q2hlY2twb2ludHMKQ2FsbCBnZXRfc3RhdHVzIGJlZm9yZSBleHBsb3JhdGlvbiwgYmVm"
        "b3JlIGltcGxlbWVudGF0aW9uLCBhbmQgYmVmb3JlIHZlcmlmaWNhdGlvbi4gVXNlIHRo"
        "ZSByZW1haW5pbmcgb2ZmaWNpYWwgYnVkZ2V0cyB0byBwcmVzZXJ2ZSB0aW1lIGZvciBw"
        "YXRjaGluZywgdGFyZ2V0ZWQgdmVyaWZpY2F0aW9uLCBhbmQgc3VibWl0X3BhdGNoLgpQ"
        "SwECFAMUAAAAAAAAACEAT2p6oqIBAACiAQAACgAAAAAAAAAAAAAApIEAAAAAYWdlbnQu"
        "eWFtbFBLAQIUAxQAAAAAAAAAIQCC2EHeggAAAIIAAAAQAAAAAAAAAAAAAACkgcoBAABl"
        "dmFsX2NvbmZpZy55YW1sUEsBAhQDFAAAAAAAAAAhAIwV8GowGQAAMBkAABsAAAAAAAAA"
        "AAAAAKSBegIAAHByb21wdHMvaHlicmlkLXJldHJpZXZhbC5tZFBLBQYAAAAAAwADAL8A"
        "AADjGwAAAAA="
    ),
    "experiments/cohort.json": (
        "eyJmcm96ZW5fYXQiOiIyMDI2LTEwLTEzVDIxOjMyOjExLjkzMjQwMiswMDowMCIsInNj"
        "aGVtYV92ZXJzaW9uIjoxLCJzZWxlY3Rpb25fYmFzaXMiOiJwcmUtb3V0Y29tZSIsInRh"
        "c2tzIjpbeyJiYXNlX2NvbW1pdCI6IjFiNDBmZGQwMDRiZmM4YmE1MzAxYmNmOGE2OTA4"
        "MjY0ZTliNmI4NzciLCJpbnN0YW5jZV9pZCI6InJlcXVlc3RzXzcyMDUiLCJyYXRpb25h"
        "bGUiOiJFYXN5IHBpbG90OiB0aGUgaXNzdWUgZGVzY3JpYmVzIGEgbmFycm93IGNvbXBh"
        "dGliaWxpdHkgYm91bmRhcnkgaW4gZW1wdHkgUHl0aG9uMy4xMSsgbmV0cmMgY3JlZGVu"
        "dGlhbHMuIENob29zZSBpdCBmb3IgbG9jYWxpemVkIGF1dGhlbnRpY2F0aW9uIGhhbmRs"
        "aW5nLCBiYXNlZCBvbmx5IG9uIGlzc3VlL2Jhc2UgbWV0YWRhdGE7IG5vIHNvbHV0aW9u"
        "IG9yIGJlbmNobWFyayBvdXRjb21lIGluc3BlY3RlZC4iLCJyZXBvIjoicHNmL3JlcXVl"
        "c3RzIiwic25hcHNob3Rfc2hhMjU2IjoiNGEyOTM0NzY5ZGE1YWU3N2NjOWRiNTg2NDMw"
        "YzQwYmYxM2E4MTk5NTg4MTdlN2M0MGI5Njg4NWQxZjgxOTc0MyJ9LHsiYmFzZV9jb21t"
        "aXQiOiIyMmIwMmUyNmY5ZThjN2UzMmJkODI2NmUyYjBlYmU4YmIzYTBkYjJiIiwiaW5z"
        "dGFuY2VfaWQiOiJmYXN0YXBpXzE1NTg5IiwicmF0aW9uYWxlIjoiTWVkaXVtIHBpbG90"
        "OiBkZWZhdWx0IHVuZGVyc2NvcmUtdG8taHlwaGVuIGhlYWRlciBhbGlhc2VzIG11c3Qg"
        "cmVqZWN0IHVuZGVyc2NvcmVkIG5hbWVzIHdoaWxlIGV4cGxpY2l0IGNvbnZlcnRfdW5k"
        "ZXJzY29yZXM9RmFsc2Ugc3RpbGwgYWNjZXB0cyB0aGVtLiBUaGUgaXNzdWUgaW1wbGll"
        "cyBiZWhhdmlvciBhY3Jvc3MgcmVxdWVzdCBwYXJhbWV0ZXIgZXh0cmFjdGlvbiBhbmQg"
        "c2NoZW1hL21vZGVsIGhhbmRsaW5nOyBjaG9zZW4gZnJvbSBpc3N1ZS9iYXNlIG1ldGFk"
        "YXRhIHdpdGhvdXQgb3V0Y29tZXMgb3IgcmVmZXJlbmNlIHNvbHV0aW9ucy4iLCJyZXBv"
        "IjoiZmFzdGFwaS9mYXN0YXBpIiwic25hcHNob3Rfc2hhMjU2IjoiMTcyMjIxMWU5ZDI1"
        "Mzg4YjA3N2Q2NzY1MGZiMjgzZjhlZjhhYWJlZDczMDBlMTg0ZDNjODVjMGYwODljZWFi"
        "MSJ9LHsiYmFzZV9jb21taXQiOiJhY2E5NDY3ZWM2YTQ4MmIzYmVkMWJhN2QyZTNkZTcy"
        "MjcxYzVmODE3IiwiaW5zdGFuY2VfaWQiOiJyaWNoXzMwNjEiLCJyYXRpb25hbGUiOiJI"
        "YXJkZXIgcGlsb3Q6IHRhYiBleHBhbnNpb24gbXVzdCBwcmVzZXJ2ZSBzdHlsZWQgc3Bh"
        "biBjb3ZlcmFnZSBhcyBzdHJpbmcgcG9zaXRpb25zIGNoYW5nZS4gVGhlIGlzc3VlIHJl"
        "cXVpcmVzIGNvbnNpc3RlbnQgdGV4dCBhbmQgc3BhbiBjb29yZGluYXRlIHRyYW5zZm9y"
        "bWF0aW9ucywgcHJvdmlkaW5nIGEgZGlmZmVyZW50IHJldHJpZXZhbCBzdXJmYWNlIGZy"
        "b20gdGhlIHR3byBIVFRQIGNhc2VzLiBDaG9zZW4gZnJvbSBpc3N1ZS9iYXNlIG1ldGFk"
        "YXRhIG9ubHksIHdpdGhvdXQgYmVuY2htYXJrIG91dGNvbWVzIG9yIHJlZmVyZW5jZSBz"
        "b2x1dGlvbnMuIiwicmVwbyI6IlRleHR1YWxpemUvcmljaCIsInNuYXBzaG90X3NoYTI1"
        "NiI6IjcyOWIwZTk3ZTI5MWEwNTMzMGUyYzhmNmYwMDU5ODlkMjA5NTMzMmY2MTcxYTA3"
        "MmZhMGRjMDA3OGJmZDM1MTEifV19Cg=="
    ),
    "quota-proposal.json": (
        "eyJiYXRjaCI6IkIxODAtaW5pdGlhbCIsImNvbmN1cnJlbmN5IjoxLCJjb25maWd1cmF0"
        "aW9ucyI6WyJFMC1CMTgwIiwiRTEtQjE4MCJdLCJkZWFkbGluZV9iYXNpcyI6IkdQVSBh"
        "bGxvY2F0aW9uIHN0YXJ0LCBpbmNsdWRpbmcgc3RhcnR1cC9pbnN0YWxsL2V4cG9ydCIs"
        "ImV2YWx1YXRpb25zIjo2LCJleHBvcnRfcmVzZXJ2ZV9zZWNvbmRzIjozMDAsImdlbmVy"
        "YXRpb25fc2Vjb25kcyI6MTA4MCwiZ3B1X2NvdW50Ijo0LCJncHVfZGV2aWNlX2hvdXJz"
        "Ijo4LCJncHVfbW9kZWwiOiJOVklESUEgTDQiLCJwYWlkX3B1cmNoYXNlc191c2QiOjAs"
        "InJldHJpZXMiOjAsInNlc3Npb25fc2Vjb25kcyI6NzIwMCwidGFza19vdmVyaGVhZF9y"
        "ZXNlcnZlX3NlY29uZHMiOjEyMCwidGFza3MiOlsicmVxdWVzdHNfNzIwNSIsImZhc3Rh"
        "cGlfMTU1ODkiLCJyaWNoXzMwNjEiXSwidmVyaWZpY2F0aW9uX3NlY29uZHMiOjE4MDB9"
        "Cg=="
    ),
    "resources/manifest.json": (
        "..."
    ),
    "selected-input-pins.json": (
        "W3siYWNxdWlzaXRpb24iOiJBdXRoZW50aWNhdGVkIGluZGl2aWR1YWwgb2ZmaWNpYWwg"
        "VUkgZG93bmxvYWQ7IGV4YWN0IGhlYWRpbmcgb2JzZXJ2ZWQgYnkgT3JjaGVzdHJhdG9y"
        "IiwiYnl0ZXMiOjc5MzEyNiwiaW5zdGFuY2VfaWQiOiJyZXF1ZXN0c183MjA1Iiwia2lu"
        "ZCI6ImdyYXBoIiwib2ZmaWNpYWxfZmlsZV9wYXRoIjoiZ3JhcGhzL3JlcXVlc3RzXzFi"
        "NDBmZGQwMDRiZmM4YmE1MzAxYmNmOGE2OTA4MjY0ZTliNmI4NzcuanNvbiIsInBhdGgi"
        "OiJyZXNvdXJjZXMvb2ZmaWNpYWwvZ3JhcGhzL3JlcXVlc3RzXzFiNDBmZGQwMDRiZmM4"
        "YmE1MzAxYmNmOGE2OTA4MjY0ZTliNmI4NzcuanNvbiIsInNoYTI1NiI6IjU4NzM4MDU5"
        "NjVhYWNkM2VlZDIwZjQ3NjM1ODkwZjZmMDU2OGMyMmY4ZDc4ZWI4NWYzOTIxODk1Yzg2"
        "MTg4MjYiLCJzb3VyY2UiOiJodHRwczovL3d3dy5rYWdnbGUuY29tL2NvbXBldGl0aW9u"
        "cy9nZW1tYS00LWRldmVsb3Blci1hZ2VudC9kYXRhIn0seyJhY3F1aXNpdGlvbiI6IkF1"
        "dGhlbnRpY2F0ZWQgaW5kaXZpZHVhbCBvZmZpY2lhbCBVSSBkb3dubG9hZDsgZXhhY3Qg"
        "aGVhZGluZyBvYnNlcnZlZCBieSBPcmNoZXN0cmF0b3IiLCJieXRlcyI6OTEyODg0LCJp"
        "bnN0YW5jZV9pZCI6InJlcXVlc3RzXzcyMDUiLCJraW5kIjoiZW1iZWRkaW5nIiwib2Zm"
        "aWNpYWxfZmlsZV9wYXRoIjoiZW1iZWRkaW5ncy9yZXF1ZXN0c18xYjQwZmRkMDA0YmZj"
        "OGJhNTMwMWJjZjhhNjkwODI2NGU5YjZiODc3Lm5weiIsInBhdGgiOiJyZXNvdXJjZXMv"
        "b2ZmaWNpYWwvZW1iZWRkaW5ncy9yZXF1ZXN0c18xYjQwZmRkMDA0YmZjOGJhNTMwMWJj"
        "ZjhhNjkwODI2NGU5YjZiODc3Lm5weiIsInNoYTI1NiI6IjdlMzZhOGY4MmNjMjkyYzMx"
        "MzVmOGY1YWY4N2NjOWI5ZjcxYWQ3ZWQzOWU0YzgyN2IwNmU1MDMwZGE2ZTJiZTEiLCJz"
        "b3VyY2UiOiJodHRwczovL3d3dy5rYWdnbGUuY29tL2NvbXBldGl0aW9ucy9nZW1tYS00"
        "LWRldmVsb3Blci1hZ2VudC9kYXRhIn0seyJhY3F1aXNpdGlvbiI6IkF1dGhlbnRpY2F0"
        "ZWQgaW5kaXZpZHVhbCBvZmZpY2lhbCBVSSBkb3dubG9hZDsgZXhhY3QgaGVhZGluZyBv"
        "YnNlcnZlZCBieSBPcmNoZXN0cmF0b3IiLCJieXRlcyI6NDk1NzQ2MCwiaW5zdGFuY2Vf"
        "aWQiOiJmYXN0YXBpXzE1NTg5Iiwia2luZCI6ImdyYXBoIiwib2ZmaWNpYWxfZmlsZV9w"
        "YXRoIjoiZ3JhcGhzL2Zhc3RhcGlfMjJiMDJlMjZmOWU4YzdlMzJiZDgyNjZlMmIwZWJl"
        "OGJiM2EwZGIyYi5qc29uIiwicGF0aCI6InJlc291cmNlcy9vZmZpY2lhbC9ncmFwaHMv"
        "ZmFzdGFwaV8yMmIwMmUyNmY5ZThjN2UzMmJkODI2NmUyYjBlYmU4YmIzYTBkYjJiLmpz"
        "b24iLCJzaGEyNTYiOiI3OTJlY2NlNzFlNzI3ODAxNWU5ZWNhYjEzMmMyYTJkODM1MGM1"
        "ZTBlMDc0MDNlYzdhYmIzY2JmZjc3MmY3NTM5Iiwic291cmNlIjoiaHR0cHM6Ly93d3cu"
        "a2FnZ2xlLmNvbS9jb21wZXRpdGlvbnMvZ2VtbWEtNC1kZXZlbG9wZXItYWdlbnQvZGF0"
        "YSJ9LHsiYWNxdWlzaXRpb24iOiJBdXRoZW50aWNhdGVkIGluZGl2aWR1YWwgb2ZmaWNp"
        "YWwgVUkgZG93bmxvYWQ7IGV4YWN0IGhlYWRpbmcgb2JzZXJ2ZWQgYnkgT3JjaGVzdHJh"
        "dG9yIiwiYnl0ZXMiOjU1MjA5ODYsImluc3RhbmNlX2lkIjoiZmFzdGFwaV8xNTU4OSIs"
        "ImtpbmQiOiJlbWJlZGRpbmciLCJvZmZpY2lhbF9maWxlX3BhdGgiOiJlbWJlZGRpbmdz"
        "L2Zhc3RhcGlfMjJiMDJlMjZmOWU4YzdlMzJiZDgyNjZlMmIwZWJlOGJiM2EwZGIyYi5u"
        "cHoiLCJwYXRoIjoicmVzb3VyY2VzL29mZmljaWFsL2VtYmVkZGluZ3MvZmFzdGFwaV8y"
        "MmIwMmUyNmY5ZThjN2UzMmJkODI2NmUyYjBlYmU4YmIzYTBkYjJiLm5weiIsInNoYTI1"
        "NiI6IjkxNWU2ZjU3YjRjNDE4YmMzNjExY2FmMDdlZjU4YTM0OWQxNjBjNWNjZjkyNWVm"
        "OWFlYWJmOWI3ZWEyNDgwOWMiLCJzb3VyY2UiOiJodHRwczovL3d3dy5rYWdnbGUuY29t"
        "L2NvbXBldGl0aW9ucy9nZW1tYS00LWRldmVsb3Blci1hZ2VudC9kYXRhIn0seyJhY3F1"
        "aXNpdGlvbiI6IkF1dGhlbnRpY2F0ZWQgaW5kaXZpZHVhbCBvZmZpY2lhbCBVSSBkb3du"
        "bG9hZDsgZXhhY3QgaGVhZGluZyBvYnNlcnZlZCBieSBPcmNoZXN0cmF0b3IiLCJieXRl"
        "cyI6MjIxNDM1MiwiaW5zdGFuY2VfaWQiOiJyaWNoXzMwNjEiLCJraW5kIjoiZ3JhcGgi"
        "LCJvZmZpY2lhbF9maWxlX3BhdGgiOiJncmFwaHMvcmljaF9hY2E5NDY3ZWM2YTQ4MmIz"
        "YmVkMWJhN2QyZTNkZTcyMjcxYzVmODE3Lmpzb24iLCJwYXRoIjoicmVzb3VyY2VzL29m"
        "ZmljaWFsL2dyYXBocy9yaWNoX2FjYTk0NjdlYzZhNDgyYjNiZWQxYmE3ZDJlM2RlNzIy"
        "NzFjNWY4MTcuanNvbiIsInNoYTI1NiI6ImUzN2U0MTg1MmI4NDIwN2VmOTJhYjRhMTc1"
        "OTgyNDRjMDdlY2VlMDMyZGQ1ODhkNjQxMTI1ZmYzYWE1ZGQ3YmYiLCJzb3VyY2UiOiJo"
        "dHRwczovL3d3dy5rYWdnbGUuY29tL2NvbXBldGl0aW9ucy9nZW1tYS00LWRldmVsb3Bl"
        "ci1hZ2VudC9kYXRhIn0seyJhY3F1aXNpdGlvbiI6IkF1dGhlbnRpY2F0ZWQgaW5kaXZp"
        "ZHVhbCBvZmZpY2lhbCBVSSBkb3dubG9hZDsgZXhhY3QgaGVhZGluZyBvYnNlcnZlZCBi"
        "eSBPcmNoZXN0cmF0b3IiLCJieXRlcyI6MjM1OTA2OCwiaW5zdGFuY2VfaWQiOiJyaWNo"
        "XzMwNjEiLCJraW5kIjoiZW1iZWRkaW5nIiwib2ZmaWNpYWxfZmlsZV9wYXRoIjoiZW1i"
        "ZWRkaW5ncy9yaWNoX2FjYTk0NjdlYzZhNDgyYjNiZWQxYmE3ZDJlM2RlNzIyNzFjNWY4"
        "MTcubnB6IiwicGF0aCI6InJlc291cmNlcy9vZmZpY2lhbC9lbWJlZGRpbmdzL3JpY2hf"
        "YWNhOTQ2N2VjNmE0ODJiM2JlZDFiYTdkMmUzZGU3MjI3MWM1ZjgxNy5ucHoiLCJzaGEy"
        "NTYiOiJiZmE0MWQ0NTE3NjQ4YmNjODQzY2U4Nzk0NTVlOWUyN2FmNzkyN2E5NzlmMGM4"
        "YWExYjM0ODllNzAyMzBlYTcwIiwic291cmNlIjoiaHR0cHM6Ly93d3cua2FnZ2xlLmNv"
        "bS9jb21wZXRpdGlvbnMvZ2VtbWEtNC1kZXZlbG9wZXItYWdlbnQvZGF0YSJ9LHsiYWNx"
        "dWlzaXRpb24iOiJBdXRoZW50aWNhdGVkIGluZGl2aWR1YWwgb2ZmaWNpYWwgVUkgZG93"
        "bmxvYWQ7IGV4YWN0IGhlYWRpbmcgb2JzZXJ2ZWQgYnkgT3JjaGVzdHJhdG9yIiwiYnl0"
        "ZXMiOjM3NDAyNDc0LCJkb3dubG9hZGVkX2F0IjoiMjAyNi0xMC0xMyIsImluc3RhbmNl"
        "X2lkIjoicmVxdWVzdHNfNzIwNSIsImtpbmQiOiJzbmFwc2hvdCIsIm9mZmljaWFsX2Zp"
        "bGVfcGF0aCI6InNuYXBzaG90cy9yZXF1ZXN0c183MjA1LnRneiIsInBhdGgiOiJyZXNv"
        "dXJjZXMvb2ZmaWNpYWwvc25hcHNob3RzL3JlcXVlc3RzXzcyMDUudGd6Iiwic2hhMjU2"
        "IjoiNDJkYzNkMzYwZjg0MmE1YzdhNjZkMDAwOGFjMDU0NDdmY2Q4MDdkYzMzZWU4Y2Mx"
        "OGQyZjJkNDg3NjI4MzEwNiIsInNvdXJjZSI6Imh0dHBzOi8vd3d3LmthZ2dsZS5jb20v"
        "Y29tcGV0aXRpb25zL2dlbW1hLTQtZGV2ZWxvcGVyLWFnZW50L2RhdGEiLCJ2YWxpZGF0"
        "aW9uIjp7ImJhc2VfdHJlZV9pZGVudGl0eV9tYXRjaGVzIjp0cnVlLCJjb21taXRfb2Jq"
        "ZWN0cyI6NjM5OSwiY29tbWl0X29pZF9tYXRjaGVzX29yaWdpbmFsIjpmYWxzZSwiZXhw"
        "b3J0X2NvbnRyYWN0X3NvdXJjZSI6IkhBUk5FU1NfUkVBRE1FLm1kIHNlY3Rpb240LjIs"
        "IGF1dGhlbnRpY2F0ZWQgU0hBNDQ1MDAwMjE5YjgwMzk0MmM0NjkzOTFlYzhiNjk3OTFh"
        "MzFlOWNlYzJmOTM0NzM4Yzc2N2UzNGI1Y2EwYWExZjogZnJlc2ggZ2l0IGZhc3QtZXhw"
        "b3J0IHVwIHRvIGJhc2VfY29tbWl0IGZvbGxvd2VkIGJ5IGdpdCBmYXN0LWltcG9ydDsg"
        "ZXhwb3J0ZWQgT0lEcyBtYXkgZGlmZmVyIHdoaWxlIHRyZWUgaWRlbnRpdHkgaXMgcHJl"
        "c2VydmVkIiwiZm9yd2FyZF9vcl91bnJlbGF0ZWRfY29tbWl0X29iamVjdHMiOjAsImd6"
        "aXBfY3JjX2NvbXBsZXRlIjp0cnVlLCJoZWFkIjoiNDQzZjBiNzM1NTg3ODFiN2RhYzBl"
        "MzA0YWQyNjhhMzBlZTM1YWFlYiIsIm9yaWdpbmFsX2Jhc2VfY29tbWl0IjoiMWI0MGZk"
        "ZDAwNGJmYzhiYTUzMDFiY2Y4YTY5MDgyNjRlOWI2Yjg3NyIsIm9yaWdpbmFsX2Jhc2Vf"
        "dHJlZV9zaGEiOiI2ODNkNDRiZWZlMGQzNTE5NDY5MDExYzM1ZTQ2ZjMzMmVlYmJjMDEw"
        "Iiwib3JpZ2luYWxfdHJlZV9zb3VyY2UiOiJodHRwczovL2FwaS5naXRodWIuY29tL3Jl"
        "cG9zL3BzZi9yZXF1ZXN0cy9naXQvY29tbWl0cy8xYjQwZmRkMDA0YmZjOGJhNTMwMWJj"
        "ZjhhNjkwODI2NGU5YjZiODc3IiwicmVhY2hhYmxlX2ZvcndhcmRfY29tbWl0cyI6MCwi"
        "c25hcHNob3RfaGVhZF90cmVlX3NoYSI6IjY4M2Q0NGJlZmUwZDM1MTk0NjkwMTFjMzVl"
        "NDZmMzMyZWViYmMwMTAiLCJ0YXJfZW50cmllcyI6MTkzLCJ0YXJfcGFyc2VfY29tcGxl"
        "dGUiOnRydWV9fSx7ImFjcXVpc2l0aW9uIjoiQXV0aGVudGljYXRlZCBpbmRpdmlkdWFs"
        "IG9mZmljaWFsIFVJIGRvd25sb2FkOyBleGFjdCBoZWFkaW5nIG9ic2VydmVkIGJ5IE9y"
        "Y2hlc3RyYXRvciIsImJ5dGVzIjozMTYxNjczOTIsImRvd25sb2FkZWRfYXQiOiIyMDI2"
        "LTEwLTEzIiwiaW5zdGFuY2VfaWQiOiJmYXN0YXBpXzE1NTg5Iiwia2luZCI6InNuYXBz"
        "aG90Iiwib2ZmaWNpYWxfZmlsZV9wYXRoIjoic25hcHNob3RzL2Zhc3RhcGlfMTU1ODku"
        "dGd6IiwicGF0aCI6InJlc291cmNlcy9vZmZpY2lhbC9zbmFwc2hvdHMvZmFzdGFwaV8x"
        "NTU4OS50Z3oiLCJzaGEyNTYiOiJlYmFmNjcxYzI4YmNjYjg1YWE5ZDhhODU3MzUwMTli"
        "NGRhYTZkNzhkNTQxYTY3OTI4OGM2NGVlZDg4ZGM3NmYwIiwic291cmNlIjoiaHR0cHM6"
        "Ly93d3cua2FnZ2xlLmNvbS9jb21wZXRpdGlvbnMvZ2VtbWEtNC1kZXZlbG9wZXItYWdl"
        "bnQvZGF0YSIsInZhbGlkYXRpb24iOnsiYmFzZV90cmVlX2lkZW50aXR5X21hdGNoZXMi"
        "OnRydWUsImNvbW1pdF9vYmplY3RzIjo3MTg1LCJjb21taXRfb2lkX21hdGNoZXNfb3Jp"
        "Z2luYWwiOmZhbHNlLCJleHBvcnRfY29udHJhY3Rfc291cmNlIjoiSEFSTkVTU19SRUFE"
        "TUUubWQgc2VjdGlvbjQuMiwgYXV0aGVudGljYXRlZCBTSEE0NDUwMDAyMTliODAzOTQy"
        "YzQ2OTM5MWVjOGI2OTc5MWEzMWU5Y2VjMmY5MzQ3MzhjNzY3ZTM0YjVjYTBhYTFmOiBm"
        "cmVzaCBnaXQgZmFzdC1leHBvcnQgdXAgdG8gYmFzZV9jb21taXQgZm9sbG93ZWQgYnkg"
        "Z2l0IGZhc3QtaW1wb3J0OyBleHBvcnRlZCBPSURzIG1heSBkaWZmZXIgd2hpbGUgdHJl"
        "ZSBpZGVudGl0eSBpcyBwcmVzZXJ2ZWQiLCJmb3J3YXJkX29yX3VucmVsYXRlZF9jb21t"
        "aXRfb2JqZWN0cyI6MCwiZ3ppcF9jcmNfY29tcGxldGUiOnRydWUsImhlYWQiOiIyZmY5"
        "MmQ0ZjAwZjQxNWFmMjQwM2E3MGRhNTI4YmJkZjc3ZTZjZjRiIiwib3JpZ2luYWxfYmFz"
        "ZV9jb21taXQiOiIyMmIwMmUyNmY5ZThjN2UzMmJkODI2NmUyYjBlYmU4YmIzYTBkYjJi"
        "Iiwib3JpZ2luYWxfYmFzZV90cmVlX3NoYSI6IjkwZmMyZWI4ODM2ZGU0NGU3NWZkNGEy"
        "NGM4M2Y0Y2QwN2MwMDAwNzAiLCJvcmlnaW5hbF90cmVlX3NvdXJjZSI6Imh0dHBzOi8v"
        "YXBpLmdpdGh1Yi5jb20vcmVwb3MvZmFzdGFwaS9mYXN0YXBpL2dpdC9jb21taXRzLzIy"
        "YjAyZTI2ZjllOGM3ZTMyYmQ4MjY2ZTJiMGViZThiYjNhMGRiMmIiLCJyZWFjaGFibGVf"
        "Zm9yd2FyZF9jb21taXRzIjowLCJzbmFwc2hvdF9oZWFkX3RyZWVfc2hhIjoiOTBmYzJl"
        "Yjg4MzZkZTQ0ZTc1ZmQ0YTI0YzgzZjRjZDA3YzAwMDA3MCIsInRhcl9lbnRyaWVzIjoz"
        "NDMwLCJ0YXJfcGFyc2VfY29tcGxldGUiOnRydWV9fSx7ImFjcXVpc2l0aW9uIjoiQXV0"
        "aGVudGljYXRlZCBpbmRpdmlkdWFsIG9mZmljaWFsIFVJIGRvd25sb2FkOyBleGFjdCBo"
        "ZWFkaW5nIG9ic2VydmVkIGJ5IE9yY2hlc3RyYXRvciIsImJ5dGVzIjo5MDU2ODA5Miwi"
        "ZG93bmxvYWRlZF9hdCI6IjIwMjYtMTAtMTMiLCJpbnN0YW5jZV9pZCI6InJpY2hfMzA2"
        "MSIsImtpbmQiOiJzbmFwc2hvdCIsIm9mZmljaWFsX2ZpbGVfcGF0aCI6InNuYXBzaG90"
        "cy9yaWNoXzMwNjEudGd6IiwicGF0aCI6InJlc291cmNlcy9vZmZpY2lhbC9zbmFwc2hv"
        "dHMvcmljaF8zMDYxLnRneiIsInNoYTI1NiI6ImYzMmM5ZWQxMTUzOGRiMmJmMDQ5MzE0"
        "ZmY3YzhmMDhjMDFjYTIzMDA2YmJkOTk0YzI1YWQ3MmMyODE1ZmU5MDUiLCJzb3VyY2Ui"
        "OiJodHRwczovL3d3dy5rYWdnbGUuY29tL2NvbXBldGl0aW9ucy9nZW1tYS00LWRldmVs"
        "b3Blci1hZ2VudC9kYXRhIiwidmFsaWRhdGlvbiI6eyJiYXNlX3RyZWVfaWRlbnRpdHlf"
        "bWF0Y2hlcyI6dHJ1ZSwiY29tbWl0X29iamVjdHMiOjM3ODcsImNvbW1pdF9vaWRfbWF0"
        "Y2hlc19vcmlnaW5hbCI6ZmFsc2UsImV4cG9ydF9jb250cmFjdF9zb3VyY2UiOiJIQVJO"
        "RVNTX1JFQURNRS5tZCBzZWN0aW9uNC4yLCBhdXRoZW50aWNhdGVkIFNIQTQ0NTAwMDIx"
        "OWI4MDM5NDJjNDY5MzkxZWM4YjY5NzkxYTMxZTljZWMyZjkzNDczOGM3NjdlMzRiNWNh"
        "MGFhMWY6IGZyZXNoIGdpdCBmYXN0LWV4cG9ydCB1cCB0byBiYXNlX2NvbW1pdCBmb2xs"
        "b3dlZCBieSBnaXQgZmFzdC1pbXBvcnQ7IGV4cG9ydGVkIE9JRHMgbWF5IGRpZmZlciB3"
        "aGlsZSB0cmVlIGlkZW50aXR5IGlzIHByZXNlcnZlZCIsImZvcndhcmRfb3JfdW5yZWxh"
        "dGVkX2NvbW1pdF9vYmplY3RzIjowLCJnemlwX2NyY19jb21wbGV0ZSI6dHJ1ZSwiaGVh"
        "ZCI6IjdhZjM3YmRiNjYzMGI3YTg0NzM2M2RkYWNmZWEzN2VkODU0OGVlNjQiLCJvcmln"
        "aW5hbF9iYXNlX2NvbW1pdCI6ImFjYTk0NjdlYzZhNDgyYjNiZWQxYmE3ZDJlM2RlNzIy"
        "NzFjNWY4MTciLCJvcmlnaW5hbF9iYXNlX3RyZWVfc2hhIjoiMGU0ZGFjZWQ2YTQzY2Zj"
        "ZGIwZmEwZDlhNzQ3OWY3NDczZDg2OWUxMyIsIm9yaWdpbmFsX3RyZWVfc291cmNlIjoi"
        "aHR0cHM6Ly9hcGkuZ2l0aHViLmNvbS9yZXBvcy9UZXh0dWFsaXplL3JpY2gvZ2l0L2Nv"
        "bW1pdHMvYWNhOTQ2N2VjNmE0ODJiM2JlZDFiYTdkMmUzZGU3MjI3MWM1ZjgxNyIsInJl"
        "YWNoYWJsZV9mb3J3YXJkX2NvbW1pdHMiOjAsInNuYXBzaG90X2hlYWRfdHJlZV9zaGEi"
        "OiIwZTRkYWNlZDZhNDNjZmNkYjBmYTBkOWE3NDc5Zjc0NzNkODY5ZTEzIiwidGFyX2Vu"
        "dHJpZXMiOjU4MiwidGFyX3BhcnNlX2NvbXBsZXRlIjp0cnVlfX1dCg=="
    ),
    "src/mostaza/__init__.py": (
        "IiIiSG9zdC1zaWRlIGRldmVsb3BtZW50LCBldmFsdWF0aW9uIGFuZCByZXNlYXJjaCB0"
        "b29scyBmb3IgTW9zdGF6YS4KClRoaXMgcGFja2FnZSBpcyBub3QgcGFydCBvZiB0aGUg"
        "ZGVjbGFyYXRpdmUgS2FnZ2xlIHN1Ym1pc3Npb24uCiIiIgoKX192ZXJzaW9uX18gPSAi"
        "MC4xLjAiCg=="
    ),
    "src/mostaza/evaluation/__init__.py": (
        "IiIiSG9zdC1zaWRlIGV2YWx1YXRpb24gcmVjb3JkczsgdGhlIG9mZmljaWFsIGhhcm5l"
        "c3Mgb3ducyB0YXNrIGV4ZWN1dGlvbi4iIiIK"
    ),
    "src/mostaza/evaluation/contracts.py": (
        "IiIiU3RyaWN0IHZlcnNpb25lZCBjb250cmFjdHMgZm9yIGltbXV0YWJsZSwgcHJvdmVu"
        "YW5jZS1iZWFyaW5nIHJlc2VhcmNoIHJlY29yZHMuIiIiCgpmcm9tIF9fZnV0dXJlX18g"
        "aW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBv"
        "cnQgbWF0aAppbXBvcnQgcmUKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRp"
        "bWVkZWx0YQpmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpNT0RFTCA9ICJnZW1tYS00LTMx"
        "Yi1pdC1xYXQtdzRhMTYtY3QiCk1FVFJJQ19VTklUUyA9IHsKICAgICJzeW1ib2xzX2V4"
        "cGxvcmVkIjogImNvdW50IiwKICAgICJmaWxlc19leHBsb3JlZCI6ICJjb3VudCIsCiAg"
        "ICAicmVhZF9vcGVyYXRpb25zIjogImNvdW50IiwKICAgICJzZWFyY2hfb3BlcmF0aW9u"
        "cyI6ICJjb3VudCIsCiAgICAibG9jYWxpemF0aW9uX3NlY29uZHMiOiAic2Vjb25kcyIs"
        "CiAgICAidG9rZW5zIjogInRva2VucyIsCiAgICAiY29kZV9jaGFyYWN0ZXJzIjogImNo"
        "YXJhY3RlcnMiLAogICAgImdyYXBoX2V4cGFuc2lvbnMiOiAiY291bnQiLAogICAgInJl"
        "cGVhdGVkX2NvbnRleHRfY2hhcmFjdGVycyI6ICJjaGFyYWN0ZXJzIiwKICAgICJyZWNh"
        "bGxfYXRfayI6ICJyYXRpbyIsCiAgICAicmVsZXZhbnRfY29udGV4dF9yYXRpbyI6ICJy"
        "YXRpbyIsCiAgICAidG9vbF9jYWxscyI6ICJjb3VudCIsCiAgICAidG90YWxfc2Vjb25k"
        "cyI6ICJzZWNvbmRzIiwKICAgICJyZXBhaXJzIjogImNvdW50IiwKfQpHUk9VTkRfVFJV"
        "VEhfTUVUUklDUyA9IHsicmVjYWxsX2F0X2siLCAicmVsZXZhbnRfY29udGV4dF9yYXRp"
        "byJ9Ck1BTklGRVNUX0ZJRUxEUyA9IHsKICAgICJzY2hlbWFfdmVyc2lvbiIsCiAgICAi"
        "ZXhwZXJpbWVudF9pZCIsCiAgICAicnVuX2lkIiwKICAgICJhcm0iLAogICAgInJ1bl9r"
        "aW5kIiwKICAgICJ0aW1lc3RhbXAiLAogICAgImNvZGVfY29tbWl0IiwKICAgICJ3b3Jr"
        "aW5nX3RyZWVfc2hhMjU2IiwKICAgICJjb25maWdfc2hhMjU2IiwKICAgICJhcmNoaXZl"
        "X3NoYTI1NiIsCiAgICAibW9kZWwiLAogICAgInJlc291cmNlcyIsCiAgICAiY29ob3J0"
        "IiwKICAgICJpbmZlcmVuY2UiLAogICAgImJ1ZGdldHMiLAogICAgImVudmlyb25tZW50"
        "IiwKICAgICJsaW1pdGF0aW9ucyIsCiAgICAic291cmNlX3J1bl9pZCIsCn0KUkVDT1JE"
        "X0ZJRUxEUyA9IHsKICAgICJzY2hlbWFfdmVyc2lvbiIsCiAgICAiZXhwZXJpbWVudF9p"
        "ZCIsCiAgICAicnVuX2lkIiwKICAgICJhcm0iLAogICAgInJ1bl9raW5kIiwKICAgICJp"
        "bnN0YW5jZV9pZCIsCiAgICAic3RhdHVzIiwKICAgICJyZWFzb24iLAogICAgInBhdGNo"
        "IiwKICAgICJ0ZXN0cyIsCiAgICAicmVzb2x2ZWQiLAogICAgIm1ldHJpY3MiLAogICAg"
        "ImFydGlmYWN0cyIsCiAgICAiZ3JvdW5kX3RydXRoIiwKfQoKCmRlZiBjYW5vbmljYWxf"
        "anNvbih2YWx1ZTogQW55KSAtPiBieXRlczoKICAgIHJldHVybiAoCiAgICAgICAganNv"
        "bi5kdW1wcygKICAgICAgICAgICAgdmFsdWUsCiAgICAgICAgICAgIHNvcnRfa2V5cz1U"
        "cnVlLAogICAgICAgICAgICBzZXBhcmF0b3JzPSgiLCIsICI6IiksCiAgICAgICAgICAg"
        "IGVuc3VyZV9hc2NpaT1GYWxzZSwKICAgICAgICAgICAgYWxsb3dfbmFuPUZhbHNlLAog"
        "ICAgICAgICkKICAgICAgICArICJcbiIKICAgICkuZW5jb2RlKCkKCgpkZWYgc2hhMjU2"
        "KGRhdGE6IGJ5dGVzKSAtPiBzdHI6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYoZGF0"
        "YSkuaGV4ZGlnZXN0KCkKCgpkZWYgcmVxdWlyZShjb25kaXRpb246IGJvb2wsIG1lc3Nh"
        "Z2U6IHN0cikgLT4gTm9uZToKICAgIGlmIG5vdCBjb25kaXRpb246CiAgICAgICAgcmFp"
        "c2UgVmFsdWVFcnJvcihtZXNzYWdlKQoKCmRlZiBkaWdlc3QodmFsdWU6IEFueSkgLT4g"
        "Ym9vbDoKICAgIHJldHVybiBpc2luc3RhbmNlKHZhbHVlLCBzdHIpIGFuZCByZS5mdWxs"
        "bWF0Y2gociJbMC05YS1mXXs2NH0iLCB2YWx1ZSkgaXMgbm90IE5vbmUKCgpkZWYgbm9u"
        "ZW1wdHkodmFsdWU6IEFueSkgLT4gYm9vbDoKICAgIHJldHVybiBpc2luc3RhbmNlKHZh"
        "bHVlLCBzdHIpIGFuZCBib29sKHZhbHVlLnN0cmlwKCkpCgoKZGVmIGFydGlmYWN0KHZh"
        "bHVlOiBBbnkpIC0+IE5vbmU6CiAgICByZXF1aXJlKAogICAgICAgIGlzaW5zdGFuY2Uo"
        "dmFsdWUsIGRpY3QpIGFuZCBzZXQodmFsdWUpID09IHsic2hhMjU2IiwgInNvdXJjZSJ9"
        "LAogICAgICAgICJBcnRpZmFjdCBtdXN0IGRlY2xhcmUgZGlnZXN0IGFuZCBzb3VyY2Ui"
        "LAogICAgKQogICAgcmVxdWlyZSgKICAgICAgICBkaWdlc3QodmFsdWVbInNoYTI1NiJd"
        "KSBhbmQgbm9uZW1wdHkodmFsdWVbInNvdXJjZSJdKSwKICAgICAgICAiSW52YWxpZCBh"
        "cnRpZmFjdCBwcm92ZW5hbmNlIiwKICAgICkKCgpkZWYgdmFsaWRhdGVfbWFuaWZlc3Qo"
        "bWFuaWZlc3Q6IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOgogICAgcmVxdWlyZShzZXQo"
        "bWFuaWZlc3QpID09IE1BTklGRVNUX0ZJRUxEUywgIlVua25vd24gb3IgbWlzc2luZyBt"
        "YW5pZmVzdCBmaWVsZHMiKQogICAgcmVxdWlyZSgKICAgICAgICB0eXBlKG1hbmlmZXN0"
        "WyJzY2hlbWFfdmVyc2lvbiJdKSBpcyBpbnQgYW5kIG1hbmlmZXN0WyJzY2hlbWFfdmVy"
        "c2lvbiJdID09IDEsCiAgICAgICAgIlVuc3VwcG9ydGVkIG1hbmlmZXN0IHNjaGVtYSIs"
        "CiAgICApCiAgICBmb3IgZmllbGQgaW4gKCJleHBlcmltZW50X2lkIiwgInJ1bl9pZCIs"
        "ICJzb3VyY2VfcnVuX2lkIik6CiAgICAgICAgcmVxdWlyZShub25lbXB0eShtYW5pZmVz"
        "dFtmaWVsZF0pLCBmIk1pc3Npbmcge2ZpZWxkfSIpCiAgICByZXF1aXJlKG1hbmlmZXN0"
        "WyJhcm0iXSBpbiAoIkUwIiwgIkUxIiksICJVbmtub3duIGFybSIpCiAgICByZXF1aXJl"
        "KAogICAgICAgIG1hbmlmZXN0WyJydW5fa2luZCJdIGluICgicmVhbF9rYWdnbGUiLCAi"
        "c3ludGhldGljX3Rlc3QiKSwgIlVua25vd24gcnVuIGtpbmQiCiAgICApCiAgICByZXF1"
        "aXJlKG1hbmlmZXN0WyJtb2RlbCJdID09IE1PREVMLCAiVGhlIHJlcXVpcmVkIG1vZGVs"
        "IG11c3QgYmUgcGlubmVkIikKICAgIHJlcXVpcmUoCiAgICAgICAgaXNpbnN0YW5jZSht"
        "YW5pZmVzdFsiY29kZV9jb21taXQiXSwgc3RyKQogICAgICAgIGFuZCByZS5mdWxsbWF0"
        "Y2gociJbMC05YS1mXXs0MH0iLCBtYW5pZmVzdFsiY29kZV9jb21taXQiXSkgaXMgbm90"
        "IE5vbmUsCiAgICAgICAgIkFuIGV4YWN0IGNvZGUgY29tbWl0IGlzIHJlcXVpcmVkIiwK"
        "ICAgICkKICAgIGZvciBmaWVsZCBpbiAoIndvcmtpbmdfdHJlZV9zaGEyNTYiLCAiY29u"
        "ZmlnX3NoYTI1NiIsICJhcmNoaXZlX3NoYTI1NiIpOgogICAgICAgIHJlcXVpcmUoZGln"
        "ZXN0KG1hbmlmZXN0W2ZpZWxkXSksIGYiSW52YWxpZCB7ZmllbGR9IikKICAgIHRyeToK"
        "ICAgICAgICBzdGFtcCA9IGRhdGV0aW1lLmZyb21pc29mb3JtYXQobWFuaWZlc3RbInRp"
        "bWVzdGFtcCJdLnJlcGxhY2UoIloiLCAiKzAwOjAwIikpCiAgICBleGNlcHQgKFZhbHVl"
        "RXJyb3IsIFR5cGVFcnJvciwgQXR0cmlidXRlRXJyb3IpOgogICAgICAgIHJhaXNlIFZh"
        "bHVlRXJyb3IoIkludmFsaWQgVVRDIHRpbWVzdGFtcCIpIGZyb20gTm9uZQogICAgcmVx"
        "dWlyZShzdGFtcC51dGNvZmZzZXQoKSA9PSB0aW1lZGVsdGEoMCksICJUaW1lc3RhbXAg"
        "bXVzdCBiZSBVVEMiKQogICAgZm9yIGZpZWxkIGluICgiaW5mZXJlbmNlIiwgImJ1ZGdl"
        "dHMiLCAiZW52aXJvbm1lbnQiKToKICAgICAgICByZXF1aXJlKAogICAgICAgICAgICBp"
        "c2luc3RhbmNlKG1hbmlmZXN0W2ZpZWxkXSwgZGljdCkgYW5kIGJvb2wobWFuaWZlc3Rb"
        "ZmllbGRdKSwKICAgICAgICAgICAgZiJFeHBsaWNpdCB7ZmllbGR9IGlzIHJlcXVpcmVk"
        "IiwKICAgICAgICApCiAgICByZXF1aXJlKAogICAgICAgIGlzaW5zdGFuY2UobWFuaWZl"
        "c3RbImxpbWl0YXRpb25zIl0sIGxpc3QpCiAgICAgICAgYW5kIGFsbChub25lbXB0eSh4"
        "KSBmb3IgeCBpbiBtYW5pZmVzdFsibGltaXRhdGlvbnMiXSksCiAgICAgICAgIkludmFs"
        "aWQgbGltaXRhdGlvbnMiLAogICAgKQogICAgcmVzb3VyY2VzID0gbWFuaWZlc3RbInJl"
        "c291cmNlcyJdCiAgICByZXF1aXJlKAogICAgICAgIGlzaW5zdGFuY2UocmVzb3VyY2Vz"
        "LCBkaWN0KQogICAgICAgIGFuZCBzZXQocmVzb3VyY2VzKSA9PSB7Imhhcm5lc3MiLCAi"
        "d2hlZWxob3VzZSIsICJkYXRhc2V0In0sCiAgICAgICAgIlJlcXVpcmVkIHJlc291cmNl"
        "IHBpbnMgbWlzc2luZyIsCiAgICApCiAgICBmb3IgcmVzb3VyY2UgaW4gcmVzb3VyY2Vz"
        "LnZhbHVlcygpOgogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIGlzaW5zdGFuY2Uo"
        "cmVzb3VyY2UsIGRpY3QpCiAgICAgICAgICAgIGFuZCBzZXQocmVzb3VyY2UpID09IHsi"
        "dmVyc2lvbiIsICJzaGEyNTYiLCAic291cmNlIn0sCiAgICAgICAgICAgICJJbnZhbGlk"
        "IHJlc291cmNlIHBpbiIsCiAgICAgICAgKQogICAgICAgIHJlcXVpcmUobm9uZW1wdHko"
        "cmVzb3VyY2VbInZlcnNpb24iXSksICJSZXNvdXJjZSB2ZXJzaW9uIG1pc3NpbmciKQog"
        "ICAgICAgIGFydGlmYWN0KHsic2hhMjU2IjogcmVzb3VyY2VbInNoYTI1NiJdLCAic291"
        "cmNlIjogcmVzb3VyY2VbInNvdXJjZSJdfSkKICAgIGNvaG9ydCA9IG1hbmlmZXN0WyJj"
        "b2hvcnQiXQogICAgcmVxdWlyZSgKICAgICAgICBpc2luc3RhbmNlKGNvaG9ydCwgbGlz"
        "dCkgYW5kIGJvb2woY29ob3J0KSwKICAgICAgICAiQSBub25lbXB0eSBmcm96ZW4gY29o"
        "b3J0IGlzIHJlcXVpcmVkIiwKICAgICkKICAgIHNlZW46IHNldFtzdHJdID0gc2V0KCkK"
        "ICAgIGZvciB0YXNrIGluIGNvaG9ydDoKICAgICAgICByZXF1aXJlKAogICAgICAgICAg"
        "ICBpc2luc3RhbmNlKHRhc2ssIGRpY3QpCiAgICAgICAgICAgIGFuZCBzZXQodGFzaykK"
        "ICAgICAgICAgICAgPT0geyJpbnN0YW5jZV9pZCIsICJyZXBvIiwgImJhc2VfY29tbWl0"
        "IiwgInNuYXBzaG90X3NoYTI1NiIsICJyYXRpb25hbGUifSwKICAgICAgICAgICAgIklu"
        "dmFsaWQgZnJvemVuIHRhc2siLAogICAgICAgICkKICAgICAgICBmb3IgZmllbGQgaW4g"
        "KCJpbnN0YW5jZV9pZCIsICJyZXBvIiwgInJhdGlvbmFsZSIpOgogICAgICAgICAgICBy"
        "ZXF1aXJlKG5vbmVtcHR5KHRhc2tbZmllbGRdKSwgIkZyb3plbiB0YXNrIGNvbnRleHQg"
        "bWlzc2luZyIpCiAgICAgICAgcmVxdWlyZSh0YXNrWyJpbnN0YW5jZV9pZCJdIG5vdCBp"
        "biBzZWVuLCAiRHVwbGljYXRlIGNvaG9ydCB0YXNrIikKICAgICAgICBzZWVuLmFkZCh0"
        "YXNrWyJpbnN0YW5jZV9pZCJdKQogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIGlz"
        "aW5zdGFuY2UodGFza1siYmFzZV9jb21taXQiXSwgc3RyKQogICAgICAgICAgICBhbmQg"
        "cmUuZnVsbG1hdGNoKHIiWzAtOWEtZl17NDB9IiwgdGFza1siYmFzZV9jb21taXQiXSkg"
        "aXMgbm90IE5vbmUsCiAgICAgICAgICAgICJGcm96ZW4gYmFzZSBjb21taXQgcmVxdWly"
        "ZWQiLAogICAgICAgICkKICAgICAgICByZXF1aXJlKGRpZ2VzdCh0YXNrWyJzbmFwc2hv"
        "dF9zaGEyNTYiXSksICJTbmFwc2hvdCBoYXNoIHJlcXVpcmVkIikKICAgICMgQ2F0Y2gg"
        "bm9uLUpTT04gdmFsdWVzL05hTiBpbiBhcmJpdHJhcnkgaW5mZXJlbmNlL2Vudmlyb25t"
        "ZW50IG1hcHBpbmdzLgogICAgY2Fub25pY2FsX2pzb24obWFuaWZlc3QpCgoKZGVmIHVu"
        "YXZhaWxhYmxlX21ldHJpY3MocmVhc29uOiBzdHIpIC0+IGRpY3Rbc3RyLCBBbnldOgog"
        "ICAgcmVxdWlyZShub25lbXB0eShyZWFzb24pLCAiTWlzc2luZyBtZXRyaWMgcmVhc29u"
        "IHJlcXVpcmVkIikKICAgIHJldHVybiB7CiAgICAgICAgbmFtZTogewogICAgICAgICAg"
        "ICAidmFsdWUiOiBOb25lLAogICAgICAgICAgICAidW5pdCI6IHVuaXQsCiAgICAgICAg"
        "ICAgICJzb3VyY2UiOiBOb25lLAogICAgICAgICAgICAiYXZhaWxhYmlsaXR5IjogInVu"
        "YXZhaWxhYmxlIiwKICAgICAgICAgICAgInJlYXNvbiI6IHJlYXNvbiwKICAgICAgICB9"
        "CiAgICAgICAgZm9yIG5hbWUsIHVuaXQgaW4gTUVUUklDX1VOSVRTLml0ZW1zKCkKICAg"
        "IH0KCgpkZWYgdmFsaWRhdGVfcmVjb3JkKHJlY29yZDogZGljdFtzdHIsIEFueV0sIG1h"
        "bmlmZXN0OiBkaWN0W3N0ciwgQW55XSkgLT4gTm9uZToKICAgIHJlcXVpcmUoc2V0KHJl"
        "Y29yZCkgPT0gUkVDT1JEX0ZJRUxEUywgIlVua25vd24gb3IgbWlzc2luZyB0YXNrIHJl"
        "Y29yZCBmaWVsZHMiKQogICAgcmVxdWlyZSgKICAgICAgICB0eXBlKHJlY29yZFsic2No"
        "ZW1hX3ZlcnNpb24iXSkgaXMgaW50IGFuZCByZWNvcmRbInNjaGVtYV92ZXJzaW9uIl0g"
        "PT0gMSwKICAgICAgICAiVW5zdXBwb3J0ZWQgcmVjb3JkIHNjaGVtYSIsCiAgICApCiAg"
        "ICBmb3IgZmllbGQgaW4gKCJleHBlcmltZW50X2lkIiwgInJ1bl9pZCIsICJhcm0iLCAi"
        "cnVuX2tpbmQiKToKICAgICAgICByZXF1aXJlKHJlY29yZFtmaWVsZF0gPT0gbWFuaWZl"
        "c3RbZmllbGRdLCBmIlJlY29yZCB7ZmllbGR9IG1pc21hdGNoIikKICAgIHJlcXVpcmUo"
        "CiAgICAgICAgcmVjb3JkWyJpbnN0YW5jZV9pZCJdIGluIHt0WyJpbnN0YW5jZV9pZCJd"
        "IGZvciB0IGluIG1hbmlmZXN0WyJjb2hvcnQiXX0sCiAgICAgICAgIlJlY29yZCBvdXRz"
        "aWRlIGZyb3plbiBjb2hvcnQiLAogICAgKQogICAgc3RhdHVzID0gcmVjb3JkWyJzdGF0"
        "dXMiXQogICAgcmVxdWlyZShzdGF0dXMgaW4gKCJQQVNTIiwgIkZBSUwiLCAiRVJST1Ii"
        "LCAiTk9UX1JVTiIpLCAiVW5rbm93biB0YXNrIHN0YXR1cyIpCiAgICByZXF1aXJlKAog"
        "ICAgICAgIHJlY29yZFsicmVhc29uIl0gaXMgTm9uZSBvciBub25lbXB0eShyZWNvcmRb"
        "InJlYXNvbiJdKSwgIkludmFsaWQgc3RhdHVzIHJlYXNvbiIKICAgICkKICAgIGlmIHN0"
        "YXR1cyBpbiAoIkVSUk9SIiwgIk5PVF9SVU4iKToKICAgICAgICByZXF1aXJlKG5vbmVt"
        "cHR5KHJlY29yZFsicmVhc29uIl0pLCAiRXJyb3Ivbm90LXJ1biByZWFzb24gcmVxdWly"
        "ZWQiKQogICAgcGF0Y2ggPSByZWNvcmRbInBhdGNoIl0KICAgIHJlcXVpcmUoCiAgICAg"
        "ICAgaXNpbnN0YW5jZShwYXRjaCwgZGljdCkKICAgICAgICBhbmQgc2V0KHBhdGNoKSA9"
        "PSB7InByZXNlbnQiLCAidmFsaWQiLCAic2hhMjU2IiwgInZhbGlkaXR5X3NvdXJjZSIs"
        "ICJyZWFzb24ifSwKICAgICAgICAiSW52YWxpZCBwYXRjaCBmaWVsZHMiLAogICAgKQog"
        "ICAgcmVxdWlyZSgKICAgICAgICB0eXBlKHBhdGNoWyJwcmVzZW50Il0pIGlzIGJvb2wK"
        "ICAgICAgICBhbmQgKHBhdGNoWyJ2YWxpZCJdIGlzIE5vbmUgb3IgdHlwZShwYXRjaFsi"
        "dmFsaWQiXSkgaXMgYm9vbCksCiAgICAgICAgIkludmFsaWQgcGF0Y2ggc3RhdGUiLAog"
        "ICAgKQogICAgcmVxdWlyZSgKICAgICAgICBkaWdlc3QocGF0Y2hbInNoYTI1NiJdKSBp"
        "ZiBwYXRjaFsicHJlc2VudCJdIGVsc2UgcGF0Y2hbInNoYTI1NiJdIGlzIE5vbmUsCiAg"
        "ICAgICAgIlBhdGNoIHByZXNlbmNlL2RpZ2VzdCBtaXNtYXRjaCIsCiAgICApCiAgICBy"
        "ZXF1aXJlKAogICAgICAgIG5vbmVtcHR5KHBhdGNoWyJ2YWxpZGl0eV9zb3VyY2UiXSkK"
        "ICAgICAgICBpZiBwYXRjaFsidmFsaWQiXSBpcyBub3QgTm9uZQogICAgICAgIGVsc2Ug"
        "bm9uZW1wdHkocGF0Y2hbInJlYXNvbiJdKSwKICAgICAgICAiUGF0Y2ggdmFsaWRpdHkg"
        "ZXZpZGVuY2Ugb3IgcmVhc29uIHJlcXVpcmVkIiwKICAgICkKICAgIHJlcXVpcmUoCiAg"
        "ICAgICAgcGF0Y2hbInByZXNlbnQiXSBvciBwYXRjaFsidmFsaWQiXSBpcyBub3QgVHJ1"
        "ZSwgIkFic2VudCBwYXRjaCBjYW5ub3QgYmUgdmFsaWQiCiAgICApCiAgICB0ZXN0cyA9"
        "IHJlY29yZFsidGVzdHMiXQogICAgcmVxdWlyZSgKICAgICAgICBpc2luc3RhbmNlKHRl"
        "c3RzLCBkaWN0KQogICAgICAgIGFuZCBzZXQodGVzdHMpID09IHsib3V0Y29tZSIsICJl"
        "eGl0X2NvZGUiLCAic291cmNlIiwgInJlYXNvbiJ9LAogICAgICAgICJJbnZhbGlkIHRl"
        "c3QgZmllbGRzIiwKICAgICkKICAgIHJlcXVpcmUoCiAgICAgICAgdGVzdHNbIm91dGNv"
        "bWUiXSBpbiAoIlBBU1MiLCAiRkFJTCIsICJFUlJPUiIsICJOT1RfUlVOIiksICJJbnZh"
        "bGlkIHRlc3Qgb3V0Y29tZSIKICAgICkKICAgIHJlcXVpcmUoCiAgICAgICAgdGVzdHNb"
        "ImV4aXRfY29kZSJdIGlzIE5vbmUgb3IgdHlwZSh0ZXN0c1siZXhpdF9jb2RlIl0pIGlz"
        "IGludCwKICAgICAgICAiSW52YWxpZCB0ZXN0IGV4aXQgY29kZSIsCiAgICApCiAgICBy"
        "ZXF1aXJlKAogICAgICAgIG5vbmVtcHR5KHRlc3RzWyJzb3VyY2UiXSkKICAgICAgICBp"
        "ZiB0ZXN0c1sib3V0Y29tZSJdIGluICgiUEFTUyIsICJGQUlMIikKICAgICAgICBlbHNl"
        "IG5vbmVtcHR5KHRlc3RzWyJyZWFzb24iXSksCiAgICAgICAgIlRlc3QgZXZpZGVuY2Ug"
        "b3IgcmVhc29uIHJlcXVpcmVkIiwKICAgICkKICAgIGlmIHRlc3RzWyJvdXRjb21lIl0g"
        "PT0gIlBBU1MiOgogICAgICAgIHJlcXVpcmUodGVzdHNbImV4aXRfY29kZSJdID09IDAs"
        "ICJQYXNzaW5nIHRlc3RzIHJlcXVpcmUgZXhpdCB6ZXJvIikKICAgIGlmIHRlc3RzWyJv"
        "dXRjb21lIl0gPT0gIkZBSUwiOgogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIHRl"
        "c3RzWyJleGl0X2NvZGUiXSBpcyBub3QgTm9uZSBhbmQgdGVzdHNbImV4aXRfY29kZSJd"
        "ID4gMCwKICAgICAgICAgICAgIkZhaWxlZCB0ZXN0cyByZXF1aXJlIGEgcG9zaXRpdmUg"
        "ZXhpdCBjb2RlIiwKICAgICAgICApCiAgICByZXF1aXJlKAogICAgICAgIHJlY29yZFsi"
        "cmVzb2x2ZWQiXSBpcyBOb25lIG9yIHR5cGUocmVjb3JkWyJyZXNvbHZlZCJdKSBpcyBi"
        "b29sLAogICAgICAgICJJbnZhbGlkIHJlc29sdXRpb24gc3RhdGUiLAogICAgKQogICAg"
        "aWYgc3RhdHVzID09ICJQQVNTIjoKICAgICAgICByZXF1aXJlKAogICAgICAgICAgICBy"
        "ZWNvcmRbInJlc29sdmVkIl0gaXMgVHJ1ZQogICAgICAgICAgICBhbmQgcGF0Y2hbInBy"
        "ZXNlbnQiXQogICAgICAgICAgICBhbmQgcGF0Y2hbInZhbGlkIl0gaXMgVHJ1ZQogICAg"
        "ICAgICAgICBhbmQgdGVzdHNbIm91dGNvbWUiXSA9PSAiUEFTUyIsCiAgICAgICAgICAg"
        "ICJQQVNTIHJlcXVpcmVzIHZlcmlmaWVkIHJlc29sdXRpb24iLAogICAgICAgICkKICAg"
        "IGlmIHN0YXR1cyA9PSAiRVJST1IiOgogICAgICAgIHJlcXVpcmUocmVjb3JkWyJyZXNv"
        "bHZlZCJdIGlzIG5vdCBUcnVlLCAiRVJST1IgY2Fubm90IGNsYWltIHJlc29sdXRpb24i"
        "KQogICAgaWYgc3RhdHVzID09ICJGQUlMIjoKICAgICAgICByZXF1aXJlKHJlY29yZFsi"
        "cmVzb2x2ZWQiXSBpcyBGYWxzZSwgIkZBSUwgcmVxdWlyZXMgdW5yZXNvbHZlZCBvdXRj"
        "b21lIikKICAgIGlmIHN0YXR1cyA9PSAiTk9UX1JVTiI6CiAgICAgICAgcmVxdWlyZSgK"
        "ICAgICAgICAgICAgcmVjb3JkWyJyZXNvbHZlZCJdIGlzIE5vbmUKICAgICAgICAgICAg"
        "YW5kIG5vdCBwYXRjaFsicHJlc2VudCJdCiAgICAgICAgICAgIGFuZCB0ZXN0c1sib3V0"
        "Y29tZSJdID09ICJOT1RfUlVOIiwKICAgICAgICAgICAgIk5PVF9SVU4gY2Fubm90IGNv"
        "bnRhaW4gYW4gb3V0Y29tZSIsCiAgICAgICAgKQogICAgYXJ0aWZhY3RzID0gcmVjb3Jk"
        "WyJhcnRpZmFjdHMiXQogICAgcmVxdWlyZShpc2luc3RhbmNlKGFydGlmYWN0cywgZGlj"
        "dCksICJBcnRpZmFjdHMgbXVzdCBiZSBhbiBvYmplY3QiKQogICAgZm9yIHZhbHVlIGlu"
        "IGFydGlmYWN0cy52YWx1ZXMoKToKICAgICAgICBhcnRpZmFjdCh2YWx1ZSkKICAgIGlm"
        "IHJlY29yZFsicnVuX2tpbmQiXSA9PSAicmVhbF9rYWdnbGUiIGFuZCBzdGF0dXMgIT0g"
        "Ik5PVF9SVU4iOgogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIHsicmVzdWx0Iiwg"
        "InRyYWNlIn0gPD0gc2V0KGFydGlmYWN0cyksCiAgICAgICAgICAgICJSZWFsIGV4ZWN1"
        "dGlvbiByZXF1aXJlcyB0cmFjZS9yZXN1bHQgaGFzaGVzIiwKICAgICAgICApCiAgICBn"
        "cm91bmQgPSByZWNvcmRbImdyb3VuZF90cnV0aCJdCiAgICBpZiBncm91bmQgaXMgbm90"
        "IE5vbmU6CiAgICAgICAgcmVxdWlyZSgKICAgICAgICAgICAgaXNpbnN0YW5jZShncm91"
        "bmQsIGRpY3QpIGFuZCBzZXQoZ3JvdW5kKSA9PSB7Im1hcHBpbmciLCAiZGVmaW5pdGlv"
        "bnMifSwKICAgICAgICAgICAgIkludmFsaWQgcG9zdC1ydW4gZ3JvdW5kIHRydXRoIiwK"
        "ICAgICAgICApCiAgICAgICAgYXJ0aWZhY3QoZ3JvdW5kWyJtYXBwaW5nIl0pCiAgICAg"
        "ICAgcmVxdWlyZShpc2luc3RhbmNlKGdyb3VuZFsiZGVmaW5pdGlvbnMiXSwgZGljdCks"
        "ICJNZXRyaWMgZGVmaW5pdGlvbnMgbWlzc2luZyIpCiAgICBtZXRyaWNzID0gcmVjb3Jk"
        "WyJtZXRyaWNzIl0KICAgIHJlcXVpcmUoCiAgICAgICAgaXNpbnN0YW5jZShtZXRyaWNz"
        "LCBkaWN0KSBhbmQgc2V0KG1ldHJpY3MpID09IHNldChNRVRSSUNfVU5JVFMpLAogICAg"
        "ICAgICJFdmVyeSBtZXRyaWMgbXVzdCBiZSBwcmVzZW50LCB1c2luZyBleHBsaWNpdCBt"
        "aXNzaW5nIHZhbHVlcyIsCiAgICApCiAgICBmb3IgbmFtZSwgbWV0cmljIGluIG1ldHJp"
        "Y3MuaXRlbXMoKToKICAgICAgICByZXF1aXJlKAogICAgICAgICAgICBpc2luc3RhbmNl"
        "KG1ldHJpYywgZGljdCkKICAgICAgICAgICAgYW5kIHNldChtZXRyaWMpID09IHsidmFs"
        "dWUiLCAidW5pdCIsICJzb3VyY2UiLCAiYXZhaWxhYmlsaXR5IiwgInJlYXNvbiJ9LAog"
        "ICAgICAgICAgICAiSW52YWxpZCBtZXRyaWMgZmllbGRzIiwKICAgICAgICApCiAgICAg"
        "ICAgcmVxdWlyZShtZXRyaWNbInVuaXQiXSA9PSBNRVRSSUNfVU5JVFNbbmFtZV0sICJN"
        "ZXRyaWMgdW5pdCBtaXNtYXRjaCIpCiAgICAgICAgYXZhaWxhYmxlID0gbWV0cmljWyJh"
        "dmFpbGFiaWxpdHkiXQogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIGF2YWlsYWJs"
        "ZSBpbiAoIm1lYXN1cmVkIiwgImVzdGltYXRlZCIsICJ1bmF2YWlsYWJsZSIpLAogICAg"
        "ICAgICAgICAiSW52YWxpZCBhdmFpbGFiaWxpdHkiLAogICAgICAgICkKICAgICAgICBp"
        "ZiBhdmFpbGFibGUgPT0gInVuYXZhaWxhYmxlIjoKICAgICAgICAgICAgcmVxdWlyZSgK"
        "ICAgICAgICAgICAgICAgIG1ldHJpY1sidmFsdWUiXSBpcyBOb25lCiAgICAgICAgICAg"
        "ICAgICBhbmQgbWV0cmljWyJzb3VyY2UiXSBpcyBOb25lCiAgICAgICAgICAgICAgICBh"
        "bmQgbm9uZW1wdHkobWV0cmljWyJyZWFzb24iXSksCiAgICAgICAgICAgICAgICAiTWlz"
        "c2luZyBtZXRyaWMgcmVxdWlyZXMgbnVsbCBhbmQgcmVhc29uIiwKICAgICAgICAgICAg"
        "KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHZhbHVlID0gbWV0cmljWyJ2YWx1ZSJd"
        "CiAgICAgICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgICAgICB0eXBlKHZhbHVlKSBp"
        "biAoaW50LCBmbG9hdCkgYW5kIG1hdGguaXNmaW5pdGUodmFsdWUpIGFuZCB2YWx1ZSA+"
        "PSAwLAogICAgICAgICAgICAgICAgIk1ldHJpYyB2YWx1ZSBtdXN0IGJlIGZpbml0ZSBh"
        "bmQgbm9ubmVnYXRpdmUiLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJlcXVpcmUo"
        "CiAgICAgICAgICAgICAgICBub25lbXB0eShtZXRyaWNbInNvdXJjZSJdKSBhbmQgbWV0"
        "cmljWyJyZWFzb24iXSBpcyBOb25lLAogICAgICAgICAgICAgICAgIkF2YWlsYWJsZSBt"
        "ZXRyaWMgcmVxdWlyZXMgc291cmNlIiwKICAgICAgICAgICAgKQogICAgICAgICAgICBp"
        "ZiBtZXRyaWNbInVuaXQiXSBpbiAoImNvdW50IiwgInRva2VucyIsICJjaGFyYWN0ZXJz"
        "Iik6CiAgICAgICAgICAgICAgICByZXF1aXJlKHR5cGUodmFsdWUpIGlzIGludCwgIkRp"
        "c2NyZXRlIG1ldHJpYyByZXF1aXJlcyBhbiBpbnRlZ2VyIikKICAgICAgICAgICAgaWYg"
        "bWV0cmljWyJ1bml0Il0gPT0gInJhdGlvIjoKICAgICAgICAgICAgICAgIHJlcXVpcmUo"
        "dmFsdWUgPD0gMSwgIlJhdGlvIG91dCBvZiByYW5nZSIpCiAgICAgICAgICAgIGlmIG5h"
        "bWUgaW4gR1JPVU5EX1RSVVRIX01FVFJJQ1M6CiAgICAgICAgICAgICAgICByZXF1aXJl"
        "KAogICAgICAgICAgICAgICAgICAgIGdyb3VuZCBpcyBub3QgTm9uZSBhbmQgbm9uZW1w"
        "dHkoZ3JvdW5kWyJkZWZpbml0aW9ucyJdLmdldChuYW1lKSksCiAgICAgICAgICAgICAg"
        "ICAgICAgIlJlbGV2YW5jZSBtZXRyaWNzIHJlcXVpcmUgcG9zdC1ydW4gbWFwcGluZyBh"
        "bmQgZGVmaW5pdGlvbnMiLAogICAgICAgICAgICAgICAgKQogICAgICAgIGlmIHN0YXR1"
        "cyA9PSAiTk9UX1JVTiI6CiAgICAgICAgICAgIHJlcXVpcmUoYXZhaWxhYmxlID09ICJ1"
        "bmF2YWlsYWJsZSIsICJOT1RfUlVOIGNhbm5vdCBoYXZlIG1lYXN1cmVtZW50cyIpCiAg"
        "ICBjYW5vbmljYWxfanNvbihyZWNvcmQpCgoKZGVmIHZhbGlkYXRlX3J1bihtYW5pZmVz"
        "dDogZGljdFtzdHIsIEFueV0sIHJlY29yZHM6IGxpc3RbZGljdFtzdHIsIEFueV1dKSAt"
        "PiBOb25lOgogICAgdmFsaWRhdGVfbWFuaWZlc3QobWFuaWZlc3QpCiAgICBzZWVuOiBz"
        "ZXRbc3RyXSA9IHNldCgpCiAgICBmb3IgcmVjb3JkIGluIHJlY29yZHM6CiAgICAgICAg"
        "dmFsaWRhdGVfcmVjb3JkKHJlY29yZCwgbWFuaWZlc3QpCiAgICAgICAgcmVxdWlyZShy"
        "ZWNvcmRbImluc3RhbmNlX2lkIl0gbm90IGluIHNlZW4sICJEdXBsaWNhdGUgdGFzay9h"
        "cm0gcmVjb3JkIikKICAgICAgICBzZWVuLmFkZChyZWNvcmRbImluc3RhbmNlX2lkIl0p"
        "CiAgICByZXF1aXJlKAogICAgICAgIHNlZW4gPT0ge3RbImluc3RhbmNlX2lkIl0gZm9y"
        "IHQgaW4gbWFuaWZlc3RbImNvaG9ydCJdfSwKICAgICAgICAiRXZlcnkgY29ob3J0IHRh"
        "c2sgcmVxdWlyZXMgYSByZWNvcmQsIGluY2x1ZGluZyBOT1RfUlVOIiwKICAgICkK"
    ),
    "src/mostaza/evaluation/runner.py": (
        "IiIiVGhpbiBleGVjdXRpb24gYW5kIHByb3ZlbmFuY2UgYWRhcHRlciB0byBwaW5uZWQg"
        "b2ZmaWNpYWwgS2FnZ2xlIEFQSXMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFu"
        "bm90YXRpb25zCgppbXBvcnQgZGF0YWNsYXNzZXMKaW1wb3J0IGltcG9ydGxpYgppbXBv"
        "cnQgemlwZmlsZQpmcm9tIGltcG9ydGxpYi5tZXRhZGF0YSBpbXBvcnQgdmVyc2lvbgpm"
        "cm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJv"
        "bSBtb3N0YXphLmV2YWx1YXRpb24uY29udHJhY3RzIGltcG9ydCAoCiAgICBNT0RFTCwK"
        "ICAgIGNhbm9uaWNhbF9qc29uLAogICAgcmVxdWlyZSwKICAgIHNoYTI1NiwKICAgIHZh"
        "bGlkYXRlX21hbmlmZXN0LAopCmZyb20gbW9zdGF6YS5ldmFsdWF0aW9uLnRhc2tzIGlt"
        "cG9ydCBsb2FkX2FnZW50X3Rhc2tzLCB2YWxpZGF0ZV9zbmFwc2hvdF9oaXN0b3J5CmZy"
        "b20gbW9zdGF6YS5wYWNrYWdpbmcgaW1wb3J0IGNvbGxlY3Rfc3VibWlzc2lvbgoKV0hF"
        "RUxfUElOUyA9IHsKICAgICJzd2VnZW1tYS0wLjIuMTEtcHkzLW5vbmUtYW55LndobCI6"
        "ICgKICAgICAgICAiOWFkODc2NTgxZjNhY2RlOGVjM2MzNjk4NThmMDgzNGUyYjlmNDFh"
        "M2NmZmNlM2NhNDJmMDljYjA5MGFlYTlkMyIKICAgICksCiAgICAiYWRrX3N1Ym1pc3Np"
        "b24tMC4yLjEzLXB5My1ub25lLWFueS53aGwiOiAoCiAgICAgICAgImYxNTZmMGJkODQ1"
        "ODBjY2I5MzM0NzI2ZDExNGQ3YTU0ODFmMTZkM2E2ZTI4ODdjYzRkMjc2Y2VkMGVmOGQ5"
        "OWQiCiAgICApLAogICAgImFka19ldmFsX2NvcmUtMC4xLjAtcHkzLW5vbmUtYW55Lndo"
        "bCI6ICgKICAgICAgICAiZTYxNGUxYjJmZThkODg2YTYzNDg3MWU0YzU3Zjk4ODdlMWNj"
        "NjZlOGMyNzMwMGY4NTAzYTQzYzZlM2NhODJmMyIKICAgICksCn0KCgpkZWYgZmlsZV9k"
        "aWdlc3QocGF0aDogUGF0aCkgLT4gc3RyOgogICAgaW1wb3J0IGhhc2hsaWIKCiAgICBo"
        "YXNoZXIgPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBh"
        "cyBzdHJlYW06CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0u"
        "cmVhZCgxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGhhc2hlci51cGRhdGUo"
        "YmxvY2spCiAgICByZXR1cm4gaGFzaGVyLmhleGRpZ2VzdCgpCgoKZGVmIGRlc2NyaWJl"
        "X3J1bnRpbWUoY29uZmlnOiBBbnksIHZsbG1fY29uZmlnOiBBbnkpIC0+IGRpY3Rbc3Ry"
        "LCBBbnldOgogICAgIiIiQ2FwdHVyZSBhY3R1YWwgb3JnYW5pemVyIGNvbmZpZ3VyYXRp"
        "b24gb2JqZWN0cywgd2l0aG91dCBlbmRwb2ludCBjcmVkZW50aWFscy4iIiIKICAgIGxv"
        "YWRlciA9IGltcG9ydGxpYi5pbXBvcnRfbW9kdWxlKCJhZGtfc3VibWlzc2lvbi55YW1s"
        "X2xvYWRlciIpCiAgICByYXcgPSBsb2FkZXIubG9hZF95YW1sKAogICAgICAgIFBhdGgo"
        "Y29uZmlnLnN1Ym1pc3Npb25fZGlyKSAvICJhZ2VudC55YW1sIiwKICAgICAgICByb290"
        "X2Rpcj1QYXRoKGNvbmZpZy5zdWJtaXNzaW9uX2RpciksCiAgICAgICAgbGltaXRzPWNv"
        "bmZpZy5saW1pdHMsCiAgICApCgogICAgZGVmIG1vZGVsX2R1bXAodmFsdWU6IEFueSkg"
        "LT4gQW55OgogICAgICAgIHJldHVybiBOb25lIGlmIHZhbHVlIGlzIE5vbmUgZWxzZSB2"
        "YWx1ZS5tb2RlbF9kdW1wKG1vZGU9Impzb24iKQoKICAgIHZhbHVlID0gewogICAgICAg"
        "ICJpbmZlcmVuY2UiOiB7CiAgICAgICAgICAgICJzZXJ2ZXIiOiBkYXRhY2xhc3Nlcy5h"
        "c2RpY3QodmxsbV9jb25maWcpLAogICAgICAgICAgICAiYWdlbnRfZ2VuZXJhdGVfY29u"
        "dGVudF9jb25maWciOiByYXcuZ2V0KCJnZW5lcmF0ZV9jb250ZW50X2NvbmZpZyIpLAog"
        "ICAgICAgICAgICAib2ZmaWNpYWxfZ2VuZXJhdGlvbl9kZWZhdWx0cyI6IGNvbmZpZy5n"
        "ZW5lcmF0aW9uX2NvbnN0cmFpbnRzLmRlZmF1bHRzLAogICAgICAgICAgICAiY2FjaGUi"
        "OiBtb2RlbF9kdW1wKGNvbmZpZy5jb250ZXh0X2NhY2hlX2NvbmZpZyksCiAgICAgICAg"
        "ICAgICJjb21wYWN0aW9uIjogbW9kZWxfZHVtcChjb25maWcuZXZlbnRzX2NvbXBhY3Rp"
        "b25fY29uZmlnKSwKICAgICAgICB9LAogICAgICAgICJidWRnZXRzIjogewogICAgICAg"
        "ICAgICAiZXZhbHVhdGlvbiI6IGRhdGFjbGFzc2VzLmFzZGljdChjb25maWcuYnVkZ2V0"
        "KSwKICAgICAgICAgICAgImhhcm5lc3MiOiBkYXRhY2xhc3Nlcy5hc2RpY3QoY29uZmln"
        "Lmhhcm5lc3MpLAogICAgICAgIH0sCiAgICAgICAgImNvbmZpZ19zaGEyNTYiOiBzaGEy"
        "NTYoCiAgICAgICAgICAgIGNhbm9uaWNhbF9qc29uKAogICAgICAgICAgICAgICAgewog"
        "ICAgICAgICAgICAgICAgICAgIG5hbWU6IHNoYTI1NihkYXRhKQogICAgICAgICAgICAg"
        "ICAgICAgIGZvciBuYW1lLCBkYXRhIGluIGNvbGxlY3Rfc3VibWlzc2lvbigKICAgICAg"
        "ICAgICAgICAgICAgICAgICAgUGF0aChjb25maWcuc3VibWlzc2lvbl9kaXIpCiAgICAg"
        "ICAgICAgICAgICAgICAgKS5pdGVtcygpCiAgICAgICAgICAgICAgICB9CiAgICAgICAg"
        "ICAgICkKICAgICAgICApLAogICAgfQogICAgIyBOb3JtYWxpemUgdHVwbGVzIGluIG9m"
        "ZmljaWFsIGRhdGFjbGFzc2VzIGV4YWN0bHkgYXMgcGVyc2lzdGVkIEpTT04gZG9lcy4K"
        "ICAgIGltcG9ydCBqc29uCgogICAgcmVzdWx0OiBkaWN0W3N0ciwgQW55XSA9IGpzb24u"
        "bG9hZHMoY2Fub25pY2FsX2pzb24odmFsdWUpKQogICAgcmV0dXJuIHJlc3VsdAoKCmRl"
        "ZiBzbmFwc2hvdF9kaWdlc3QoCiAgICBzbmFwc2hvdHNfZGlyOiBQYXRoLCBpbnN0YW5j"
        "ZV9pZDogc3RyLCByZXBvOiBzdHIsIGRhdGFfcm9vdDogUGF0aAopIC0+IHN0cjoKICAg"
        "ICIiIkJpbmQgYWxsIG9mZmljaWFsIHJlY29uc3RydWN0aW9uIGlucHV0czsgcm9sZXMg"
        "Zm9sbG93IG9yZ2FuaXplciByZXNvbHV0aW9uLiIiIgogICAgcmVzb2x2ZSA9IGltcG9y"
        "dGxpYi5pbXBvcnRfbW9kdWxlKAogICAgICAgICJzd2VnZW1tYS5kZWR1cGxpY2F0aW9u"
        "IgogICAgKS5yZXNvbHZlX3Rhc2tfc25hcHNob3RfcGF0aHMKICAgIGVmZmVjdGl2ZSwg"
        "YmFzZSwgZGVsdGEgPSByZXNvbHZlKHNuYXBzaG90c19kaXIsIGluc3RhbmNlX2lkLCBy"
        "ZXBvKQogICAgY29tcG9uZW50cyA9IFtdCiAgICBmb3Igcm9sZSwgcGF0aCBpbiAoKCJl"
        "ZmZlY3RpdmUiLCBlZmZlY3RpdmUpLCAoImJhc2UiLCBiYXNlKSwgKCJkZWx0YSIsIGRl"
        "bHRhKSk6CiAgICAgICAgaWYgcGF0aCBpcyBub3QgTm9uZToKICAgICAgICAgICAgcGF0"
        "aCA9IFBhdGgocGF0aCkKICAgICAgICAgICAgcmVxdWlyZShwYXRoLmlzX2ZpbGUoKSwg"
        "IkZyb3plbiBzbmFwc2hvdCBjb21wb25lbnQgbWlzc2luZyIpCiAgICAgICAgICAgIGNv"
        "bXBvbmVudHMuYXBwZW5kKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAg"
        "ICAgICJyb2xlIjogcm9sZSwKICAgICAgICAgICAgICAgICAgICAicGF0aCI6IHN0cihw"
        "YXRoLnJlbGF0aXZlX3RvKGRhdGFfcm9vdCkpLAogICAgICAgICAgICAgICAgICAgICJz"
        "aGEyNTYiOiBmaWxlX2RpZ2VzdChwYXRoKSwKICAgICAgICAgICAgICAgIH0KICAgICAg"
        "ICAgICAgKQogICAgcmV0dXJuIHNoYTI1NihjYW5vbmljYWxfanNvbihjb21wb25lbnRz"
        "KSkKCgpkZWYgdmFsaWRhdGVfYmluZGluZ3MoCiAgICBjb25maWc6IEFueSwKICAgIG1h"
        "bmlmZXN0OiBkaWN0W3N0ciwgQW55XSwKICAgICosCiAgICBzZXJ2ZXI6IEFueSwKICAg"
        "IGFyY2hpdmVfcGF0aDogUGF0aCwKICAgIHdoZWVsaG91c2VfZGlyOiBQYXRoLAopIC0+"
        "IE5vbmU6CiAgICAiIiJGYWlsIGJlZm9yZSBhbnkgYWdlbnQgY2FsbCBpZiBkZWNsYXJh"
        "dGlvbnMgZGlmZmVyIGZyb20gZWZmZWN0aXZlIGlucHV0cy4iIiIKICAgIHZhbGlkYXRl"
        "X21hbmlmZXN0KG1hbmlmZXN0KQogICAgcmVxdWlyZSgKICAgICAgICBtYW5pZmVzdFsi"
        "cnVuX2tpbmQiXSA9PSAicmVhbF9rYWdnbGUiLCAiT2ZmaWNpYWwgcnVubmVyIHJlcXVp"
        "cmVzIHJlYWwgcnVuIGtpbmQiCiAgICApCiAgICByZXF1aXJlKG5vdCBjb25maWcuc2tp"
        "cF9hZ2VudF9wYXRjaCwgIlBhdGNoIGdlbmVyYXRpb24gY2Fubm90IGJlIHNraXBwZWQi"
        "KQogICAgcmVxdWlyZShjb25maWcuc2FuZGJveCA9PSAic3VicHJvY2VzcyIsICJVc2Ug"
        "dGhlIG9mZmljaWFsIG5vdGVib29rIHNhbmRib3ggYmFja2VuZCIpCiAgICByZXF1aXJl"
        "KAogICAgICAgIG5vdCBjb25maWcuc2hhcmRfaW5kZXggYW5kIG5vdCBjb25maWcubnVt"
        "X3NoYXJkcyBhbmQgY29uZmlnLmNvbmN1cnJlbmN5ID09IDEsCiAgICAgICAgIlBoYXNl"
        "IDEgcnVucyB0aGUgY29tcGxldGUgY29ob3J0IHNlcmlhbGx5IHdpdGhvdXQgc2hhcmRz"
        "IiwKICAgICkKICAgIHJlcXVpcmUoCiAgICAgICAgY29uZmlnLnRhc2tfaWRzIGlzIE5v"
        "bmUKICAgICAgICBvciBjb25maWcudGFza19pZHMgPT0gW3RbImluc3RhbmNlX2lkIl0g"
        "Zm9yIHQgaW4gbWFuaWZlc3RbImNvaG9ydCJdXSwKICAgICAgICAiQ29uZmlndXJlZCB0"
        "YXNrIElEcyBkaWZmZXIgZnJvbSBjb2hvcnQiLAogICAgKQogICAgbW9kZWxfcGF0aCA9"
        "IFBhdGgoc2VydmVyLmNvbmZpZy5tb2RlbCkKICAgIHJlcXVpcmUoCiAgICAgICAgbW9k"
        "ZWxfcGF0aC5uYW1lID09ICIyIiBhbmQgbW9kZWxfcGF0aC5wYXJlbnQubmFtZSA9PSBN"
        "T0RFTCwKICAgICAgICAiUmVxdWlyZWQgcGlubmVkIDMxQiBtb2RlbCBwYXRoIG1pc3Np"
        "bmciLAogICAgKQogICAgcmVxdWlyZSgKICAgICAgICBzZXJ2ZXIuY29uZmlnLnRlbnNv"
        "cl9wYXJhbGxlbF9zaXplID09IDQKICAgICAgICBhbmQgbm90IHNlcnZlci5jb25maWcu"
        "ZW5hYmxlX2xvcmEKICAgICAgICBhbmQgc2VydmVyLmNvbmZpZy5ob3N0ID09ICIxMjcu"
        "MC4wLjEiLAogICAgICAgICJPZmZpY2lhbCBMNHg0IGxvY2FsIG1vZGVsIHNlcnZpbmcg"
        "cmVxdWlyZWQiLAogICAgKQogICAgcmVxdWlyZSgKICAgICAgICBjb25maWcubW9kZWxz"
        "Lmxpc3RfYXZhaWxhYmxlKCkgPT0gW01PREVMXSwKICAgICAgICAiUmVnaXN0cnkgbXVz"
        "dCBjb250YWluIG9ubHkgdGhlIHJlcXVpcmVkIG1vZGVsIiwKICAgICkKICAgIGJpbmRp"
        "bmcgPSBjb25maWcubW9kZWxzLmdldChNT0RFTCkKICAgIHJlcXVpcmUoCiAgICAgICAg"
        "Z2V0YXR0cihiaW5kaW5nLCAibW9kZWwiLCBOb25lKQogICAgICAgID09ICJvcGVuYWkv"
        "IgogICAgICAgICsgc3RyKGdldGF0dHIoc2VydmVyLCAiZWZmZWN0aXZlX21vZGVsX3Bh"
        "dGgiLCBzZXJ2ZXIuY29uZmlnLm1vZGVsKSksCiAgICAgICAgIlJlZ2lzdHJ5IG1vZGVs"
        "IGRvZXMgbm90IG1hdGNoIHRoZSBhY3R1YWwgc2VydmVyIiwKICAgICkKICAgIHJlcXVp"
        "cmUoCiAgICAgICAgZ2V0YXR0cihiaW5kaW5nLCAiX2FkZGl0aW9uYWxfYXJncyIsIHt9"
        "KS5nZXQoImFwaV9iYXNlIikgPT0gc2VydmVyLmJhc2VfdXJsLAogICAgICAgICJSZWdp"
        "c3RyeSBlbmRwb2ludCBkb2VzIG5vdCBtYXRjaCB0aGUgbG9jYWwgc2VydmVyIiwKICAg"
        "ICkKICAgIHJ1bnRpbWUgPSBkZXNjcmliZV9ydW50aW1lKGNvbmZpZywgc2VydmVyLmNv"
        "bmZpZykKICAgIGZvciBmaWVsZCBpbiAoImluZmVyZW5jZSIsICJidWRnZXRzIiwgImNv"
        "bmZpZ19zaGEyNTYiKToKICAgICAgICByZXF1aXJlKAogICAgICAgICAgICBjYW5vbmlj"
        "YWxfanNvbihydW50aW1lW2ZpZWxkXSkgPT0gY2Fub25pY2FsX2pzb24obWFuaWZlc3Rb"
        "ZmllbGRdKSwKICAgICAgICAgICAgZiJBY3R1YWwgcnVudGltZSBkaWZmZXJzIGZyb20g"
        "bWFuaWZlc3Qge2ZpZWxkfSIsCiAgICAgICAgKQogICAgcmVxdWlyZSgKICAgICAgICBm"
        "aWxlX2RpZ2VzdChhcmNoaXZlX3BhdGgpID09IG1hbmlmZXN0WyJhcmNoaXZlX3NoYTI1"
        "NiJdLCAiQXJjaGl2ZSBoYXNoIG1pc21hdGNoIgogICAgKQogICAgZmlsZXMgPSBjb2xs"
        "ZWN0X3N1Ym1pc3Npb24oUGF0aChjb25maWcuc3VibWlzc2lvbl9kaXIpKQogICAgd2l0"
        "aCB6aXBmaWxlLlppcEZpbGUoYXJjaGl2ZV9wYXRoKSBhcyBhcmNoaXZlOgogICAgICAg"
        "IG5hbWVzID0gYXJjaGl2ZS5uYW1lbGlzdCgpCiAgICAgICAgcmVxdWlyZSgKICAgICAg"
        "ICAgICAgbGVuKG5hbWVzKSA9PSBsZW4oc2V0KG5hbWVzKSkgYW5kIHNldChuYW1lcykg"
        "PT0gc2V0KGZpbGVzKSwKICAgICAgICAgICAgIkFyY2hpdmUgZGlmZmVycyBmcm9tIGV4"
        "ZWN1dGVkIHN1Ym1pc3Npb24gY2xvc3VyZSIsCiAgICAgICAgKQogICAgICAgIGZvciBu"
        "YW1lLCBjb250ZW50IGluIGZpbGVzLml0ZW1zKCk6CiAgICAgICAgICAgIHJlcXVpcmUo"
        "CiAgICAgICAgICAgICAgICBhcmNoaXZlLnJlYWQobmFtZSkgPT0gY29udGVudCwgIkFy"
        "Y2hpdmUvZXhlY3V0ZWQgY29uZmlnIGJ5dGVzIGRpZmZlciIKICAgICAgICAgICAgKQog"
        "ICAgZGlzY292ZXJ5ID0gaW1wb3J0bGliLmltcG9ydF9tb2R1bGUoImFka19zdWJtaXNz"
        "aW9uLmRpc2NvdmVyeSIpCiAgICByZXF1aXJlKAogICAgICAgIGRpc2NvdmVyeS5kaXNj"
        "b3Zlcl9kZWNsYXJlZF9tb2RlbHMoUGF0aChjb25maWcuc3VibWlzc2lvbl9kaXIpKSA9"
        "PSB7TU9ERUx9LAogICAgICAgICJFdmVyeSBkZWNsYXJhdGl2ZSBhZ2VudCBtdXN0IHVz"
        "ZSB0aGUgcmVxdWlyZWQgbW9kZWwiLAogICAgKQogICAgZm9yIHBhY2thZ2UsIGV4cGVj"
        "dGVkIGluICgKICAgICAgICAoInN3ZWdlbW1hIiwgIjAuMi4xMSIpLAogICAgICAgICgi"
        "YWRrLXN1Ym1pc3Npb24iLCAiMC4yLjEzIiksCiAgICAgICAgKCJhZGstZXZhbC1jb3Jl"
        "IiwgIjAuMS4wIiksCiAgICApOgogICAgICAgIHJlcXVpcmUodmVyc2lvbihwYWNrYWdl"
        "KSA9PSBleHBlY3RlZCwgIlBpbm5lZCBvZmZpY2lhbCBwYWNrYWdlcyByZXF1aXJlZCIp"
        "CiAgICBmb3IgbmFtZSwgZXhwZWN0ZWQgaW4gV0hFRUxfUElOUy5pdGVtcygpOgogICAg"
        "ICAgIHJlcXVpcmUoCiAgICAgICAgICAgIGZpbGVfZGlnZXN0KHdoZWVsaG91c2VfZGly"
        "IC8gbmFtZSkgPT0gZXhwZWN0ZWQsCiAgICAgICAgICAgICJQaW5uZWQgb2ZmaWNpYWwg"
        "d2hlZWwgY2hhbmdlZCIsCiAgICAgICAgKQogICAgcmVxdWlyZSgKICAgICAgICBtYW5p"
        "ZmVzdFsicmVzb3VyY2VzIl1bImhhcm5lc3MiXVsic2hhMjU2Il0KICAgICAgICA9PSBX"
        "SEVFTF9QSU5TWyJzd2VnZW1tYS0wLjIuMTEtcHkzLW5vbmUtYW55LndobCJdLAogICAg"
        "ICAgICJIYXJuZXNzIGRlY2xhcmF0aW9uIGRpZmZlcnMgZnJvbSB2ZXJpZmllZCB3aGVl"
        "bCIsCiAgICApCiAgICB3aGVlbF9pbnZlbnRvcnkgPSBbCiAgICAgICAgeyJuYW1lIjog"
        "cC5uYW1lLCAic2hhMjU2IjogZmlsZV9kaWdlc3QocCl9CiAgICAgICAgZm9yIHAgaW4g"
        "c29ydGVkKHdoZWVsaG91c2VfZGlyLmdsb2IoIioud2hsIikpCiAgICBdCiAgICByZXF1"
        "aXJlKAogICAgICAgIHNoYTI1NihjYW5vbmljYWxfanNvbih3aGVlbF9pbnZlbnRvcnkp"
        "KQogICAgICAgID09IG1hbmlmZXN0WyJyZXNvdXJjZXMiXVsid2hlZWxob3VzZSJdWyJz"
        "aGEyNTYiXSwKICAgICAgICAiV2hlZWxob3VzZSBpbnZlbnRvcnkgZGlmZmVycyBmcm9t"
        "IG1hbmlmZXN0IiwKICAgICkKICAgIHRhc2tzX3BhdGggPSBQYXRoKGNvbmZpZy50YXNr"
        "c19wYXRoKQogICAgcmVxdWlyZSgKICAgICAgICBmaWxlX2RpZ2VzdCh0YXNrc19wYXRo"
        "KSA9PSBtYW5pZmVzdFsicmVzb3VyY2VzIl1bImRhdGFzZXQiXVsic2hhMjU2Il0sCiAg"
        "ICAgICAgIkRhdGFzZXQgY2hhbmdlZCBhZnRlciBzZWxlY3Rpb24iLAogICAgKQogICAg"
        "dmlzaWJsZSA9IHt0YXNrLmluc3RhbmNlX2lkOiB0YXNrIGZvciB0YXNrIGluIGxvYWRf"
        "YWdlbnRfdGFza3ModGFza3NfcGF0aCl9CiAgICBmb3Igc2VsZWN0ZWQgaW4gbWFuaWZl"
        "c3RbImNvaG9ydCJdOgogICAgICAgIHRhc2sgPSB2aXNpYmxlLmdldChzZWxlY3RlZFsi"
        "aW5zdGFuY2VfaWQiXSkKICAgICAgICByZXF1aXJlKAogICAgICAgICAgICB0YXNrIGlz"
        "IG5vdCBOb25lCiAgICAgICAgICAgIGFuZCB0YXNrLmJhc2VfY29tbWl0ID09IHNlbGVj"
        "dGVkWyJiYXNlX2NvbW1pdCJdCiAgICAgICAgICAgIGFuZCB0YXNrLnJlcG8gPT0gc2Vs"
        "ZWN0ZWRbInJlcG8iXSwKICAgICAgICAgICAgIkZyb3plbiB0YXNrIG1ldGFkYXRhIG1p"
        "c21hdGNoIiwKICAgICAgICApCiAgICAgICAgcmVxdWlyZSgKICAgICAgICAgICAgc25h"
        "cHNob3RfZGlnZXN0KAogICAgICAgICAgICAgICAgUGF0aChjb25maWcuc25hcHNob3Rz"
        "X2RpciksCiAgICAgICAgICAgICAgICBzZWxlY3RlZFsiaW5zdGFuY2VfaWQiXSwKICAg"
        "ICAgICAgICAgICAgIHNlbGVjdGVkWyJyZXBvIl0sCiAgICAgICAgICAgICAgICB0YXNr"
        "c19wYXRoLnBhcmVudCwKICAgICAgICAgICAgKQogICAgICAgICAgICA9PSBzZWxlY3Rl"
        "ZFsic25hcHNob3Rfc2hhMjU2Il0sCiAgICAgICAgICAgICJGcm96ZW4gcmVjb25zdHJ1"
        "Y3Rpb24gaW5wdXRzIGNoYW5nZWQiLAogICAgICAgICkKICAgICAgICByZWNlaXB0cyA9"
        "IG1hbmlmZXN0WyJlbnZpcm9ubWVudCJdLmdldCgic25hcHNob3RfdmFsaWRhdGlvbiIs"
        "IHt9KQogICAgICAgIHJlcXVpcmUoc2VsZWN0ZWRbImluc3RhbmNlX2lkIl0gaW4gcmVj"
        "ZWlwdHMsICJTbmFwc2hvdCBoaXN0b3J5IHJlY2VpcHQgbWlzc2luZyIpCiAgICAgICAg"
        "cmVzb2x2ZSA9IGltcG9ydGxpYi5pbXBvcnRfbW9kdWxlKAogICAgICAgICAgICAic3dl"
        "Z2VtbWEuZGVkdXBsaWNhdGlvbiIKICAgICAgICApLnJlc29sdmVfdGFza19zbmFwc2hv"
        "dF9wYXRocwogICAgICAgIGVmZmVjdGl2ZSwgYmFzZSwgZGVsdGEgPSByZXNvbHZlKAog"
        "ICAgICAgICAgICBjb25maWcuc25hcHNob3RzX2Rpciwgc2VsZWN0ZWRbImluc3RhbmNl"
        "X2lkIl0sIHNlbGVjdGVkWyJyZXBvIl0KICAgICAgICApCiAgICAgICAgcmVxdWlyZSgK"
        "ICAgICAgICAgICAgYmFzZSBpcyBOb25lIGFuZCBkZWx0YSBpcyBOb25lLAogICAgICAg"
        "ICAgICAiSGlzdG9yeSByZWNlaXB0IHJlcXVpcmVzIHJldmlld2VkIHN0YW5kYWxvbmUg"
        "c25hcHNob3QiLAogICAgICAgICkKICAgICAgICB2YWxpZGF0ZV9zbmFwc2hvdF9oaXN0"
        "b3J5KAogICAgICAgICAgICBQYXRoKGVmZmVjdGl2ZSksIHNlbGVjdGVkLCByZWNlaXB0"
        "c1tzZWxlY3RlZFsiaW5zdGFuY2VfaWQiXV0KICAgICAgICApCgoKYXN5bmMgZGVmIGV2"
        "YWx1YXRlX29mZmljaWFsKAogICAgY29uZmlnOiBBbnksCiAgICBtYW5pZmVzdDogZGlj"
        "dFtzdHIsIEFueV0sCiAgICAqLAogICAgc2VydmVyOiBBbnkgPSBOb25lLAogICAgYXJj"
        "aGl2ZV9wYXRoOiBQYXRoIHwgTm9uZSA9IE5vbmUsCiAgICB3aGVlbGhvdXNlX2Rpcjog"
        "UGF0aCB8IE5vbmUgPSBOb25lLAopIC0+IGxpc3RbUGF0aF06CiAgICAiIiJEZWxlZ2F0"
        "ZSB0byBvZmZpY2lhbCBldmFsdWF0ZV90YXNrIGFuZCBwZXJzaXN0IHNvdXJjZS1ub3Jt"
        "YWxpemFibGUgcmF3IHJlc3VsdHMuCgogICAgTW9kZWwgc3RhcnR1cCByZW1haW5zIHRo"
        "ZSBvZmZpY2lhbCBub3RlYm9vaydzIFZsbG1TZXJ2ZXIgcmVzcG9uc2liaWxpdHkuIFRo"
        "ZQogICAgcmF3IGV4dGVuc2lvbiByZWNvcmRzIGV4YWN0bHkgd2hpY2ggZ2VuZXJhdGVk"
        "IHBhdGNoIHRoZSBvcmdhbml6ZXIgdmVyaWZpZWQ7CiAgICBubyByZWZlcmVuY2UgcGF0"
        "Y2gvdGVzdCBjb250ZW50cyBlbnRlciB0aGVzZSBvdXRwdXRzLgogICAgIiIiCiAgICB2"
        "YWxpZGF0ZV9tYW5pZmVzdChtYW5pZmVzdCkKICAgIHJlcXVpcmUoCiAgICAgICAgbWFu"
        "aWZlc3RbInJ1bl9raW5kIl0gPT0gInJlYWxfa2FnZ2xlIiwgIk9mZmljaWFsIHJ1bm5l"
        "ciByZXF1aXJlcyByZWFsIHJ1biBraW5kIgogICAgKQogICAgcmVxdWlyZShQYXRoKCIv"
        "a2FnZ2xlL2lucHV0IikuaXNfZGlyKCksICJPZmZpY2lhbCBleGVjdXRpb24gcmVxdWly"
        "ZXMgS2FnZ2xlIikKICAgIHJlcXVpcmUoCiAgICAgICAgc2VydmVyIGlzIG5vdCBOb25l"
        "IGFuZCBhcmNoaXZlX3BhdGggaXMgbm90IE5vbmUgYW5kIHdoZWVsaG91c2VfZGlyIGlz"
        "IG5vdCBOb25lLAogICAgICAgICJBY3R1YWwgc2VydmVyLCBhcmNoaXZlIGFuZCB3aGVl"
        "bGhvdXNlIGJpbmRpbmdzIGFyZSByZXF1aXJlZCIsCiAgICApCiAgICByZXF1aXJlKAog"
        "ICAgICAgIHNlcnZlci5wcm9jZXNzIGlzIG5vdCBOb25lIGFuZCBzZXJ2ZXIucHJvY2Vz"
        "cy5wb2xsKCkgaXMgTm9uZSwKICAgICAgICAiQWN0dWFsIG1vZGVsIHNlcnZlciBwcm9j"
        "ZXNzIG11c3QgYmUgcnVubmluZyIsCiAgICApCiAgICByZXF1aXJlKFBhdGgoc2VydmVy"
        "LmNvbmZpZy5tb2RlbCkuaXNfZGlyKCksICJBY3R1YWwgcmVxdWlyZWQgbW9kZWwgbW91"
        "bnQgbWlzc2luZyIpCiAgICBhc3NlcnQgYXJjaGl2ZV9wYXRoIGlzIG5vdCBOb25lIGFu"
        "ZCB3aGVlbGhvdXNlX2RpciBpcyBub3QgTm9uZQogICAgdmFsaWRhdGVfYmluZGluZ3Mo"
        "CiAgICAgICAgY29uZmlnLAogICAgICAgIG1hbmlmZXN0LAogICAgICAgIHNlcnZlcj1z"
        "ZXJ2ZXIsCiAgICAgICAgYXJjaGl2ZV9wYXRoPWFyY2hpdmVfcGF0aCwKICAgICAgICB3"
        "aGVlbGhvdXNlX2Rpcj13aGVlbGhvdXNlX2RpciwKICAgICkKICAgIG9mZmljaWFsX21v"
        "ZGVscyA9IGltcG9ydGxpYi5pbXBvcnRfbW9kdWxlKCJzd2VnZW1tYS5tb2RlbHMiKQog"
        "ICAgb2ZmaWNpYWxfZXZhbHVhdGUgPSBpbXBvcnRsaWIuaW1wb3J0X21vZHVsZSgic3dl"
        "Z2VtbWEuZXZhbHVhdGUiKQogICAgdmVyaWZpY2F0aW9uID0gaW1wb3J0bGliLmltcG9y"
        "dF9tb2R1bGUoInN3ZWdlbW1hLmhhcm5lc3MudmVyaWZpY2F0aW9uIikKICAgIHRhc2tz"
        "ID0gewogICAgICAgIHRhc2suaW5zdGFuY2VfaWQ6IHRhc2sKICAgICAgICBmb3IgdGFz"
        "ayBpbiBvZmZpY2lhbF9tb2RlbHMubG9hZF90YXNrcyhQYXRoKGNvbmZpZy50YXNrc19w"
        "YXRoKSkKICAgIH0KICAgIG91dHB1dCA9IFBhdGgoY29uZmlnLnJlc3VsdHNfZGlyKQog"
        "ICAgcmVxdWlyZSgKICAgICAgICBub3Qgb3V0cHV0LmV4aXN0cygpIG9yIG5vdCBhbnko"
        "b3V0cHV0Lml0ZXJkaXIoKSksCiAgICAgICAgIlJlc3VsdCBkaXJlY3RvcnkgbXVzdCBi"
        "ZSBmcmVzaCIsCiAgICApCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlz"
        "dF9vaz1UcnVlKQogICAgZXZhbHVhdG9yID0gb2ZmaWNpYWxfZXZhbHVhdGUuRXZhbHVh"
        "dG9yKGNvbmZpZykKICAgIHBhdGhzOiBsaXN0W1BhdGhdID0gW10KICAgIHRyeToKICAg"
        "ICAgICBmb3IgaW5kZXgsIHNlbGVjdGVkIGluIGVudW1lcmF0ZShtYW5pZmVzdFsiY29o"
        "b3J0Il0sIHN0YXJ0PTEpOgogICAgICAgICAgICAjIFVzZSB0aGUgcGlubmVkIGV2YWx1"
        "YXRvcidzIG93biBob3N0LW9ubHkgc2VjcmV0IGh5ZHJhdGlvbiBzbyB0aGUKICAgICAg"
        "ICAgICAgIyByZWNvcmRlZCBlZmZlY3RpdmUgcGF0Y2ggbWF0Y2hlcyBpdHMgcHJvdGVj"
        "dGVkIHRhcmdldC10ZXN0IHBhdGhzLgogICAgICAgICAgICB0YXNrID0gZXZhbHVhdG9y"
        "Ll9oeWRyYXRlX3Rhc2tfZnJvbV9zZWNyZXQodGFza3Nbc2VsZWN0ZWRbImluc3RhbmNl"
        "X2lkIl1dKQogICAgICAgICAgICByZXN1bHQgPSBhd2FpdCBldmFsdWF0b3IuZXZhbHVh"
        "dGVfdGFzaygKICAgICAgICAgICAgICAgIHRhc2s9dGFzaywgdGFza19pbmRleD1pbmRl"
        "eCwgdG90YWxfdGFza3M9bGVuKG1hbmlmZXN0WyJjb2hvcnQiXSkKICAgICAgICAgICAg"
        "KQogICAgICAgICAgICBpZiBub3QgcmVzdWx0LnRyYWNlX3BhdGggYW5kIHJlc3VsdC50"
        "cmFjZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHJlc3VsdC50cmFjZV9wYXRo"
        "ID0gdmVyaWZpY2F0aW9uLnNhdmVfdHJhY2VfYXJ0aWZhY3QoCiAgICAgICAgICAgICAg"
        "ICAgICAgcmVzdWx0LnRyYWNlLCBvdXRwdXQsIHRhc2suaW5zdGFuY2VfaWQKICAgICAg"
        "ICAgICAgICAgICkKICAgICAgICAgICAgcmF3ID0gcmVzdWx0Lm1vZGVsX2R1bXAobW9k"
        "ZT0ianNvbiIsIGV4Y2x1ZGU9eyJ0cmFjZSJ9KQogICAgICAgICAgICBlZmZlY3RpdmUg"
        "PSB2ZXJpZmljYXRpb24uc3RyaXBfcHJvdGVjdGVkX2ZpbGVzX2Zyb21fcGF0Y2goCiAg"
        "ICAgICAgICAgICAgICByZXN1bHQuYWdlbnRfcGF0Y2gsCiAgICAgICAgICAgICAgICBl"
        "eHRyYV9wcm90ZWN0ZWRfcGF0aHM9dGFzay50ZXN0X2ZpbGVzLAogICAgICAgICAgICAp"
        "CiAgICAgICAgICAgIHJhd1siX21vc3RhemEiXSA9IHsKICAgICAgICAgICAgICAgICJz"
        "Y2hlbWFfdmVyc2lvbiI6IDEsCiAgICAgICAgICAgICAgICAiZWZmZWN0aXZlX2FnZW50"
        "X3BhdGNoIjogZWZmZWN0aXZlLAogICAgICAgICAgICAgICAgIm9yaWdpbmFsX3BhdGNo"
        "X3NoYTI1NiI6IHNoYTI1NihyZXN1bHQuYWdlbnRfcGF0Y2guZW5jb2RlKCkpLAogICAg"
        "ICAgICAgICB9CiAgICAgICAgICAgIHBhdGggPSBvdXRwdXQgLyBmIm9mZmljaWFsLXJl"
        "c3VsdC17aW5kZXg6MDRkfS5qc29uIgogICAgICAgICAgICB3aXRoIHBhdGgub3Blbigi"
        "eGIiKSBhcyBzdHJlYW06CiAgICAgICAgICAgICAgICBzdHJlYW0ud3JpdGUoY2Fub25p"
        "Y2FsX2pzb24ocmF3KSkKICAgICAgICAgICAgcGF0aHMuYXBwZW5kKHBhdGgpCiAgICBm"
        "aW5hbGx5OgogICAgICAgIGV2YWx1YXRvci5kb2NrZXIuY2xlYW51cF9hbGwoKQogICAg"
        "cmV0dXJuIHBhdGhzCg=="
    ),
    "src/mostaza/evaluation/tasks.py": (
        "IiIiU2VwYXJhdGUgYWdlbnQtdmlzaWJsZSBpc3N1ZSBkYXRhIGZyb20gcHJpdmF0ZSBw"
        "b3N0LWdlbmVyYXRpb24gcmVmZXJlbmNlcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBv"
        "cnQgYW5ub3RhdGlvbnMKCmltcG9ydCBpbwppbXBvcnQganNvbgppbXBvcnQgcmUKaW1w"
        "b3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRhcmZpbGUKZnJvbSBkYXRhY2xhc3NlcyBpbXBv"
        "cnQgYXNkaWN0LCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJl"
        "UG9zaXhQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCgpAZGF0YWNsYXNzKGZyb3pl"
        "bj1UcnVlKQpjbGFzcyBBZ2VudFRhc2s6CiAgICAiIiJUaGUgZW50aXJlIGFsbG93ZWQg"
        "aXNzdWUgY29udGV4dC4gTmV2ZXIgc2VyaWFsaXplIHRoZSBzb3VyY2UgdGFzayBvYmpl"
        "Y3QuIiIiCgogICAgaW5zdGFuY2VfaWQ6IHN0cgogICAgcmVwbzogc3RyCiAgICBiYXNl"
        "X2NvbW1pdDogc3RyCiAgICBwcm9ibGVtX3N0YXRlbWVudDogc3RyCgogICAgZGVmIHRv"
        "X2pzb24oc2VsZikgLT4gc3RyOgogICAgICAgIHJldHVybiBqc29uLmR1bXBzKGFzZGlj"
        "dChzZWxmKSwgc29ydF9rZXlzPVRydWUsIGVuc3VyZV9hc2NpaT1GYWxzZSkKCgpkZWYg"
        "YWdlbnRfdGFzayhtZXRhZGF0YTogZGljdFtzdHIsIEFueV0pIC0+IEFnZW50VGFzazoK"
        "ICAgIGZpZWxkcyA9ICgiaW5zdGFuY2VfaWQiLCAicmVwbyIsICJiYXNlX2NvbW1pdCIs"
        "ICJwcm9ibGVtX3N0YXRlbWVudCIpCiAgICBpZiBhbnkobm90IGlzaW5zdGFuY2UobWV0"
        "YWRhdGEuZ2V0KGspLCBzdHIpIG9yIG5vdCBtZXRhZGF0YVtrXSBmb3IgayBpbiBmaWVs"
        "ZHMpOgogICAgICAgICMgRG8gbm90IGluY2x1ZGUgcmVqZWN0ZWQgdmFsdWVzOiBlcnJv"
        "ciBtZXNzYWdlcyBtYXkgYmVjb21lIGFnZW50IGxvZ3MuCiAgICAgICAgcmFpc2UgVmFs"
        "dWVFcnJvcigiVGFzayByZXF1aXJlcyBub25lbXB0eSBhbGxvd2xpc3RlZCBzdHJpbmcg"
        "ZmllbGRzIikKICAgIGlmIG5vdCByZS5mdWxsbWF0Y2gociJbMC05YS1mXXs0MH0iLCBt"
        "ZXRhZGF0YVsiYmFzZV9jb21taXQiXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi"
        "VGFzayBiYXNlX2NvbW1pdCBtdXN0IGJlIGFuIGV4YWN0IEdpdCBjb21taXQiKQogICAg"
        "cmV0dXJuIEFnZW50VGFzaygqKntrZXk6IG1ldGFkYXRhW2tleV0gZm9yIGtleSBpbiBm"
        "aWVsZHN9KQoKCmRlZiBsb2FkX2FnZW50X3Rhc2tzKHBhdGg6IFBhdGgpIC0+IGxpc3Rb"
        "QWdlbnRUYXNrXToKICAgIHRhc2tzOiBsaXN0W0FnZW50VGFza10gPSBbXQogICAgc2Vl"
        "bjogc2V0W3N0cl0gPSBzZXQoKQogICAgZm9yIGxpbmUgaW4gcGF0aC5yZWFkX3RleHQo"
        "ZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpOgogICAgICAgIGlmIG5vdCBsaW5l"
        "LnN0cmlwKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdHJ5OgogICAgICAg"
        "ICAgICBkYXRhID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgIGV4Y2VwdCBqc29uLkpT"
        "T05EZWNvZGVFcnJvcjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxp"
        "ZCB0YXNrIG1ldGFkYXRhIEpTT04iKSBmcm9tIE5vbmUKICAgICAgICBpZiBub3QgaXNp"
        "bnN0YW5jZShkYXRhLCBkaWN0KToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi"
        "VGFzayBtZXRhZGF0YSBtdXN0IGJlIGFuIG9iamVjdCIpCiAgICAgICAgdGFzayA9IGFn"
        "ZW50X3Rhc2soZGF0YSkKICAgICAgICBpZiB0YXNrLmluc3RhbmNlX2lkIGluIHNlZW46"
        "CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkR1cGxpY2F0ZSB0YXNrIElEIikK"
        "ICAgICAgICBzZWVuLmFkZCh0YXNrLmluc3RhbmNlX2lkKQogICAgICAgIHRhc2tzLmFw"
        "cGVuZCh0YXNrKQogICAgcmV0dXJuIHRhc2tzCgoKZGVmIGV4cG9ydF9iYXNlX3NuYXBz"
        "aG90KHJlcG9zaXRvcnk6IFBhdGgsIHRhc2s6IEFnZW50VGFzaywgZGVzdGluYXRpb246"
        "IFBhdGgpIC0+IE5vbmU6CiAgICAiIiJFeHBvcnQgZXhhY3RseSBiYXNlX2NvbW1pdCwg"
        "d2l0aG91dCByZWZzLCBoaXN0b3J5LCB1bnRyYWNrZWQgZmlsZXMgb3IgbGlua3MuCgog"
        "ICAgVGhlIHNvdXJjZSByZXBvc2l0b3J5IGFuZCB0YXNrIEpTT05MIG11c3QgcmVtYWlu"
        "IG91dHNpZGUgdGhlIGFnZW50IHNhbmRib3guCiAgICBUaGlzIGhvc3QgaGVscGVyIGlz"
        "IG5vdCBhbiBpbXBsZW1lbnRhdGlvbiBvZiBLYWdnbGUncyBzYW5kYm94IHBpcGVsaW5l"
        "LgogICAgIiIiCiAgICBpZiBkZXN0aW5hdGlvbi5leGlzdHMoKSBvciBkZXN0aW5hdGlv"
        "bi5pc19zeW1saW5rKCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU25hcHNob3Qg"
        "ZGVzdGluYXRpb24gbXVzdCBub3QgYWxyZWFkeSBleGlzdCIpCiAgICBpZiBub3QgcmUu"
        "ZnVsbG1hdGNoKHIiWzAtOWEtZl17NDB9IiwgdGFzay5iYXNlX2NvbW1pdCk6CiAgICAg"
        "ICAgcmFpc2UgVmFsdWVFcnJvcigiVGFzayBiYXNlX2NvbW1pdCBtdXN0IGJlIGFuIGV4"
        "YWN0IEdpdCBjb21taXQiKQogICAgcmVzdWx0ID0gc3VicHJvY2Vzcy5ydW4oCiAgICAg"
        "ICAgWyJnaXQiLCAiLUMiLCBzdHIocmVwb3NpdG9yeS5yZXNvbHZlKCkpLCAiYXJjaGl2"
        "ZSIsIHRhc2suYmFzZV9jb21taXRdLAogICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUs"
        "CiAgICAgICAgY2hlY2s9RmFsc2UsCiAgICApCiAgICBpZiByZXN1bHQucmV0dXJuY29k"
        "ZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYW5ub3QgZXhwb3J0IHRoZSBzZWxl"
        "Y3RlZCBiYXNlIGNvbW1pdCIpCiAgICB3aXRoIHRhcmZpbGUub3BlbihmaWxlb2JqPWlv"
        "LkJ5dGVzSU8ocmVzdWx0LnN0ZG91dCkpIGFzIGFyY2hpdmU6CiAgICAgICAgbWVtYmVy"
        "cyA9IGFyY2hpdmUuZ2V0bWVtYmVycygpCiAgICAgICAgZm9yIG1lbWJlciBpbiBtZW1i"
        "ZXJzOgogICAgICAgICAgICBwYXRoID0gUHVyZVBvc2l4UGF0aChtZW1iZXIubmFtZSkK"
        "ICAgICAgICAgICAgaWYgKAogICAgICAgICAgICAgICAgcGF0aC5pc19hYnNvbHV0ZSgp"
        "CiAgICAgICAgICAgICAgICBvciAiLi4iIGluIHBhdGgucGFydHMKICAgICAgICAgICAg"
        "ICAgIG9yIGFueShwYXJ0LmNhc2Vmb2xkKCkgPT0gIi5naXQiIGZvciBwYXJ0IGluIHBh"
        "dGgucGFydHMpCiAgICAgICAgICAgICAgICBvciBub3QgKG1lbWJlci5pc2ZpbGUoKSBv"
        "ciBtZW1iZXIuaXNkaXIoKSkKICAgICAgICAgICAgKToKICAgICAgICAgICAgICAgIHJh"
        "aXNlIFZhbHVlRXJyb3IoIlVuc2FmZSBzbmFwc2hvdCBtZW1iZXIiKQogICAgICAgIGRl"
        "c3RpbmF0aW9uLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgICAgICBmb3IgbWVtYmVyIGlu"
        "IG1lbWJlcnM6CiAgICAgICAgICAgIHRhcmdldCA9IGRlc3RpbmF0aW9uLmpvaW5wYXRo"
        "KCpQdXJlUG9zaXhQYXRoKG1lbWJlci5uYW1lKS5wYXJ0cykKICAgICAgICAgICAgaWYg"
        "bWVtYmVyLmlzZGlyKCk6CiAgICAgICAgICAgICAgICB0YXJnZXQubWtkaXIocGFyZW50"
        "cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAg"
        "ICAgICAgdGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRy"
        "dWUpCiAgICAgICAgICAgICAgICBzdHJlYW0gPSBhcmNoaXZlLmV4dHJhY3RmaWxlKG1l"
        "bWJlcikKICAgICAgICAgICAgICAgIGlmIHN0cmVhbSBpcyBOb25lOgogICAgICAgICAg"
        "ICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgc25hcHNob3QgbWVtYmVy"
        "IikKICAgICAgICAgICAgICAgIHdpdGggc3RyZWFtLCB0YXJnZXQub3BlbigieGIiKSBh"
        "cyBvdXRwdXQ6CiAgICAgICAgICAgICAgICAgICAgb3V0cHV0LndyaXRlKHN0cmVhbS5y"
        "ZWFkKCkpCiAgICAgICAgICAgICAgICB0YXJnZXQuY2htb2QoMG83NTUgaWYgbWVtYmVy"
        "Lm1vZGUgJiAwbzExMSBlbHNlIDBvNjQ0KQoKCmRlZiB2YWxpZGF0ZV9zbmFwc2hvdF9o"
        "aXN0b3J5KAogICAgYXJjaGl2ZV9wYXRoOiBQYXRoLCBzZWxlY3RlZDogZGljdFtzdHIs"
        "IEFueV0sIHJlY2VpcHQ6IGRpY3Rbc3RyLCBBbnldCikgLT4gTm9uZToKICAgICIiIlZh"
        "bGlkYXRlIGF1dGhlbnRpY2F0ZWQgZmFzdC1leHBvcnQgc25hcHNob3QgaGlzdG9yeSBh"
        "Z2FpbnN0IGl0cyB0cmVlIHJlY2VpcHQuCgogICAgT2ZmaWNpYWwgZmFzdC1pbXBvcnQg"
        "bWF5IHJld3JpdGUgY29tbWl0IElEcy4gT3JpZ2luYWwgYmFzZSBpZGVudGl0eSBpcyBi"
        "b3VuZAogICAgdGhyb3VnaCB0aGUgdmVyaWZpZWQgb3JpZ2luYWwgdHJlZSBhbmQgc291"
        "cmNlIGhhc2gsIG5ldmVyIHRpbWVzdGFtcHMgb3IgYQogICAgcmVxdWlyZW1lbnQgdGhh"
        "dCB0aGUgb3JpZ2luYWwgY29tbWl0IG9iamVjdCBzdXJ2aXZlIGV4cG9ydC4KICAgICIi"
        "IgogICAgaW1wb3J0IG9zCiAgICBpbXBvcnQgdGVtcGZpbGUKCiAgICBmcm9tIG1vc3Rh"
        "emEuZXZhbHVhdGlvbi5jb250cmFjdHMgaW1wb3J0IHJlcXVpcmUKICAgIGZyb20gbW9z"
        "dGF6YS5yZXNvdXJjZXMgaW1wb3J0IHNoYTI1NiBhcyBmaWxlX2RpZ2VzdAoKICAgIHZh"
        "bGlkYXRpb24gPSByZWNlaXB0LmdldCgidmFsaWRhdGlvbiIsIHt9KQogICAgcmVxdWly"
        "ZSgKICAgICAgICByZWNlaXB0LmdldCgia2luZCIpID09ICJzbmFwc2hvdCIKICAgICAg"
        "ICBhbmQgcmVjZWlwdC5nZXQoImluc3RhbmNlX2lkIikgPT0gc2VsZWN0ZWRbImluc3Rh"
        "bmNlX2lkIl0KICAgICAgICBhbmQgZmlsZV9kaWdlc3QoYXJjaGl2ZV9wYXRoKSA9PSBy"
        "ZWNlaXB0LmdldCgic2hhMjU2IiksCiAgICAgICAgIlNuYXBzaG90IHJlY2VpcHQvaGFz"
        "aCBtaXNtYXRjaCIsCiAgICApCiAgICByZXF1aXJlKAogICAgICAgIHZhbGlkYXRpb24u"
        "Z2V0KCJvcmlnaW5hbF9iYXNlX2NvbW1pdCIpID09IHNlbGVjdGVkWyJiYXNlX2NvbW1p"
        "dCJdCiAgICAgICAgYW5kIHZhbGlkYXRpb24uZ2V0KCJiYXNlX3RyZWVfaWRlbnRpdHlf"
        "bWF0Y2hlcyIpIGlzIFRydWUKICAgICAgICBhbmQgdmFsaWRhdGlvbi5nZXQoIm9yaWdp"
        "bmFsX2Jhc2VfdHJlZV9zaGEiKQogICAgICAgID09IHZhbGlkYXRpb24uZ2V0KCJzbmFw"
        "c2hvdF9oZWFkX3RyZWVfc2hhIikKICAgICAgICBhbmQgdmFsaWRhdGlvbi5nZXQoIm9y"
        "aWdpbmFsX3RyZWVfc291cmNlIikKICAgICAgICA9PSBmImh0dHBzOi8vYXBpLmdpdGh1"
        "Yi5jb20vcmVwb3Mve3NlbGVjdGVkWydyZXBvJ119L2dpdC9jb21taXRzL3tzZWxlY3Rl"
        "ZFsnYmFzZV9jb21taXQnXX0iLAogICAgICAgICJPcmlnaW5hbCBiYXNlL3RyZWUgaWRl"
        "bnRpdHkgcmVjZWlwdCBtaXNzaW5nIiwKICAgICkKICAgIHJlcXVpcmUoCiAgICAgICAg"
        "dmFsaWRhdGlvbi5nZXQoInJlYWNoYWJsZV9mb3J3YXJkX2NvbW1pdHMiKSA9PSAwCiAg"
        "ICAgICAgYW5kIHZhbGlkYXRpb24uZ2V0KCJmb3J3YXJkX29yX3VucmVsYXRlZF9jb21t"
        "aXRfb2JqZWN0cyIpID09IDAsCiAgICAgICAgIlNuYXBzaG90IHJlY2VpcHQgcmVjb3Jk"
        "cyB1bnNhZmUgaGlzdG9yeSIsCiAgICApCiAgICBlbnZpcm9ubWVudCA9IHsKICAgICAg"
        "ICBrZXk6IHZhbHVlIGZvciBrZXksIHZhbHVlIGluIG9zLmVudmlyb24uaXRlbXMoKSBp"
        "ZiBub3Qga2V5LnN0YXJ0c3dpdGgoIkdJVF8iKQogICAgfQogICAgZW52aXJvbm1lbnQu"
        "dXBkYXRlKAogICAgICAgIEdJVF9DT05GSUdfTk9TWVNURU09IjEiLAogICAgICAgIEdJ"
        "VF9DT05GSUdfR0xPQkFMPW9zLmRldm51bGwsCiAgICAgICAgR0lUX05PX1JFUExBQ0Vf"
        "T0JKRUNUUz0iMSIsCiAgICApCiAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVj"
        "dG9yeShwcmVmaXg9Im1vc3RhemEtc25hcHNob3QtaGlzdG9yeS0iKSBhcyB0ZW1wb3Jh"
        "cnk6CiAgICAgICAgZ2l0X2RpciA9IFBhdGgodGVtcG9yYXJ5KSAvICIuZ2l0IgogICAg"
        "ICAgIHdpdGggdGFyZmlsZS5vcGVuKGFyY2hpdmVfcGF0aCwgInI6Z3oiKSBhcyBhcmNo"
        "aXZlOgogICAgICAgICAgICBmb3IgbWVtYmVyIGluIGFyY2hpdmU6CiAgICAgICAgICAg"
        "ICAgICBwYXRoID0gUHVyZVBvc2l4UGF0aChtZW1iZXIubmFtZSkKICAgICAgICAgICAg"
        "ICAgIHJlcXVpcmUoCiAgICAgICAgICAgICAgICAgICAgbm90IHBhdGguaXNfYWJzb2x1"
        "dGUoKSBhbmQgIi4uIiBub3QgaW4gcGF0aC5wYXJ0cywKICAgICAgICAgICAgICAgICAg"
        "ICAiVW5zYWZlIHNuYXBzaG90IGFyY2hpdmUgbWVtYmVyIiwKICAgICAgICAgICAgICAg"
        "ICkKICAgICAgICAgICAgICAgIGlmIHBhdGgucGFydHMgYW5kIHBhdGgucGFydHNbMF0g"
        "PT0gIi5naXQiOgogICAgICAgICAgICAgICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAg"
        "ICAgICAgICAgICAgIG1lbWJlci5pc2RpcigpIG9yIG1lbWJlci5pc2ZpbGUoKSwKICAg"
        "ICAgICAgICAgICAgICAgICAgICAgIkdpdCBzbmFwc2hvdCBsaW5rcyBhcmUgZm9yYmlk"
        "ZGVuIiwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgaWYg"
        "bWVtYmVyLmlzZmlsZSgpOgogICAgICAgICAgICAgICAgICAgICAgICB0YXJnZXQgPSBQ"
        "YXRoKHRlbXBvcmFyeSkuam9pbnBhdGgoKnBhdGgucGFydHMpCiAgICAgICAgICAgICAg"
        "ICAgICAgICAgIHRhcmdldC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9v"
        "az1UcnVlKQogICAgICAgICAgICAgICAgICAgICAgICBzdHJlYW0gPSBhcmNoaXZlLmV4"
        "dHJhY3RmaWxlKG1lbWJlcikKICAgICAgICAgICAgICAgICAgICAgICAgcmVxdWlyZShz"
        "dHJlYW0gaXMgbm90IE5vbmUsICJVbnJlYWRhYmxlIHNuYXBzaG90IEdpdCBvYmplY3Qi"
        "KQogICAgICAgICAgICAgICAgICAgICAgICBhc3NlcnQgc3RyZWFtIGlzIG5vdCBOb25l"
        "CiAgICAgICAgICAgICAgICAgICAgICAgIHdpdGggc3RyZWFtLCB0YXJnZXQub3Blbigi"
        "eGIiKSBhcyBvdXRwdXQ6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBvdXRwdXQu"
        "d3JpdGUoc3RyZWFtLnJlYWQoKSkKICAgICAgICByZXF1aXJlKGdpdF9kaXIuaXNfZGly"
        "KCksICJTbmFwc2hvdCBHaXQgbWV0YWRhdGEgbWlzc2luZyIpCiAgICAgICAgZm9yIHJl"
        "bGF0aXZlIGluICgib2JqZWN0cy9pbmZvL2FsdGVybmF0ZXMiLCAic2hhbGxvdyIsICJp"
        "bmZvL2dyYWZ0cyIpOgogICAgICAgICAgICByZXF1aXJlKAogICAgICAgICAgICAgICAg"
        "bm90IChnaXRfZGlyIC8gcmVsYXRpdmUpLmV4aXN0cygpLAogICAgICAgICAgICAgICAg"
        "IkV4dGVybmFsIG9yIGluY29tcGxldGUgR2l0IGhpc3RvcnkgZm9yYmlkZGVuIiwKICAg"
        "ICAgICAgICAgKQoKICAgICAgICBkZWYgZ2l0KCphcmd1bWVudHM6IHN0cikgLT4gc3Ry"
        "OgogICAgICAgICAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAg"
        "ICAgIFsKICAgICAgICAgICAgICAgICAgICAiZ2l0IiwKICAgICAgICAgICAgICAgICAg"
        "ICAiLS1naXQtZGlyIiwKICAgICAgICAgICAgICAgICAgICBzdHIoZ2l0X2RpciksCiAg"
        "ICAgICAgICAgICAgICAgICAgIi1jIiwKICAgICAgICAgICAgICAgICAgICAiY29yZS5o"
        "b29rc1BhdGg9IiArIG9zLmRldm51bGwsCiAgICAgICAgICAgICAgICAgICAgIi1jIiwK"
        "ICAgICAgICAgICAgICAgICAgICAiY29yZS5mc21vbml0b3I9ZmFsc2UiLAogICAgICAg"
        "ICAgICAgICAgICAgICphcmd1bWVudHMsCiAgICAgICAgICAgICAgICBdLAogICAgICAg"
        "ICAgICAgICAgZW52PWVudmlyb25tZW50LAogICAgICAgICAgICAgICAgY2FwdHVyZV9v"
        "dXRwdXQ9VHJ1ZSwKICAgICAgICAgICAgICAgIHRleHQ9VHJ1ZSwKICAgICAgICAgICAg"
        "ICAgIGNoZWNrPUZhbHNlLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJlcXVpcmUo"
        "cmVzdWx0LnJldHVybmNvZGUgPT0gMCwgIkNhbm5vdCB2ZXJpZnkgc25hcHNob3QgR2l0"
        "IGhpc3RvcnkiKQogICAgICAgICAgICByZXR1cm4gcmVzdWx0LnN0ZG91dC5zdHJpcCgp"
        "CgogICAgICAgIGhlYWQgPSBnaXQoInJldi1wYXJzZSIsICJIRUFEIikKICAgICAgICB0"
        "cmVlID0gZ2l0KCJyZXYtcGFyc2UiLCAiSEVBRF57dHJlZX0iKQogICAgICAgIHJlcXVp"
        "cmUoCiAgICAgICAgICAgIGhlYWQgPT0gdmFsaWRhdGlvbi5nZXQoImhlYWQiKQogICAg"
        "ICAgICAgICBhbmQgdHJlZSA9PSB2YWxpZGF0aW9uLmdldCgib3JpZ2luYWxfYmFzZV90"
        "cmVlX3NoYSIpLAogICAgICAgICAgICAiRXhwb3J0ZWQgSEVBRC90cmVlIGRpZmZlcnMg"
        "ZnJvbSB2ZXJpZmllZCBvcmlnaW5hbC1iYXNlIHJlY2VpcHQiLAogICAgICAgICkKICAg"
        "ICAgICBhbmNlc3RvcnMgPSBzZXQoZ2l0KCJyZXYtbGlzdCIsICJIRUFEIikuc3BsaXRs"
        "aW5lcygpKQogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIG5vdCBnaXQoInJldi1s"
        "aXN0IiwgIi0tYWxsIiwgIi0tbm90IiwgIkhFQUQiKSwKICAgICAgICAgICAgIkZvcndh"
        "cmQgb3IgdW5yZWxhdGVkIEdpdCByZWZzIGRldGVjdGVkIiwKICAgICAgICApCiAgICAg"
        "ICAgZm9yIHJlZiBpbiBnaXQoImZvci1lYWNoLXJlZiIsICItLWZvcm1hdD0lKHJlZm5h"
        "bWUpIikuc3BsaXRsaW5lcygpOgogICAgICAgICAgICByZXF1aXJlKAogICAgICAgICAg"
        "ICAgICAgZ2l0KCJyZXYtcGFyc2UiLCByZWYgKyAiXntjb21taXR9IikgaW4gYW5jZXN0"
        "b3JzLAogICAgICAgICAgICAgICAgIkdpdCByZWZlcmVuY2UgaXMgbm90IGFuIGV4cG9y"
        "dGVkIEhFQUQgYW5jZXN0b3IiLAogICAgICAgICAgICApCiAgICAgICAgb2JqZWN0cyA9"
        "IGdpdCgKICAgICAgICAgICAgImNhdC1maWxlIiwKICAgICAgICAgICAgIi0tYmF0Y2gt"
        "YWxsLW9iamVjdHMiLAogICAgICAgICAgICAiLS1iYXRjaC1jaGVjaz0lKG9iamVjdHR5"
        "cGUpICUob2JqZWN0bmFtZSkiLAogICAgICAgICkKICAgICAgICBjb21taXRzID0gewog"
        "ICAgICAgICAgICBsaW5lLnNwbGl0KClbMV0KICAgICAgICAgICAgZm9yIGxpbmUgaW4g"
        "b2JqZWN0cy5zcGxpdGxpbmVzKCkKICAgICAgICAgICAgaWYgbGluZS5zdGFydHN3aXRo"
        "KCJjb21taXQgIikKICAgICAgICB9CiAgICAgICAgcmVxdWlyZSgKICAgICAgICAgICAg"
        "Y29tbWl0cyA8PSBhbmNlc3RvcnMgYW5kIGxlbihjb21taXRzKSA9PSB2YWxpZGF0aW9u"
        "LmdldCgiY29tbWl0X29iamVjdHMiKSwKICAgICAgICAgICAgIkZvcndhcmQvdW5yZWxh"
        "dGVkIG9yIGNoYW5nZWQgR2l0IGNvbW1pdC1vYmplY3QgaW52ZW50b3J5IiwKICAgICAg"
        "ICApCg=="
    ),
    "src/mostaza/packaging.py": (
        "..."
    ),
    "src/mostaza/resources.py": (
        "IiIiQm91bmRlZCBvZmZpY2lhbCBhc3NldCByZXRyaWV2YWwgYW5kIGZhaWwtY2xvc2Vk"
        "IHByb3ZlbmFuY2UgY2hlY2tzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5v"
        "dGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9y"
        "dCB0ZW1wZmlsZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1w"
        "b3J0IEFueQpmcm9tIHVybGxpYi5wYXJzZSBpbXBvcnQgdXJsc3BsaXQKZnJvbSB1cmxs"
        "aWIucmVxdWVzdCBpbXBvcnQgUmVxdWVzdCwgdXJsb3BlbgoKTUFYX1JFU09VUkNFX0JZ"
        "VEVTID0gMTYgKiAxMDI0ICogMTAyNAoKCmNsYXNzIFJlc291cmNlRXJyb3IoVmFsdWVF"
        "cnJvcik6CiAgICAiIiJBbiBhdXRoZW50aWMgcmVxdWlyZWQgcmVzb3VyY2UgaXMgdW5h"
        "dmFpbGFibGUgb3IgY2hhbmdlZC4iIiIKCgpkZWYgc2hhMjU2KHBhdGg6IFBhdGgpIC0+"
        "IHN0cjoKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggcGF0aC5v"
        "cGVuKCJyYiIpIGFzIHN0cmVhbToKICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1i"
        "ZGE6IHN0cmVhbS5yZWFkKDY1NTM2KSwgYiIiKToKICAgICAgICAgICAgZGlnZXN0LnVw"
        "ZGF0ZShibG9jaykKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgcmVz"
        "b3VyY2VfcGF0aChyb290OiBQYXRoLCByZWxhdGl2ZTogc3RyKSAtPiBQYXRoOgogICAg"
        "cGFydHMgPSByZWxhdGl2ZS5zcGxpdCgiLyIpCiAgICBpZiAoCiAgICAgICAgbm90IHJl"
        "bGF0aXZlLnN0YXJ0c3dpdGgoInJlc291cmNlcy9vZmZpY2lhbC8iKQogICAgICAgIG9y"
        "IGFueShwYXJ0IGluIHsiIiwgIi4iLCAiLi4ifSBmb3IgcGFydCBpbiBwYXJ0cykKICAg"
        "ICAgICBvciAiXFwiIGluIHJlbGF0aXZlCiAgICAgICAgb3IgIjoiIGluIHJlbGF0aXZl"
        "CiAgICApOgogICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoZiJJbnZhbGlkIHJlc291"
        "cmNlIHBhdGg6IHtyZWxhdGl2ZX0iKQogICAgcGF0aCA9IHJvb3QgLyByZWxhdGl2ZQog"
        "ICAgZm9yIHBhcmVudCBpbiBbcGF0aCwgKnBhdGgucGFyZW50c106CiAgICAgICAgaWYg"
        "cGFyZW50ID09IHJvb3QucGFyZW50OgogICAgICAgICAgICBicmVhawogICAgICAgIGlm"
        "IHBhcmVudC5pc19zeW1saW5rKCk6CiAgICAgICAgICAgIHJhaXNlIFJlc291cmNlRXJy"
        "b3IoZiJTeW1saW5rIHJlc291cmNlIHBhdGg6IHtyZWxhdGl2ZX0iKQogICAgaWYgbm90"
        "IHBhdGgucmVzb2x2ZSgpLmlzX3JlbGF0aXZlX3RvKHJvb3QucmVzb2x2ZSgpIC8gInJl"
        "c291cmNlcyIgLyAib2ZmaWNpYWwiKToKICAgICAgICByYWlzZSBSZXNvdXJjZUVycm9y"
        "KGYiUmVzb3VyY2UgZXNjYXBlcyBpbnZlbnRvcnk6IHtyZWxhdGl2ZX0iKQogICAgcmV0"
        "dXJuIHBhdGgKCgpkZWYgcmVhZF9tYW5pZmVzdChyb290OiBQYXRoKSAtPiBkaWN0W3N0"
        "ciwgQW55XToKICAgIHZhbHVlID0ganNvbi5sb2Fkcygocm9vdCAvICJyZXNvdXJjZXMv"
        "bWFuaWZlc3QuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgaWYgbm90IGlzaW5zdGFuY2Uo"
        "dmFsdWUsIGRpY3QpIG9yIHZhbHVlLmdldCgic2NoZW1hX3ZlcnNpb24iKSAhPSAxOgog"
        "ICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoIlVuc3VwcG9ydGVkIHJlc291cmNlIG1h"
        "bmlmZXN0IikKICAgIGVudHJpZXMgPSB2YWx1ZS5nZXQoInJlc291cmNlcyIpCiAgICBp"
        "ZiBub3QgaXNpbnN0YW5jZShlbnRyaWVzLCBsaXN0KSBvciBhbnkobm90IGlzaW5zdGFu"
        "Y2UoZSwgZGljdCkgZm9yIGUgaW4gZW50cmllcyk6CiAgICAgICAgcmFpc2UgUmVzb3Vy"
        "Y2VFcnJvcigiSW52YWxpZCByZXNvdXJjZSBpbnZlbnRvcnkiKQogICAgaWRzID0gW2Vu"
        "dHJ5LmdldCgiaWQiKSBmb3IgZW50cnkgaW4gZW50cmllc10KICAgIGlmIGFueShub3Qg"
        "aXNpbnN0YW5jZShpLCBzdHIpIGZvciBpIGluIGlkcykgb3IgbGVuKHNldChpZHMpKSAh"
        "PSBsZW4oaWRzKToKICAgICAgICByYWlzZSBSZXNvdXJjZUVycm9yKCJEdXBsaWNhdGUg"
        "b3IgaW52YWxpZCByZXNvdXJjZSBJRHMiKQogICAgcmV0dXJuIHZhbHVlCgoKZGVmIHJl"
        "cXVpcmVfcmVzb3VyY2VzKHJvb3Q6IFBhdGgsIGlkczogbGlzdFtzdHJdKSAtPiBkaWN0"
        "W3N0ciwgUGF0aF06CiAgICBlbnRyaWVzID0ge2VudHJ5WyJpZCJdOiBlbnRyeSBmb3Ig"
        "ZW50cnkgaW4gcmVhZF9tYW5pZmVzdChyb290KVsicmVzb3VyY2VzIl19CiAgICByZXN1"
        "bHQ6IGRpY3Rbc3RyLCBQYXRoXSA9IHt9CiAgICBmb3IgcmVzb3VyY2VfaWQgaW4gaWRz"
        "OgogICAgICAgIGVudHJ5ID0gZW50cmllcy5nZXQocmVzb3VyY2VfaWQpCiAgICAgICAg"
        "aWYgZW50cnkgaXMgTm9uZSBvciBlbnRyeS5nZXQoInN0YXR1cyIpICE9ICJ2ZXJpZmll"
        "ZCI6CiAgICAgICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoZiJPZmZpY2lhbCBwcmVy"
        "ZXF1aXNpdGUgcGVuZGluZzoge3Jlc291cmNlX2lkfSIpCiAgICAgICAgcGF0aCA9IHJl"
        "c291cmNlX3BhdGgocm9vdCwgZW50cnlbInBhdGgiXSkKICAgICAgICBpZiBub3QgcGF0"
        "aC5pc19maWxlKCkgb3Igc2hhMjU2KHBhdGgpICE9IGVudHJ5LmdldCgic2hhMjU2Iik6"
        "CiAgICAgICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoZiJSZXNvdXJjZSBtaXNzaW5n"
        "IG9yIGhhc2ggbWlzbWF0Y2g6IHtyZXNvdXJjZV9pZH0iKQogICAgICAgIHJlc3VsdFty"
        "ZXNvdXJjZV9pZF0gPSBwYXRoCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGZldGNoX3Jl"
        "c291cmNlKHJvb3Q6IFBhdGgsIHJlc291cmNlX2lkOiBzdHIpIC0+IFBhdGg6CiAgICAi"
        "IiJGZXRjaCBvbmUgcGlubmVkIHB1YmxpYyBmaWxlLCBuZXZlciBkYXRhc2V0cywgYXJj"
        "aGl2ZXMgb3IgbW9kZWwgd2VpZ2h0cy4iIiIKICAgIGVudHJpZXMgPSB7ZW50cnlbImlk"
        "Il06IGVudHJ5IGZvciBlbnRyeSBpbiByZWFkX21hbmlmZXN0KHJvb3QpWyJyZXNvdXJj"
        "ZXMiXX0KICAgIGVudHJ5ID0gZW50cmllcy5nZXQocmVzb3VyY2VfaWQpCiAgICBpZiBl"
        "bnRyeSBpcyBOb25lIG9yIG5vdCBlbnRyeS5nZXQoImRvd25sb2FkX3VybCIpIG9yIG5v"
        "dCBlbnRyeS5nZXQoInNoYTI1NiIpOgogICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3Io"
        "IlJlc291cmNlIG5lZWRzIGFuIGF1dGhlbnRpY2F0ZWQgb2ZmaWNpYWwgZG93bmxvYWQv"
        "cmV2aWV3IikKICAgIHVybCA9IGVudHJ5WyJkb3dubG9hZF91cmwiXQogICAgcGFyc2Vk"
        "ID0gdXJsc3BsaXQodXJsKQogICAgaWYgcGFyc2VkLnNjaGVtZSAhPSAiaHR0cHMiIG9y"
        "IHBhcnNlZC5ob3N0bmFtZSAhPSAid3d3LmthZ2dsZS5jb20iOgogICAgICAgIHJhaXNl"
        "IFJlc291cmNlRXJyb3IoIkRvd25sb2FkIG11c3Qgb3JpZ2luYXRlIGF0IHRoZSBvZmZp"
        "Y2lhbCBLYWdnbGUgZW5kcG9pbnQiKQogICAgdGFyZ2V0ID0gcmVzb3VyY2VfcGF0aChy"
        "b290LCBlbnRyeVsicGF0aCJdKQogICAgdGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRz"
        "PVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBpZiB0YXJnZXQuZXhpc3RzKCk6CiAgICAg"
        "ICAgaWYgc2hhMjU2KHRhcmdldCkgIT0gZW50cnlbInNoYTI1NiJdOgogICAgICAgICAg"
        "ICByYWlzZSBSZXNvdXJjZUVycm9yKCJFeGlzdGluZyByZXNvdXJjZSBkaWZmZXJzOyBy"
        "ZWZ1c2luZyB0byBvdmVyd3JpdGUiKQogICAgICAgIHJldHVybiB0YXJnZXQKICAgIHRl"
        "bXBvcmFyeTogUGF0aCB8IE5vbmUgPSBOb25lCiAgICB0cnk6CiAgICAgICAgd2l0aCB1"
        "cmxvcGVuKAogICAgICAgICAgICBSZXF1ZXN0KHVybCwgaGVhZGVycz17IlVzZXItQWdl"
        "bnQiOiAiTW9zdGF6YS8wLjEifSksIHRpbWVvdXQ9NjAKICAgICAgICApIGFzIHJlc3Bv"
        "bnNlOgogICAgICAgICAgICBmaW5hbF9ob3N0ID0gdXJsc3BsaXQocmVzcG9uc2UudXJs"
        "KS5ob3N0bmFtZSBvciAiIgogICAgICAgICAgICBpZiBub3QgKAogICAgICAgICAgICAg"
        "ICAgZmluYWxfaG9zdCA9PSAid3d3LmthZ2dsZS5jb20iCiAgICAgICAgICAgICAgICBv"
        "ciBmaW5hbF9ob3N0ID09ICJzdG9yYWdlLmdvb2dsZWFwaXMuY29tIgogICAgICAgICAg"
        "ICAgICAgb3IgZmluYWxfaG9zdC5lbmRzd2l0aCgiLmdvb2dsZXVzZXJjb250ZW50LmNv"
        "bSIpCiAgICAgICAgICAgICk6CiAgICAgICAgICAgICAgICByYWlzZSBSZXNvdXJjZUVy"
        "cm9yKCJVbmV4cGVjdGVkIG9mZmljaWFsIGRvd25sb2FkIHJlZGlyZWN0IikKICAgICAg"
        "ICAgICAgd2l0aCB0ZW1wZmlsZS5OYW1lZFRlbXBvcmFyeUZpbGUoZGlyPXRhcmdldC5w"
        "YXJlbnQsIGRlbGV0ZT1GYWxzZSkgYXMgb3V0cHV0OgogICAgICAgICAgICAgICAgdGVt"
        "cG9yYXJ5ID0gUGF0aChvdXRwdXQubmFtZSkKICAgICAgICAgICAgICAgIHRvdGFsID0g"
        "MAogICAgICAgICAgICAgICAgd2hpbGUgYmxvY2sgOj0gcmVzcG9uc2UucmVhZCg2NTUz"
        "Nik6CiAgICAgICAgICAgICAgICAgICAgdG90YWwgKz0gbGVuKGJsb2NrKQogICAgICAg"
        "ICAgICAgICAgICAgIGlmIHRvdGFsID4gbWluKAogICAgICAgICAgICAgICAgICAgICAg"
        "ICBlbnRyeS5nZXQoIm1heF9ieXRlcyIsIE1BWF9SRVNPVVJDRV9CWVRFUyksIE1BWF9S"
        "RVNPVVJDRV9CWVRFUwogICAgICAgICAgICAgICAgICAgICk6CiAgICAgICAgICAgICAg"
        "ICAgICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoIlJlc291cmNlIGV4Y2VlZHMgYm91"
        "bmRlZCBkb3dubG9hZCBzaXplIikKICAgICAgICAgICAgICAgICAgICBvdXRwdXQud3Jp"
        "dGUoYmxvY2spCiAgICAgICAgaWYgc2hhMjU2KHRlbXBvcmFyeSkgIT0gZW50cnlbInNo"
        "YTI1NiJdOgogICAgICAgICAgICByYWlzZSBSZXNvdXJjZUVycm9yKCJEb3dubG9hZGVk"
        "IGJ5dGVzIGRpZmZlciBmcm9tIHRoZSBhcHByb3ZlZCBwaW4iKQogICAgICAgICMgSGFy"
        "ZCBsaW5rIHB1Ymxpc2hlcyBhdG9taWNhbGx5IHdpdGhvdXQgcmVwbGFjaW5nIGNvbmN1"
        "cnJlbnQvdXNlciBmaWxlcy4KICAgICAgICBvcy5saW5rKHRlbXBvcmFyeSwgdGFyZ2V0"
        "KQogICAgICAgIHJldHVybiB0YXJnZXQKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgdGVt"
        "cG9yYXJ5IGlzIG5vdCBOb25lOgogICAgICAgICAgICB0ZW1wb3JhcnkudW5saW5rKG1p"
        "c3Npbmdfb2s9VHJ1ZSkK"
    ),
}
EMBEDDED_HASHES = {
    "archives/E0-B180.zip": (
        "df3267ecadeff1fdf3193495632fa55556d140d79952368e41bb3b103f6ffde2"
    ),
    "archives/E1-B180.zip": (
        "6dd6cede0a924bc25ef5d1a7fb6f817a5f3fe8d01476d6e42fe4b2c8ff50c564"
    ),
    "experiments/cohort.json": (
        "8ff596e7cb5f3d81575723402b37ca6569d112465a543ecf29bb4251db04c755"
    ),
    "quota-proposal.json": (
        "b747adb68531d0b71d559324f32fe6612523ac6da595a282ba9eabbc3f2f1ec2"
    ),
    "resources/manifest.json": (
        "d6e6e7671d63c21ec4d7fc7c97e56ec32733a3aeb6070cf483c22b5dc79cd5e9"
    ),
    "selected-input-pins.json": (
        "8d370f8509da628280634be75b019f5bf3880612078b87a3c30f7b8cfc1e105a"
    ),
    "src/mostaza/__init__.py": (
        "e6e7452e239f7a03d6645be265792ed103fcc93b27da4e1f2b982aa8af018d3c"
    ),
    "src/mostaza/evaluation/__init__.py": (
        "2106119b057d082ec60e8a8e21aef6fcc3f7d63a3ff4d0e177ddbac3db9efcac"
    ),
    "src/mostaza/evaluation/contracts.py": (
        "b8500c062a3e81b7bc0fb6051e72cc4af07689fc1cf2f8dc60f8c58a8c87e7d3"
    ),
    "src/mostaza/evaluation/runner.py": (
        "5f56dfd22e339ac72ae6c93b0b43c4229f92959b5fc28cda1f531d952b229770"
    ),
    "src/mostaza/evaluation/tasks.py": (
        "eab3240c3ddc78114b849b81b8944a74daadd6da7e1859e7bfa99455e6a8dc91"
    ),
    "src/mostaza/packaging.py": (
        "b1944c8e7dd66036cfc9679f0dca66386c2c30a5e2066cf2d559680b90ec029f"
    ),
    "src/mostaza/resources.py": (
        "b3b628f0b40e5303d18b412664fc2f542088983838ed003960cb0886ef41530c"
    ),
}
assert set(EMBEDDED_PAYLOAD) == set(EMBEDDED_HASHES)
EMBEDDED_ROOT = Path("/kaggle/working") / ("mostaza-phase2-" + str(uuid.uuid4()))
assert Path("/kaggle/input").is_dir(), "Kaggle only; no local model download"
EMBEDDED_ROOT.mkdir(exist_ok=False)
for name, encoded in EMBEDDED_PAYLOAD.items():
    assert not Path(name).is_absolute() and ".." not in Path(name).parts
    assert "\\" not in name and ":" not in name
    data = base64.b64decode(encoded, validate=True)
    assert hashlib.sha256(data).hexdigest() == EMBEDDED_HASHES[name]
    path = EMBEDDED_ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as stream:
        stream.write(data)
sys.path.insert(0, str(EMBEDDED_ROOT / "src"))
STARTUP_OBSERVER.stage("bootstrap.complete")

In [ ]:
import importlib
import os
import shutil
import subprocess
import time
import zipfile
from importlib.metadata import version

from mostaza.evaluation.contracts import MODEL, canonical_json, require, sha256
from mostaza.evaluation.runner import file_digest, snapshot_digest
from mostaza.evaluation.tasks import load_agent_tasks, validate_snapshot_history

# This cell is deliberately after the fail-closed quota/allocation guard.
DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
WHEELHOUSE = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
MODEL_PATH = Path(f"/kaggle/input/models/google/gemma-4/other/{MODEL}/2")
require(sys.version_info[:2] == (3, 12), "Review changed Kaggle Python before running")
require(MODEL_PATH.is_dir(), "Pinned 31B revision 2 mount required")
resource_index = json.loads((EMBEDDED_ROOT / "resources/manifest.json").read_text())
cohort = json.loads((EMBEDDED_ROOT / "experiments/cohort.json").read_text())
selection = json.loads((EMBEDDED_ROOT / "selected-input-pins.json").read_text())
quota = json.loads((EMBEDDED_ROOT / "quota-proposal.json").read_text())
for entry in resource_index["resources"]:
    if entry["id"] in {"swegemma", "adk_submission", "adk_eval_core"}:
        path = WHEELHOUSE / Path(entry["path"]).name
    elif entry["id"] == "tasks":
        path = DATA_DIR / "tasks.jsonl"
    else:
        continue
    require(file_digest(path) == entry["sha256"], "Mounted official input changed")
for pin in selection:
    path = DATA_DIR / pin["official_file_path"]
    require(file_digest(path) == pin["sha256"], "Selected graph/snapshot/input changed")
visible = {
    task.instance_id: task for task in load_agent_tasks(DATA_DIR / "tasks.jsonl")
}
for selected in cohort["tasks"]:
    task = visible[selected["instance_id"]]
    require(
        task.repo == selected["repo"] and task.base_commit == selected["base_commit"],
        "Frozen task metadata mismatch",
    )
STARTUP_OBSERVER.stage("inputs.verified")
# Offline wheel installation from the attached organizer wheelhouse only.
wheel_temp = EMBEDDED_ROOT / "wheelhouse"
wheel_temp.mkdir(exist_ok=False)
for wheel in sorted(WHEELHOUSE.glob("*.whl")):
    if "cutlass" in wheel.name.lower():
        continue
    name = (
        wheel.name.replace("cu128", "+cu128") if "+" not in wheel.name else wheel.name
    )
    (wheel_temp / name).symlink_to(wheel)
os.environ.update(
    TRANSFORMERS_OFFLINE="1",
    VLLM_NO_USAGE_STATS="1",
    VLLM_WORKER_MULTIPROC_METHOD="spawn",
    LITELLM_TELEMETRY="False",
    LITELLM_LOCAL_MODEL_COST_MAP="True",
    OTEL_SDK_DISABLED="true",
    TRANSFORMERS_NO_TF="1",
    TRANSFORMERS_NO_FLAX="1",
)
STARTUP_OBSERVER.stage("packages.install.begin")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        "--force-reinstall",
        *map(str, sorted(wheel_temp.glob("*.whl"))),
    ],
    check=True,
    timeout=600,
)
STARTUP_OBSERVER.stage("packages.install.complete")
for package, expected in [
    ("swegemma", "0.2.11"),
    ("adk-submission", "0.2.13"),
    ("adk-eval-core", "0.1.0"),
    ("google-adk", "1.36.1"),
]:
    require(version(package) == expected, "Pinned official package required")
STARTUP_OBSERVER.stage("torch.import.begin")
torch = importlib.import_module("torch")
STARTUP_OBSERVER.stage("torch.import.complete")
resolve_task_snapshot_paths = importlib.import_module(
    "swegemma.deduplication"
).resolve_task_snapshot_paths

require(
    torch.cuda.device_count() == 4
    and all(torch.cuda.get_device_name(i) == "NVIDIA L4" for i in range(4)),
    "L4 x4 required",
)
for selected in cohort["tasks"]:
    require(
        snapshot_digest(
            DATA_DIR / "snapshots", selected["instance_id"], selected["repo"], DATA_DIR
        )
        == selected["snapshot_sha256"],
        "Frozen snapshot reconstruction mismatch",
    )
    effective, base, delta = resolve_task_snapshot_paths(
        DATA_DIR / "snapshots", selected["instance_id"], selected["repo"]
    )
    require(base is None and delta is None, "Reviewed standalone snapshots required")
    history = next(
        pin
        for pin in selection
        if pin.get("instance_id") == selected["instance_id"]
        and pin["kind"] == "snapshot"
    )
    validate_snapshot_history(Path(effective), selected, history)
STARTUP_OBSERVER.stage("preflight.complete")

## Native evaluator with persistent startup observations

One LlmAgent per original arm. Browser stopping is external and version-bound.


In [ ]:
STARTUP_OBSERVER.stage("execution.imports.begin")

In [ ]:
import asyncio
from datetime import UTC, datetime

import litellm
from adk_submission import (
    ModelRegistry,
    VllmConfig,
    VllmServer,
    compile_submission,
    validate_directory,
)
from adk_submission.yaml_loader import load_yaml
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator
from swegemma.harness import verification
from swegemma.models import load_tasks

from mostaza.evaluation.contracts import validate_manifest
from mostaza.evaluation.runner import describe_runtime, validate_bindings
from mostaza.packaging import MAX_BYTES, MAX_FILES, collect_submission


def scheduling_allowed(now_epoch: float, deadline_epoch: float) -> bool:
    """AC-7: full task plus export must fit the allocation-anchored deadline."""
    import math

    if not math.isfinite(now_epoch) or not math.isfinite(deadline_epoch):
        return False
    # 180 generation + 300 verification + 120 overhead + 300 export.
    return deadline_epoch - now_epoch >= 900


STARTUP_OBSERVER.stage("execution.imports.complete")
require_launch_ready()
# The browser supervisor uses the prelaunch clock and exact version.
# Notebook observations do not establish physical GPU release.
RUN_ROOT = STARTUP_OBSERVER.root
STARTUP_OBSERVER.stage("compiler.begin")
limits, constraints = build_submission_limits()
agent_dirs, compiler_receipts = {}, {}
for configuration in quota["configurations"]:
    directory = RUN_ROOT / configuration / "submission"
    directory.mkdir(parents=True, exist_ok=False)
    archive_path = EMBEDDED_ROOT / "archives" / (configuration + ".zip")
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        require(
            len(entries) <= MAX_FILES
            and sum(e.file_size for e in entries) <= MAX_BYTES,
            "Oversized archive",
        )
        require(len({e.filename for e in entries}) == len(entries), "Duplicate archive")
        for entry in entries:
            name = entry.filename
            require(
                not entry.is_dir()
                and not Path(name).is_absolute()
                and ".." not in Path(name).parts
                and "\\" not in name
                and (entry.external_attr >> 16) & 0o170000 != 0o120000,
                "Unsafe archive member",
            )
            path = directory / name
            path.parent.mkdir(parents=True, exist_ok=True)
            with path.open("xb") as stream:
                stream.write(archive.read(entry))
    require(
        set(collect_submission(directory)) == {e.filename for e in entries},
        "Unused/forbidden submission content",
    )
    validate_directory(directory, limits=limits)
    raw = load_yaml(directory / "agent.yaml", root_dir=directory, limits=limits)
    inert = ModelRegistry()
    inert.register(MODEL, "compiler-only-inert-model")

    def inert_tool(**kwargs):
        raise RuntimeError("Compiler-only tool must never execute")

    compile_submission(
        directory,
        {name: inert_tool for name in raw["tools"]},
        inert,
        limits=limits,
        generation_constraints=constraints,
    )
    compiler_receipts[configuration] = {
        "status": "PASS",
        "archive_sha256": file_digest(archive_path),
    }
    agent_dirs[configuration] = directory

STARTUP_OBSERVER.stage("compiler.complete")
litellm.drop_params = True
server_config = VllmConfig(
    model=str(MODEL_PATH),
    tensor_parallel_size=4,
    tool_call_parser="gemma4",
    reasoning_parser="gemma4",
    max_model_len=32768,
    dtype="bfloat16" if torch.cuda.is_bf16_supported() else "auto",
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=False,
    max_loras=1,
    max_lora_rank=128,
    scheduler_reserve_full_isl=False,
    startup_timeout=1200,
    host="127.0.0.1",
    port=8000,
    extra_args=[],
)
server = VllmServer(server_config)
acquired = []
# All six rows begin as NOT_RUN; errors/deadline preserve the unscheduled rows.
rows = [
    {
        "configuration_id": c,
        "instance_id": t["instance_id"],
        "status": "NOT_RUN",
        "reason": "Not scheduled",
        "result": None,
    }
    for c in quota["configurations"]
    for t in cohort["tasks"]
]


def export_sources():
    with (RUN_ROOT / "schedule.json").open("w") as stream:
        json.dump(rows, stream, sort_keys=True)
    inventory = [
        {"path": p.relative_to(RUN_ROOT).as_posix(), "sha256": file_digest(p)}
        for p in sorted(RUN_ROOT.rglob("*"))
        if p.is_file() and p.name != "source-inventory.json"
    ]
    with (RUN_ROOT / "source-inventory.json").open("w") as stream:
        json.dump(inventory, stream, sort_keys=True)


try:
    # External browser supervision includes queue and cold start.
    export_sources()
    PLATFORM_SESSION.require_startup_ready()
    PLATFORM_SESSION.require_remaining(2100)
    server.config.startup_timeout = min(
        1200, max(1, int(PLATFORM_SESSION.startup_deadline_epoch - time.time()))
    )
    OBSERVATION_NAMESPACE["start_observed_server"](server, STARTUP_OBSERVER)
    PLATFORM_SESSION.require_startup_ready()
    print("MOSTAZA_MODEL_READY", time.time(), flush=True)
    STARTUP_OBSERVER.stage("registry.begin")
    models = server.create_model_registry(
        aliases=[MODEL], model_prefix="openai/", api_key="EMPTY"
    )
    STARTUP_OBSERVER.stage("registry.complete")
    tasks = {t.instance_id: t for t in load_tasks(DATA_DIR / "tasks.jsonl")}
    STARTUP_OBSERVER.stage("tasks.loaded")
    wheel_inventory = [
        {"name": p.name, "sha256": file_digest(p)}
        for p in sorted(WHEELHOUSE.glob("*.whl"))
    ]
    index = {p["id"]: p for p in resource_index["resources"]}
    resources = {
        "harness": {
            "version": "0.2.11",
            "sha256": index["swegemma"]["sha256"],
            "source": index["swegemma"]["url"],
        },
        "wheelhouse": {
            "version": "31",
            "sha256": sha256(canonical_json(wheel_inventory)),
            "source": index["swegemma"]["url"],
        },
        "dataset": {
            "version": "sha256:" + file_digest(DATA_DIR / "tasks.jsonl"),
            "sha256": file_digest(DATA_DIR / "tasks.jsonl"),
            "source": index["tasks"]["url"],
        },
    }
    for configuration in quota["configurations"]:
        output = RUN_ROOT / configuration / "results"
        config = EvalConfig(
            tasks_path=DATA_DIR / "tasks.jsonl",
            snapshots_dir=DATA_DIR / "snapshots",
            results_dir=output,
            submission_dir=agent_dirs[configuration],
            models=models,
            sandbox="subprocess",
            limits=limits,
            generation_constraints=constraints,
            concurrency=1,
            context_cache_config=None,
            events_compaction_config=None,
            graph_dir=str(DATA_DIR / "graphs"),
            embeddings_dir=str(DATA_DIR / "embeddings"),
            wheels_dir=DATA_DIR / "wheels",
            verbose=False,
            timeout_seconds=180,
            verification_timeout_seconds=300,
            max_tool_calls=30,
            max_time_minutes=3,
            max_turns=50,
        )
        runtime = describe_runtime(config, server_config)
        manifest = {
            "schema_version": 1,
            "experiment_id": configuration,
            "run_id": str(uuid.uuid4()),
            "arm": configuration[:2],
            "run_kind": "real_kaggle",
            "timestamp": datetime.now(UTC).isoformat(),
            "code_commit": PLATFORM_SESSION.code_commit,
            "working_tree_sha256": sha256(canonical_json(EMBEDDED_HASHES)),
            "config_sha256": runtime["config_sha256"],
            "archive_sha256": file_digest(
                EMBEDDED_ROOT / "archives" / (configuration + ".zip")
            ),
            "model": MODEL,
            "resources": resources,
            "cohort": cohort["tasks"],
            "inference": runtime["inference"],
            "budgets": runtime["budgets"],
            "environment": {
                "platform": "kaggle",
                "gpu_count": 4,
                "gpu_model": "NVIDIA L4",
                "snapshot_validation": {
                    p["instance_id"]: p for p in selection if p["kind"] == "snapshot"
                },
                "embedded_input_hashes": EMBEDDED_HASHES,
            },
            "source_run_id": PLATFORM_SESSION.run_url,
            "limitations": ["Exploratory cohort; no significance or superiority claim"],
        }
        validate_manifest(manifest)
        validate_bindings(
            config,
            manifest,
            server=server,
            archive_path=EMBEDDED_ROOT / "archives" / (configuration + ".zip"),
            wheelhouse_dir=WHEELHOUSE,
        )
        with (RUN_ROOT / configuration / "manifest.json").open("xb") as stream:
            stream.write(canonical_json(manifest))
        evaluator = Evaluator(config)
        try:
            for i, selected in enumerate(cohort["tasks"], 1):
                row = next(
                    r
                    for r in rows
                    if r["configuration_id"] == configuration
                    and r["instance_id"] == selected["instance_id"]
                )
                # 180 generation + 300 verification + 120 orchestration + 300 export.
                if not scheduling_allowed(time.time(), PLATFORM_SESSION.deadline_epoch):
                    row["reason"] = (
                        "Insufficient worst-case allowance before session deadline"
                    )
                    continue
                row["status"], row["reason"] = "ERROR", "Attempt interrupted; no retry"
                export_sources()
                task = evaluator._hydrate_task_from_secret(
                    tasks[selected["instance_id"]]
                )
                STARTUP_OBSERVER.check()
                STARTUP_OBSERVER.stage("evaluation.task")
                STARTUP_OBSERVER.event(
                    "task.start",
                    configuration=configuration,
                    instance_id=selected["instance_id"],
                )
                result = await asyncio.wait_for(
                    evaluator.evaluate_task(task=task, task_index=i, total_tasks=3),
                    timeout=600,
                )
                if not result.trace_path and result.trace is not None:
                    result.trace_path = verification.save_trace_artifact(
                        result.trace, output, task.instance_id
                    )
                raw = result.model_dump(mode="json", exclude={"trace"})
                strip_patch = verification.strip_protected_files_from_patch
                raw["_mostaza"] = {
                    "schema_version": 1,
                    "effective_agent_patch": strip_patch(
                        result.agent_patch, extra_protected_paths=task.test_files
                    ),
                    "original_patch_sha256": sha256(result.agent_patch.encode()),
                }
                path = output / f"official-result-{i:04d}.json"
                with path.open("xb") as stream:
                    stream.write(canonical_json(raw))
                STARTUP_OBSERVER.event(
                    "task.acquired",
                    configuration=configuration,
                    instance_id=selected["instance_id"],
                )
                row.update(
                    status="ACQUIRED",
                    reason="Native result; resolution pending replay",
                    result=path.relative_to(RUN_ROOT).as_posix(),
                )
                export_sources()
        finally:
            evaluator.docker.cleanup_all()
finally:
    try:
        log = Path(server.log_path)
        if log.is_file():
            shutil.copyfile(log, RUN_ROOT / "vllm-server.log")
    finally:
        try:
            server.stop()
        finally:
            STARTUP_OBSERVER.close()
            PLATFORM_SESSION.record_stop_request(RUN_ROOT)
            export_sources()